# REM sleep without atonia across open sleep cohorts

Everything runs in the cloud; nothing is downloaded to your computer. Every
dataset used here is open (no application, no credential):

| cohort | what | licence |
|---|---|---|
| CAP | 22 RBD patients + healthy sleepers + other sleep disorders (Italy) | ODC-By |
| HMC | 151 clinical PSGs, Haaglanden Medisch Centrum (Netherlands) | CC BY 4.0 |
| Challenge 2018 | 994 clinical PSGs, Massachusetts General Hospital (USA) | ODC-By |

Each night is streamed from PhysioNet's open mirror, reduced to a 1-2 MB file
on this Colab machine and copied to Google Drive (`MyDrive/rbd-rwa/packs`),
and the raw file is deleted. If the session disconnects, run the cells again:
nights already copied to Drive are restored and skipped.

Run order: **Runtime -> Run all**. The first cell asks for access to Drive;
if nobody answers within 3 minutes the run continues without it.

In [ ]:
# Work happens on this machine's disk; finished nights are ALSO copied to
# Google Drive when it is connected. (Colab's Drive connection can drop out
# under heavy use, so it is never the working copy.)
import os
from google.colab import drive
try:
    drive.mount('/content/drive', timeout_ms=180000)
    os.makedirs('/content/drive/MyDrive/rbd-rwa', exist_ok=True)
    MIRROR = '/content/drive/MyDrive/rbd-rwa'
except Exception as e:
    print('Drive not connected:', e, '- results stay on this Colab machine for now')
    MIRROR = ''
PACK_MIRROR = f'--mirror {MIRROR}/packs' if MIRROR else ''
RESULT_MIRROR = f'--mirror {MIRROR}/results' if MIRROR else ''
os.environ['RBD_DURABLE_DIR'] = '/content/durable'
!pip -q install wfdb
!mkdir -p /content/rbd-rwa/src /content/durable
%cd /content/rbd-rwa

In [ ]:
%%writefile /content/rbd-rwa/src/__init__.py

In [ ]:
%%writefile /content/rbd-rwa/src/config.py
"""Paths that work identically on Colab, locally, or anywhere else.

The guiding rule: RAW recordings are disposable, DERIVED data is precious.
Raw PSG is hundreds of MB per night and lives in scratch (Colab's ephemeral
disk); the extracted envelope + hypnogram is a few MB and lives in a durable
directory (Google Drive), so a disconnected session never loses real work.
"""
import os
from pathlib import Path


def _first_existing(*candidates):
    for c in candidates:
        if c and Path(c).exists():
            return Path(c)
    return None


ON_COLAB = "COLAB_GPU" in os.environ or Path("/content").exists()

# Durable storage: Drive on Colab, the project folder otherwise.
DRIVE = _first_existing(
    os.environ.get("RBD_DRIVE_DIR"),
    "/content/drive/MyDrive",
)
PROJECT_ROOT = Path(__file__).resolve().parent.parent
DURABLE = Path(os.environ.get("RBD_DURABLE_DIR",
                              (DRIVE / "rbd-rwa") if DRIVE else PROJECT_ROOT / "data"))

# Scratch: fast, large, and wiped when the session ends. Raw EDFs only.
SCRATCH = Path(os.environ.get("RBD_SCRATCH_DIR",
                              "/content/scratch" if ON_COLAB else PROJECT_ROOT / "data" / "raw"))

DERIVED   = DURABLE / "derived"      # one compact .npz per recording
MANIFESTS = DURABLE / "manifests"    # progress records, so a restart resumes
FIGURES   = DURABLE / "figures"
RESULTS   = DURABLE / "results"

for d in (DERIVED, MANIFESTS, FIGURES, RESULTS, SCRATCH):
    d.mkdir(parents=True, exist_ok=True)

# Envelope sampling rate. Phasic RWA bursts are >= 0.1 s, so 50 Hz (20 ms bins)
# gives 5 samples across the shortest event a method needs to see, while being
# ~100x smaller than the raw signal.
ENV_FS = 50

# EMG band. 10 Hz high-pass removes movement/drift and ECG baseline; 100 Hz is
# the conventional upper edge, lowered automatically when Nyquist forbids it.
EMG_BAND = (10.0, 100.0)

# A second, common band that every recording can supply. Chin EMG in CAP is
# sampled anywhere from 100 to 512 Hz, and amplitude depends on bandwidth, so
# comparing amplitudes ACROSS recordings needs the same band for all of them.
# 45 Hz is the highest edge a 100 Hz recording allows (Nyquist minus 5).
HARMONIZED_BAND = (10.0, 45.0)


def describe():
    print(f"on Colab : {ON_COLAB}")
    print(f"durable  : {DURABLE}   (derived data, survives disconnects)")
    print(f"scratch  : {SCRATCH}   (raw EDFs, disposable)")

In [ ]:
%%writefile /content/rbd-rwa/src/edf.py
"""Minimal EDF/EDF+ reader that returns each channel at its NATIVE rate.

Why not MNE: `mne.io.read_raw_edf` resamples every channel to the highest rate
in the file. CAP stores chin EMG at 256-512 Hz but leg EMG at 128 Hz, so the
leg channels silently came back upsampled to 512 Hz. Reading the samples
directly keeps each channel's true rate, reads only the channels asked for,
and needs nothing beyond numpy - which matters on hosted notebooks where
installing packages is not always possible.

Values are returned in microvolts when the header's physical dimension is a
voltage. Some files carry no unit at all (raw converter counts); those are
returned unscaled and flagged, because an absolute amplitude threshold means
nothing for them.
"""
from dataclasses import dataclass
from pathlib import Path

import numpy as np

_TO_UV = {"uv": 1.0, "µv": 1.0, "mv": 1e3, "v": 1e6, "nv": 1e-3}


@dataclass
class Channel:
    index: int
    label: str
    dim: str
    fs: float
    n_per_record: int
    gain: float          # physical units per digital unit
    offset: float        # physical value at digital zero
    prefilter: str
    transducer: str

    @property
    def uv_scale(self):
        """Factor from physical units to microvolts, or None if not a voltage."""
        return _TO_UV.get(self.dim.strip().lower().replace("μ", "µ"))


@dataclass
class EdfHeader:
    path: Path
    n_records: int
    record_s: float
    header_bytes: int
    start_date: str
    start_time: str
    channels: list

    @property
    def duration_s(self):
        return self.n_records * self.record_s

    def find(self, label):
        for c in self.channels:
            if c.label == label:
                return c
        raise KeyError(label)


def read_header(path) -> EdfHeader:
    path = Path(path)
    with open(path, "rb") as f:
        h = f.read(256)
        ns = int(h[252:256])
        s = f.read(256 * ns)

    def fields(width, start):
        return [s[start + i * width: start + (i + 1) * width]
                .decode("latin-1").strip() for i in range(ns)], start + width * ns

    labels, o = fields(16, 0)
    trans, o = fields(80, o)
    dims, o = fields(8, o)
    pmin, o = fields(8, o)
    pmax, o = fields(8, o)
    dmin, o = fields(8, o)
    dmax, o = fields(8, o)
    pref, o = fields(80, o)
    nsamp, o = fields(8, o)

    header_bytes = int(h[184:192])
    n_records = int(h[236:244])
    record_s = float(h[244:252])
    per_rec = [int(x) for x in nsamp]
    # Records actually present. The header may say "-1" (unknown), or claim
    # more records than a truncated file holds; trust the file size.
    on_disk = (path.stat().st_size - header_bytes) // (2 * sum(per_rec))
    n_records = on_disk if n_records < 0 else min(n_records, on_disk)

    chans = []
    for i in range(ns):
        p0, p1, d0, d1 = (float(pmin[i]), float(pmax[i]),
                          float(dmin[i]), float(dmax[i]))
        gain = (p1 - p0) / (d1 - d0) if d1 != d0 else 1.0
        chans.append(Channel(
            index=i, label=labels[i], dim=dims[i],
            fs=per_rec[i] / record_s if record_s > 0 else float("nan"),
            n_per_record=per_rec[i], gain=gain, offset=p0 - gain * d0,
            prefilter=pref[i], transducer=trans[i]))
    return EdfHeader(path, n_records, record_s, header_bytes,
                     h[168:176].decode("ascii", "ignore").strip(),
                     h[176:184].decode("ascii", "ignore").strip(), chans)


def read_channel(hdr: EdfHeader, channel):
    """
    One channel as float64 at its native rate, in microvolts when the unit is
    a voltage. `channel` is a Channel, an index, or a label (first match).
    Returns (signal, fs, unit_known).
    """
    if isinstance(channel, Channel):
        ch = channel
    elif isinstance(channel, int):
        ch = hdr.channels[channel]
    else:
        ch = hdr.find(channel)
    per_rec = [c.n_per_record for c in hdr.channels]
    rec_len = sum(per_rec)
    start = sum(per_rec[:ch.index])
    raw = np.memmap(hdr.path, dtype="<i2", mode="r", offset=hdr.header_bytes,
                    shape=(hdr.n_records, rec_len))
    x = np.asarray(raw[:, start:start + ch.n_per_record], dtype=np.float64).ravel()
    del raw
    x = x * ch.gain + ch.offset
    k = ch.uv_scale
    if k is not None:
        x *= k
    return x, ch.fs, k is not None

In [ ]:
%%writefile /content/rbd-rwa/src/ecg.py
"""Remove the heartbeat from chin EMG.

The chin electrodes pick up the ECG. Each QRS complex adds a ~50-100 ms
deflection to the chin signal, so in quiet REM sleep - exactly where RWA is
measured - every heartbeat can look like a small muscle burst. In CAP's
healthy controls the chin amplitude at the R-peak is up to 2.5x the
surrounding level, enough to cross the 2x-background threshold of SINBAR.
Automated RWA tools remove it (RBDtector uses template subtraction), and so
do we:

  1. find R-peaks on the ECG channel
  2. average the filtered chin EMG around them, in 5-minute blocks, into a
     heartbeat template (a median, so genuine muscle bursts do not leak in)
  3. subtract the template at every beat
"""
import numpy as np

ECG_BIPOLAR = ("ECG1-ECG2", "EKG1-EKG2")
ECG_PAIRS = (("ECG1", "ECG2"), ("EKG1", "EKG2"))
ECG_PREFIX = ("ECG", "EKG")


def _norm(label):
    return label.upper().replace(" ", "")


def find_ecg(hdr):
    """The ECG source: a channel, a (positive, negative) pair, or None."""
    chans = hdr.channels
    for c in chans:
        if any(p in _norm(c.label) for p in ECG_BIPOLAR):
            return c
    lab = {_norm(c.label): c for c in chans}
    for a, b in ECG_PAIRS:
        if a in lab and b in lab and lab[a].fs == lab[b].fs:
            return (lab[a], lab[b])
    return next((c for c in chans if _norm(c.label).startswith(ECG_PREFIX)), None)


def r_peaks(ecg, fs, window_s=30.0):
    """
    R-peak times in seconds. Band-pass 5-20 Hz, orient so the QRS points up,
    then keep peaks >= 0.3 s apart (<= 200 bpm) that exceed 35% of the local
    99th percentile, computed in `window_s` windows so that slow changes in
    ECG amplitude through the night do not break detection.
    """
    from scipy.signal import butter, sosfiltfilt, find_peaks
    if fs < 50 or len(ecg) < fs * 60:
        return np.empty(0)
    sos = butter(3, [5.0, min(20.0, fs / 2 - 1)], btype="band", fs=fs, output="sos")
    y = sosfiltfilt(sos, np.asarray(ecg, dtype=np.float64))
    if abs(np.percentile(y, 0.1)) > abs(np.percentile(y, 99.9)):
        y = -y
    w = int(window_s * fs)
    n = int(np.ceil(len(y) / w))
    level = np.array([np.percentile(y[i * w:(i + 1) * w], 99) for i in range(n)])
    height = np.repeat(0.35 * level, w)[: len(y)]
    peaks, _ = find_peaks(y, height=height, distance=int(0.3 * fs))
    return peaks / fs


def plausible(peaks_s, duration_s):
    """True when the detected beats look like a heart rate, not noise."""
    if len(peaks_s) < 100:
        return False
    rate = 60.0 * len(peaks_s) / duration_s
    rr = np.diff(peaks_s)
    return 35.0 <= rate <= 130.0 and np.mean((rr > 0.33) & (rr < 2.0)) > 0.9


def subtract_heartbeat(x, fs, peaks_s, pre_s=0.10, post_s=0.15, block_s=300.0):
    """
    Subtract a local heartbeat template from a FILTERED signal. Returns the
    cleaned copy. Segments never overlap because beats are >= 0.3 s apart.
    """
    from scipy.signal.windows import tukey
    x = np.array(x, dtype=np.float64, copy=True)
    a, b = int(round(pre_s * fs)), int(round(post_s * fs))
    idx = np.round(np.asarray(peaks_s) * fs).astype(np.int64)
    idx = idx[(idx - a >= 0) & (idx + b <= len(x))]
    if len(idx) < 20:
        return x
    seg = idx[:, None] + np.arange(-a, b)[None, :]
    beats = x[seg]
    taper = tukey(a + b, alpha=0.3)
    overall = np.median(beats, axis=0)
    block = (idx / fs // block_s).astype(np.int64)
    for k in np.unique(block):
        m = block == k
        tmpl = np.median(beats[m], axis=0) if m.sum() >= 20 else overall
        x[seg[m]] -= tmpl * taper
    return x


def artifact_ratio(env, env_fs, peaks_s):
    """
    Heartbeat-locked envelope profile: the median at each lag around the
    R-peak, peak (-40..+80 ms) over baseline (-500..-300 ms). ~1 means no
    heartbeat artefact; 2.5 means the beat dominates quiet EMG. Medians at
    each lag (not a maximum over noisy single beats) keep this unbiased.
    """
    i = np.round(np.asarray(peaks_s) * env_fs).astype(np.int64)
    lags = np.arange(-int(0.5 * env_fs), int(0.1 * env_fs) + 1)
    i = i[(i + lags[0] >= 0) & (i + lags[-1] < len(env))]
    if len(i) < 50:
        return float("nan")
    prof = np.array([np.median(env[i + k]) for k in lags])
    t = lags / env_fs
    base = np.median(prof[(t >= -0.5) & (t <= -0.3)])
    peak = prof[(t >= -0.04) & (t <= 0.08)].max()
    return float(peak / base) if base > 0 else float("nan")

In [ ]:
%%writefile /content/rbd-rwa/src/fetch.py
"""Resumable downloads. PhysioNet's TLS drops connections mid-transfer, so
every request retries and partial files resume with an HTTP Range header.

Open PhysioNet databases are mirrored byte-for-byte on AWS Open Data
(s3://physionet-open), which is far faster from cloud notebooks than
physionet.org, so the mirror is tried first and PhysioNet is the fallback."""
import time
import urllib.request
from pathlib import Path

CAP_MIRRORS = (
    "https://physionet-open.s3.amazonaws.com/capslpdb/1.0.0/",
    "https://physionet.org/files/capslpdb/1.0.0/",
)
CAP_BASE = CAP_MIRRORS[-1]


def remote_size(url, tries=5):
    for i in range(tries):
        try:
            req = urllib.request.Request(url, method="HEAD")
            return int(urllib.request.urlopen(req, timeout=45)
                       .headers.get("Content-Length", 0))
        except Exception:
            if i == tries - 1:
                return None
            time.sleep(3)


def download(url, dest: Path, tries=10, quiet=False):
    """Fetch `url` to `dest`, resuming if a partial file is already there."""
    dest = Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    total = remote_size(url)
    if total is None:
        raise IOError(f"cannot reach {url}")
    if dest.exists() and dest.stat().st_size == total:
        return dest
    for i in range(tries):
        have = dest.stat().st_size if dest.exists() else 0
        if have == total:
            return dest
        if have > total:                          # stale file from another source
            dest.unlink()
            have = 0
        try:
            headers = {"Range": f"bytes={have}-"} if have else {}
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=120) as r, open(dest, "ab") as f:
                while chunk := r.read(1 << 20):
                    f.write(chunk)
        except Exception as e:
            if not quiet:
                print(f"    {dest.name}: {type(e).__name__}, resuming "
                      f"({dest.stat().st_size / 1e6 if dest.exists() else 0:.0f} MB)")
            time.sleep(3)
    got = dest.stat().st_size if dest.exists() else 0
    if got != total:
        raise IOError(f"{dest.name}: got {got} of {total} bytes")
    return dest


def download_any(name, dest: Path, mirrors=CAP_MIRRORS):
    """Try each mirror in turn; the first one that completes wins."""
    last = None
    for base in mirrors:
        try:
            return download(base + name, dest, quiet=True)
        except Exception as e:
            last = e
    raise IOError(f"{name}: all mirrors failed ({last})")


def cap_records():
    """All CAP record names, grouped by cohort prefix."""
    txt = None
    for base in CAP_MIRRORS:
        try:
            txt = urllib.request.urlopen(base + "RECORDS", timeout=45).read().decode()
            break
        except Exception:
            continue
    if txt is None:
        raise IOError("cannot fetch the CAP RECORDS list from any mirror")
    recs = [r.replace(".edf", "") for r in txt.split()]
    groups = {}
    for r in recs:
        key = "".join(c for c in r if not c.isdigit())
        groups.setdefault(key, []).append(r)
    return groups


def cap_demographics():
    """
    Age and sex of every CAP subject, from the database's gender-age.xlsx,
    keyed by record name. The sheet has no header and two typos ("SBD" for
    sdb, "RDB11" for rbd11). Read with the standard library, so no Excel
    package is needed. Returns {} if no mirror answers.
    """
    import io
    import zipfile
    import xml.etree.ElementTree as ET
    raw = None
    for base in CAP_MIRRORS:
        try:
            raw = urllib.request.urlopen(base + "gender-age.xlsx", timeout=60).read()
            break
        except Exception:
            continue
    if raw is None:
        return {}
    z = zipfile.ZipFile(io.BytesIO(raw))
    m = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
    shared = ["".join(t.text or "" for t in si.iter(m + "t"))
              for si in ET.fromstring(z.read("xl/sharedStrings.xml")).iter(m + "si")]
    out = {}
    for row in ET.fromstring(z.read("xl/worksheets/sheet1.xml")).iter(m + "row"):
        vals = []
        for c in row.iter(m + "c"):
            v = c.find(m + "v")
            vals.append(shared[int(v.text)] if c.get("t") == "s"
                        else (v.text if v is not None else ""))
        if len(vals) < 3:
            continue
        rec = vals[0].strip().lower().replace("sbd", "sdb").replace("rdb", "rbd")
        try:
            out[rec] = {"sex": vals[1].strip().upper()[:1], "age": float(vals[2])}
        except ValueError:
            continue
    return out


def fetch_cap(record, scratch: Path):
    """Download one CAP recording (.edf + .txt) into scratch."""
    edf = download_any(record + ".edf", Path(scratch) / f"{record}.edf")
    ann = download_any(record + ".txt", Path(scratch) / f"{record}.txt")
    return edf, ann

In [ ]:
%%writefile /content/rbd-rwa/src/cap.py
"""Parse CAP Sleep Database annotations and align them to the EDF clock.

The annotation file is a RemLogic export whose epochs are stamped with WALL
CLOCK times ("23:07:19"), while the signal array starts at whatever time the
EDF header declares. Aligning them is the step most likely to go silently
wrong: an unnoticed offset shifts every sleep stage relative to the signal and
quietly corrupts every downstream measurement, so the offset is computed
explicitly here and reported.
"""
import datetime as dt
import re
from pathlib import Path

import numpy as np

# W / S1 / S2 / S3 / S4 / R / MT  ->  compact codes
STAGE_CODE = {"W": 0, "S1": 1, "S2": 2, "S3": 3, "S4": 3, "R": 4}
STAGE_NAME = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM", -1: "excluded"}
EPOCH_SEC = 30


def edf_start_datetime(edf_path: Path) -> dt.datetime:
    """Recording start from the EDF header (bytes 168-183)."""
    with open(edf_path, "rb") as f:
        head = f.read(184)
    d = head[168:176].decode("ascii", "ignore").strip()   # dd.mm.yy
    t = head[176:184].decode("ascii", "ignore").strip()   # hh.mm.ss
    day, mon, yr = (int(x) for x in re.split(r"[.\-/]", d))
    hh, mm, ss = (int(x) for x in re.split(r"[.\-:]", t))
    # EDF year is two digits: 85-99 -> 19xx, 00-84 -> 20xx (EDF spec clipping rule)
    yr += 1900 if yr >= 85 else 2000
    return dt.datetime(yr, mon, day, hh, mm, ss)


def parse_annotations(txt_path: Path):
    """Return (times, stages) for every scored 30 s epoch, in file order."""
    raw = Path(txt_path).read_bytes().decode("latin-1")
    lines = [l.rstrip("\r") for l in raw.split("\n")]

    # The table begins at the header row; column order varies between files.
    hdr_i = next(i for i, l in enumerate(lines) if l.startswith("Sleep Stage"))
    cols = [c.strip() for c in lines[hdr_i].split("\t")]
    i_stage = cols.index("Sleep Stage")
    i_time = next(i for i, c in enumerate(cols) if c.startswith("Time"))
    i_event = cols.index("Event") if "Event" in cols else None

    times, stages = [], []
    for line in lines[hdr_i + 1:]:
        parts = line.split("\t")
        if len(parts) <= max(i_stage, i_time):
            continue
        # Only sleep-stage rows; CAP files also carry MCAP-A1/A2/A3 arousal events
        if i_event is not None and i_event < len(parts):
            if not parts[i_event].strip().upper().startswith("SLEEP"):
                continue
        stage = parts[i_stage].strip().upper()
        tm = parts[i_time].strip()
        m = re.match(r"(\d{1,2})[:.](\d{2})[:.](\d{2})", tm)
        if not m:
            continue
        times.append(tuple(int(g) for g in m.groups()))
        stages.append(STAGE_CODE.get(stage, -1))
    return times, np.array(stages, dtype=np.int8)


def build_hypnogram(edf_path: Path, txt_path: Path, n_seconds: float):
    """
    Hypnogram on the EDF's own timeline: one code per 30 s epoch from recording
    start. Returns (hypnogram, info).
    """
    start = edf_start_datetime(edf_path)
    times, stages = parse_annotations(txt_path)
    if len(times) == 0:
        raise ValueError(f"{Path(txt_path).name}: no scored epochs found")

    # Walk the clock forward, adding a day whenever it wraps past midnight.
    day = 0
    prev = None
    offsets = []
    for hh, mm, ss in times:
        cur = hh * 3600 + mm * 60 + ss
        if prev is not None and cur < prev - 3600:   # wrapped midnight
            day += 1
        prev = cur
        stamp = dt.datetime(start.year, start.month, start.day, hh, mm, ss) \
            + dt.timedelta(days=day)
        offsets.append((stamp - start).total_seconds())

    offsets = np.array(offsets)
    # The first annotation may precede the EDF start by a day if the recording
    # itself began before midnight and the header date is the following day.
    if offsets[0] < -12 * 3600:
        offsets += 24 * 3600

    n_epochs = int(np.floor(n_seconds / EPOCH_SEC))
    hyp = np.full(n_epochs, -1, dtype=np.int8)
    idx = np.round(offsets / EPOCH_SEC).astype(int)
    keep = (idx >= 0) & (idx < n_epochs)
    hyp[idx[keep]] = stages[keep]

    info = {
        "edf_start": start.isoformat(),
        "first_annotation_offset_s": float(offsets[0]),
        "n_annotated": int(len(stages)),
        "n_placed": int(keep.sum()),
        "n_dropped_outside_signal": int((~keep).sum()),
        "n_epochs": int(n_epochs),
        "counts": {STAGE_NAME[k]: int((hyp == k).sum())
                   for k in sorted(STAGE_NAME) if (hyp == k).sum()},
    }
    return hyp, info

In [ ]:
%%writefile /content/rbd-rwa/src/extract.py
"""Turn a raw overnight PSG into a small derived file.

One CAP recording is ~370 MB; what RWA quantification needs is the EMG
amplitude envelope and the hypnogram, a few MB. Extracting and discarding the
raw signal is what makes the project runnable on hosted notebooks, where disk
is ephemeral and sessions end without warning.

Per recording we keep:
    chin  (5, n_bins) float16 - chin EMG amplitude per 20 ms bin, heartbeat
              removed: RMS and mean-rectified in the native band (10-100 Hz
              or as high as the sampling rate allows), the same two in the
              common 10-45 Hz band shared by every recording, and finally the
              native-band RMS WITHOUT heartbeat removal, for comparison
    legs  (n_legs, n_bins) float16 - leg EMG RMS, native band
    hypnogram (n_epochs,) int8   - 30 s sleep stages on the EDF timeline
    meta  json - channels, native rates, bands, units, alignment

TIMING. Bins are exactly 1/ENV_FS seconds long whatever the source rate, so a
bin index converts to time without drift. (An earlier version used a whole
number of samples per bin; at 512 Hz that made 51.2 bins per second instead
of 50 and shifted the hypnogram by ~11 minutes by the end of a night.)
"""
import json
from pathlib import Path

import numpy as np

from .config import ENV_FS, EMG_BAND, HARMONIZED_BAND, DERIVED
from .cap import build_hypnogram, EPOCH_SEC
from .edf import read_header, read_channel
from .ecg import find_ecg, r_peaks, plausible, subtract_heartbeat, artifact_ratio

# Chin rows, all with the heartbeat removed except the last, which keeps it
# so the effect of heartbeat removal can itself be measured.
CHIN_ROWS = ("rms", "mav", "rms_h", "mav_h", "rms_with_ecg")
F16_MAX = 65000.0     # float16 tops out at 65504; clip (and count) above this

# How chin and leg EMG are labelled across CAP's recording systems. Bipolar
# labels win over split electrodes, which win over a lone referential name.
CHIN_BIPOLAR = ("EMG1-EMG2", "EMG-EMG", "CHIN1-CHIN2")
CHIN_PAIRS = (("EMG1", "EMG2"), ("CHIN1", "CHIN2"))
CHIN_SINGLE = ("CHIN", "MILO", "EMG", "MENT", "SUBM")   # MILO = mylohyoid
LEG_BIPOLAR = ("DX1-DX2", "SX1-SX2")                    # DX/SX = right/left
LEG_CONTAINS = ("TIB",)
LEG_PAIRS = (("DX1", "DX2"), ("SX1", "SX2"))


def _norm(label):
    return label.upper().replace(" ", "")


def _pair(chans, a, b):
    lab = {_norm(c.label): c for c in chans}
    if a in lab and b in lab and lab[a].fs == lab[b].fs:
        return (lab[a], lab[b])
    return None


def find_emg(hdr):
    """
    Locate chin and leg EMG. Each source is either a single channel or a
    (positive, negative) pair to be subtracted. Returns (chin, legs).
    """
    chans = hdr.channels
    chin = next((c for c in chans if _norm(c.label) in CHIN_BIPOLAR
                 or any(p in _norm(c.label) for p in CHIN_BIPOLAR)), None)
    if chin is None:
        chin = next((p for a, b in CHIN_PAIRS if (p := _pair(chans, a, b))), None)
    if chin is None:
        chin = next((c for name in CHIN_SINGLE for c in chans
                     if _norm(c.label) == name), None)

    legs = [c for c in chans if any(p in _norm(c.label) for p in LEG_BIPOLAR)]
    legs += [c for c in chans if any(p in _norm(c.label) for p in LEG_CONTAINS)
             and c not in legs]
    if not legs:
        legs = [p for a, b in LEG_PAIRS if (p := _pair(chans, a, b))]
    return chin, legs


def read_source(hdr, src):
    """Signal for a channel or a (positive, negative) pair of channels."""
    if isinstance(src, tuple):
        a, fs, ka = read_channel(hdr, src[0])
        b, _, kb = read_channel(hdr, src[1])
        return a - b, fs, ka and kb, f"{src[0].label}-{src[1].label}", src[0]
    x, fs, k = read_channel(hdr, src)
    return x, fs, k, src.label, src


def emg_band(fs, band=EMG_BAND):
    """The EMG pass-band actually usable at this sampling rate."""
    lo, hi = band
    hi = min(hi, fs / 2.0 - 5.0)
    if hi <= lo:
        raise ValueError(f"sampling rate {fs} Hz too low for EMG band {band}")
    return lo, hi


def bandpass(sig, fs, band=EMG_BAND, mains=None):
    """Band-pass (and optionally notch) the EMG. Returns (filtered, (lo, hi))."""
    from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt

    lo, hi = emg_band(fs, band)
    sos = butter(4, [lo, hi], btype="band", fs=fs, output="sos")
    x = sosfiltfilt(sos, np.asarray(sig, dtype=np.float64))
    if mains:
        # Mains and its harmonics that fall inside the pass-band
        for f0 in np.arange(mains, hi, mains):
            b, a = iirnotch(f0, Q=30.0, fs=fs)
            x = filtfilt(b, a, x)
    return x, (lo, hi)


def envelope(sig, fs, env_fs=ENV_FS, band=EMG_BAND, mains=None, beats_s=None):
    """
    Band-pass the EMG, subtract the heartbeat if R-peak times are given, then
    reduce it to amplitude per bin of exactly 1/env_fs seconds.
    Returns (rms, mav, (lo, hi)).
    """
    x, (lo, hi) = bandpass(sig, fs, band, mains)
    if beats_s is not None and len(beats_s):
        x = subtract_heartbeat(x, fs, beats_s)
    rms, mav = bin_envelope(x, fs, env_fs)
    return rms, mav, (lo, hi)


def bin_envelope(x, fs, env_fs=ENV_FS):
    """RMS and mean-rectified amplitude of an already filtered signal, per bin
    of exactly 1/env_fs seconds."""
    n_bins = int(np.floor(len(x) * env_fs / fs))
    if n_bins == 0:
        raise ValueError("signal shorter than one envelope bin")
    edges = np.floor(np.arange(n_bins + 1) * fs / env_fs).astype(np.int64)
    counts = np.diff(edges)
    rms = np.sqrt(np.add.reduceat(x * x, edges[:-1]) / counts)
    mav = np.add.reduceat(np.abs(x), edges[:-1]) / counts
    return rms.astype(np.float32), mav.astype(np.float32)


def _describe(label, ch, fs, unit_known, band, x):
    peak = float(np.max(np.abs(x))) if len(x) else 0.0
    return {"label": label, "fs": fs, "band": list(band),
            "unit": ch.dim, "unit_known": bool(unit_known),
            "transducer": ch.transducer, "prefilter": ch.prefilter,
            # share of samples sitting at the recording's largest magnitude:
            # well above ~1e-4 means the recorder was clipping
            "clipped_frac": float(np.mean(np.abs(x) >= 0.999 * peak)) if peak > 0 else 0.0}


def extract_recording(edf_path: Path, txt_path: Path, record: str,
                      out_dir: Path = None, overwrite=False, mains=50.0):
    """Extract one CAP recording to a compact .npz. Returns the output path."""
    out_dir = Path(out_dir or DERIVED)
    out = out_dir / f"{record}.npz"
    if out.exists() and not overwrite:
        return out

    hdr = read_header(edf_path)
    chin_src, leg_srcs = find_emg(hdr)
    if chin_src is None:
        raise ValueError(f"{record}: no chin EMG channel among "
                         f"{[c.label for c in hdr.channels]}")

    # Heartbeats, for removing the ECG from the chin EMG
    ecg_src = find_ecg(hdr)
    beats, ecg_label = np.empty(0), None
    if ecg_src is not None:
        e, fs_e, _, ecg_label, _ = read_source(hdr, ecg_src)
        beats = r_peaks(e, fs_e)
        if not plausible(beats, len(e) / fs_e):
            beats = np.empty(0)
        del e

    x, fs, unit_known, label, ch = read_source(hdr, chin_src)
    rms, mav, band = envelope(x, fs, mains=mains, beats_s=beats)
    rms_h, mav_h, band_h = envelope(x, fs, band=HARMONIZED_BAND, mains=mains,
                                    beats_s=beats)
    rms_raw, _, _ = envelope(x, fs, mains=mains)
    chin_meta = _describe(label, ch, fs, unit_known, band, x)
    chin_meta["band_harmonized"] = list(band_h)
    chin_meta["ecg"] = {
        "channel": ecg_label, "beats": int(len(beats)),
        "heart_rate_bpm": round(60.0 * len(beats) / (len(x) / fs), 1) if len(beats) else None,
        "artifact_ratio_before": round(artifact_ratio(rms_raw, ENV_FS, beats), 3),
        "artifact_ratio_after": round(artifact_ratio(rms, ENV_FS, beats), 3),
    }
    chin_rows = [rms, mav, rms_h, mav_h, rms_raw]
    del x

    leg_rows, leg_meta = [], []
    for src in leg_srcs:
        x, fs, k, label, ch = read_source(hdr, src)
        r, _, b = envelope(x, fs, mains=mains)
        leg_rows.append(r)
        leg_meta.append(_describe(label, ch, fs, k, b, x))
        del x

    n = min(len(r) for r in chin_rows + leg_rows)   # sources can differ by a bin
    chin = np.stack([r[:n] for r in chin_rows])
    legs = np.stack([r[:n] for r in leg_rows]) if leg_rows \
        else np.zeros((0, n), np.float32)

    n_seconds = n / ENV_FS
    hyp, align = build_hypnogram(edf_path, txt_path, n_seconds)

    meta = {
        "record": record,
        "chin_channel": chin_meta["label"],
        "leg_channels": [m["label"] for m in leg_meta],
        "chin_info": chin_meta,
        "leg_info": leg_meta,
        "chin_rows": list(CHIN_ROWS),
        "env_fs": ENV_FS,
        "mains_notch": mains,
        "duration_s": float(n_seconds),
        "epoch_sec": EPOCH_SEC,
        "alignment": align,
    }
    # Anything above float16's range is an artefact (tens of millivolts);
    # clip it rather than store inf, and record how much there was.
    meta["bins_clipped_f16"] = int((chin > F16_MAX).sum() + (legs > F16_MAX).sum())
    out_dir.mkdir(parents=True, exist_ok=True)
    tmp = out.with_name(out.stem + ".tmp.npz")
    np.savez_compressed(tmp, chin=np.minimum(chin, F16_MAX).astype(np.float16),
                        legs=np.minimum(legs, F16_MAX).astype(np.float16),
                        hypnogram=hyp, meta=json.dumps(meta))
    tmp.replace(out)                             # never leave a half-written file
    return out


def load_derived(path):
    """
    Read a derived file back. Returns a dict with float32 arrays:
    chin_rms, chin_mav, chin_rms_h, chin_mav_h, legs (n_legs, n) and
    hypnogram, plus meta.
    """
    z = np.load(path, allow_pickle=False)
    meta = json.loads(str(z["meta"]))
    chin = z["chin"].astype(np.float32)
    d = {f"chin_{name}": chin[i] for i, name in enumerate(meta["chin_rows"])}
    d["legs"] = z["legs"].astype(np.float32)
    d["hypnogram"] = z["hypnogram"]
    d["meta"] = meta
    return d


def summarise(path):
    d = load_derived(path)
    meta, hyp = d["meta"], d["hypnogram"]
    rem = int((hyp == 4).sum())
    ci = meta["chin_info"]
    legs = "/".join(f"{m['fs']:g}" for m in meta["leg_info"]) or "-"
    unit = "" if ci["unit_known"] else "  NO-UNITS"
    e = ci.get("ecg", {})
    ecg = (f"ECG artefact x{e['artifact_ratio_before']:.2f}->x{e['artifact_ratio_after']:.2f}"
           if e.get("beats") else "no ECG")
    return (f"{meta['record']:<8} {meta['duration_s']/3600:>5.1f} h  "
            f"chin {ci['label']} @ {ci['fs']:g} Hz{unit}  legs {legs} Hz  "
            f"REM {rem} epochs ({rem*0.5:.0f} min)  {ecg}")

In [ ]:
%%writefile /content/rbd-rwa/src/pack.py
"""Compact per-recording storage: full detail for REM, summaries for the rest.

RWA is measured in REM sleep. Everything else a method needs from the rest of
the night is a background level, which 1-second summaries of N2/N3 provide.
Keeping 20 ms resolution for REM epochs only shrinks a night from ~15 MB to
~2 MB, which is what makes a thousand-night cohort fit in Google Drive.

A pack holds, for chin EMG rows ROWS (all heartbeat-cleaned unless named
otherwise):
    rem        (n_rows, n_rem_epochs, bins_per_epoch) float16
    rem_legs   (n_legs, n_rem_epochs, bins_per_epoch) float16   (RMS)
    rem_epochs (n_rem_epochs,) int32  - epoch numbers in the night
    nrem_1s    (n_rows, n_seconds) float16 - 1 s RMS (rms rows) or mean
               (mav rows) over N2/N3
    hypnogram  (n_epochs,) int8       - the whole night, for context
    epoch_flags {name: (n_epochs,) int8} - e.g. respiratory event, arousal
    meta       json
"""
import json

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)
ROWS = ("rms", "mav", "rms_h", "mav_h", "rms_with_ecg")
F16_MAX = 65000.0


def _one_second(env, env_fs, how):
    n = len(env) // env_fs
    blocks = env[: n * env_fs].reshape(n, env_fs)
    return blocks.mean(axis=1) if how == "mean" else np.sqrt((blocks ** 2).mean(axis=1))


def build_pack(chin, legs, hypnogram, env_fs, epoch_sec, meta, epoch_flags=None):
    """
    chin: dict row -> full-night envelope (float32), for the rows in ROWS that
    exist. legs: list of full-night RMS envelopes. Returns a pack dict.
    """
    per = int(round(env_fs * epoch_sec))
    n_bins = min(len(v) for v in chin.values())
    n_ep = min(len(hypnogram), n_bins // per)
    hyp = np.asarray(hypnogram[:n_ep], dtype=np.int8)
    # the standard rows first, then any others (e.g. noise-ladder copies)
    rows = [r for r in ROWS if r in chin] + [r for r in chin if r not in ROWS]

    rem_ep = np.flatnonzero(hyp == REM_CODE).astype(np.int32)
    idx = rem_ep[:, None] * per + np.arange(per)[None, :]
    rem = np.stack([chin[r][idx] for r in rows]) if len(rem_ep) \
        else np.zeros((len(rows), 0, per), np.float32)
    rem_legs = np.stack([l[idx] for l in legs]) if (legs and len(rem_ep)) \
        else np.zeros((len(legs or []), len(rem_ep), per), np.float32)

    nrem_mask = np.isin(hyp, NREM_CODES).repeat(per)
    nrem_1s = []
    for r in rows:
        env = chin[r][: n_ep * per][nrem_mask]
        nrem_1s.append(_one_second(env, env_fs, "mean" if r.startswith("mav") else "rms"))
    nrem_1s = np.stack(nrem_1s) if nrem_1s else np.zeros((0, 0), np.float32)

    meta = dict(meta, rows=rows, env_fs=env_fs, epoch_sec=epoch_sec,
                n_rem_epochs=int(len(rem_ep)), n_epochs=int(n_ep))
    return {"rem": rem, "rem_legs": rem_legs, "rem_epochs": rem_ep,
            "nrem_1s": nrem_1s, "hypnogram": hyp,
            "epoch_flags": {k: np.asarray(v[:n_ep], np.int8)
                            for k, v in (epoch_flags or {}).items()},
            "meta": meta}


def save_pack(path, pack):
    f16 = lambda a: np.minimum(np.nan_to_num(a, nan=0.0), F16_MAX).astype(np.float16)
    extra = {f"flag_{k}": v for k, v in pack["epoch_flags"].items()}
    tmp = path.with_name(path.stem + ".tmp.npz")
    np.savez_compressed(tmp, rem=f16(pack["rem"]), rem_legs=f16(pack["rem_legs"]),
                        rem_epochs=pack["rem_epochs"], nrem_1s=f16(pack["nrem_1s"]),
                        hypnogram=pack["hypnogram"], meta=json.dumps(pack["meta"]),
                        **extra)
    tmp.replace(path)
    return path


def load_pack(path):
    z = np.load(path, allow_pickle=False)
    return {"rem": z["rem"].astype(np.float32), "rem_legs": z["rem_legs"].astype(np.float32),
            "rem_epochs": z["rem_epochs"], "nrem_1s": z["nrem_1s"].astype(np.float32),
            "hypnogram": z["hypnogram"],
            "epoch_flags": {k[5:]: z[k] for k in z.files if k.startswith("flag_")},
            "meta": json.loads(str(z["meta"]))}


def pack_from_full(d):
    """Convert a full-night derived file (see extract.load_derived) to a pack."""
    meta = d["meta"]
    chin = {r: d[f"chin_{r}"] for r in ROWS if f"chin_{r}" in d}
    legs = [row for row in d["legs"]] if len(d["legs"]) else []
    m = {k: v for k, v in meta.items() if k not in ("env_fs", "epoch_sec")}
    m["unit_known"] = bool(meta.get("chin_info", {}).get("unit_known", True))
    return build_pack(chin, legs, d["hypnogram"], meta["env_fs"], meta["epoch_sec"], m)

In [ ]:
%%writefile /content/rbd-rwa/src/cohorts.py
"""Open polysomnography cohorts, each reduced to the same compact pack.

Every cohort here is downloadable with no application and no credential:

  cap     CAP Sleep Database (PhysioNet, ODC-By). 22 RBD patients plus
          healthy sleepers and other sleep disorders. Italy, 50 Hz mains.
  chal18  PhysioNet/CinC Challenge 2018 training set (PhysioNet, ODC-By).
          994 clinical PSGs from Massachusetts General Hospital with sleep
          stages, respiratory events and arousals, age and sex. USA, 60 Hz.
  hmc     Haaglanden Medisch Centrum sleep staging database (PhysioNet,
          CC BY 4.0). 151 clinical PSGs. The Netherlands, 50 Hz.

External cohorts (paper/preregistration_external.md):

  boas    Bitbrain Open Access Sleep dataset (OpenNeuro ds005555, CC0). 128
          nights of 100 adults from the general population, Micromed SD plus,
          chin EMG at 256 Hz, no ECG. Spain, 50 Hz.
  dodh    Dreem Open Dataset, healthy (Zenodo 15900394, MIT). 25 healthy
          volunteers, Compumedics Siesta, 250 Hz. France, 50 Hz.
  dodo    Dreem Open Dataset, obstructive sleep apnoea (same). 55 patients,
          Somnomedics Somno HD, 250 Hz. Stanford, USA, 60 Hz.

A cohort adapter knows how to list records, fetch one record's files into
scratch, and read from them: chin EMG, ECG, leg EMG (if any), the hypnogram in
30 s epochs from the start of the signal, per-epoch event flags and any
demographics. `extract_pack` turns that into a pack (see pack.py).
"""
import csv
import io
import urllib.request
from pathlib import Path

import numpy as np

from .config import ENV_FS, HARMONIZED_BAND
from .ecg import r_peaks, plausible, artifact_ratio
from .extract import envelope
from .fetch import download_any
from .pack import build_pack

EPOCH_SEC = 30
PHYSIONET = ("https://physionet-open.s3.amazonaws.com/{db}/",
             "https://physionet.org/files/{db}/")


def _mirrors(db):
    return tuple(m.format(db=db) for m in PHYSIONET)


def _read_text(db, name):
    for base in _mirrors(db):
        try:
            return urllib.request.urlopen(base + name, timeout=60).read().decode("latin-1")
        except Exception:
            continue
    raise IOError(f"cannot fetch {db}/{name}")


EEG_PREFERRED = ("C4-M1", "C3-M2", "C4-A1", "C3-A2", "C4A1", "C3A2")


def _central_eeg(channels):
    """A central EEG channel of an EDF (C4 or C3, against a mastoid if possible)."""
    norm = lambda s: s.upper().replace(" ", "").replace("EEG", "")
    for want in EEG_PREFERRED:
        for c in channels:
            if norm(c.label) == want:
                return c
    return next((c for c in channels if norm(c.label).startswith(("C4", "C3"))), None)


# ══════════════════════════════════════════════════════════════════════════════
#  CAP (streamed like the other cohorts, for the noise ladder; the main CAP
#  pipeline goes through run_extract.py and is repacked by run_cohort.py)
# ══════════════════════════════════════════════════════════════════════════════

class CAP:
    name, mains = "cap", 50.0

    def records(self):
        from .fetch import cap_records
        return [r for group in cap_records().values() for r in group]

    def demographics(self):
        return {}

    def fetch(self, rec, scratch):
        from .fetch import fetch_cap
        edf, txt = fetch_cap(rec, scratch)
        return {"edf": edf, "txt": txt}

    def read(self, rec, files):
        from .cap import build_hypnogram
        from .ecg import find_ecg
        from .edf import read_header, read_channel
        from .extract import find_emg, read_source
        hdr = read_header(files["edf"])
        chin_src, _ = find_emg(hdr)
        if chin_src is None:
            raise ValueError(f"{rec}: no chin EMG among {[c.label for c in hdr.channels]}")
        chin, fs, unit_known, label, _ = read_source(hdr, chin_src)
        ecg_src = find_ecg(hdr)
        ecg, fs_e = read_source(hdr, ecg_src)[:2] if ecg_src is not None else (None, None)
        eeg_ch = _central_eeg(hdr.channels)
        eeg, fs_g = read_channel(hdr, eeg_ch)[:2] if eeg_ch else (None, None)
        hyp, _ = build_hypnogram(files["edf"], files["txt"], len(chin) / fs)
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=label,
                    ecg=ecg, fs_ecg=fs_e, legs=[], hypnogram=hyp, flags={}, extra={},
                    eeg=eeg, fs_eeg=fs_g, eeg_label=eeg_ch.label if eeg_ch else None)


# ══════════════════════════════════════════════════════════════════════════════
#  CHALLENGE 2018 (MGH)
# ══════════════════════════════════════════════════════════════════════════════

class Chal18:
    name, db, mains = "chal18", "challenge-2018/1.0.0", 60.0
    STAGES = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}

    def records(self):
        return [r.strip("/").split("/")[-1]
                for r in _read_text(self.db, "training/RECORDS").split()]

    def demographics(self):
        rows = csv.DictReader(io.StringIO(_read_text(self.db, "age-sex.csv")))
        return {r["Record"]: {"age": float(r["Age"]), "sex": r["Sex"]} for r in rows}

    def fetch(self, rec, scratch):
        out = {}
        for ext in ("hea", "mat", "arousal"):
            out[ext] = download_any(f"training/{rec}/{rec}.{ext}",
                                    Path(scratch) / f"{rec}.{ext}", _mirrors(self.db))
        return out

    def read(self, rec, files):
        import wfdb
        base = str(files["hea"])[:-4]
        names = wfdb.rdheader(base).sig_name
        eeg_name = next((n for n in ("C4-M1", "C3-M2") if n in names), None)
        sig = wfdb.rdrecord(base, channel_names=[n for n in ("Chin1-Chin2", "ECG", eeg_name)
                                                 if n in names])
        col = {n: i for i, n in enumerate(sig.sig_name)}

        def channel(name):                        # microvolts, and whether units are known
            if name not in col:
                return None, False
            unit = sig.units[col[name]].lower()
            return sig.p_signal[:, col[name]] * (1e3 if unit == "mv" else 1.0), \
                unit in ("uv", "mv")

        chin, unit_known = channel("Chin1-Chin2")
        ecg, _ = channel("ECG")
        eeg, _ = channel(eeg_name)
        fs = float(sig.fs)
        n_ep = int(len(chin) / fs // EPOCH_SEC)

        ann = wfdb.rdann(base, "arousal")
        hyp = np.full(n_ep, -1, np.int8)
        stage_marks = [(s, self.STAGES[n]) for s, n in zip(ann.sample, ann.aux_note)
                       if n in self.STAGES]
        for (s0, code), nxt in zip(stage_marks, stage_marks[1:] + [(len(chin), None)]):
            a = int(round(s0 / fs / EPOCH_SEC))
            b = int(round(nxt[0] / fs / EPOCH_SEC))
            hyp[a:min(b, n_ep)] = code

        flags = {"resp": np.zeros(n_ep, np.int8), "arousal": np.zeros(n_ep, np.int8)}
        open_ev = {}
        for s, note in zip(ann.sample, ann.aux_note):
            if note.startswith("("):
                open_ev[note[1:]] = s
            elif note.endswith(")") and note[:-1] in open_ev:
                label, s0 = note[:-1], open_ev.pop(note[:-1])
                kind = "resp" if label.startswith("resp_") else \
                    "arousal" if label.startswith("arousal_") else None
                if kind:
                    a = int(s0 / fs // EPOCH_SEC)
                    b = int(s / fs // EPOCH_SEC)
                    flags[kind][a:min(b + 1, n_ep)] = 1
        n_resp = sum(1 for n in ann.aux_note if n.startswith("(resp_"))
        sleep_h = np.isin(hyp, (1, 2, 3, 4)).sum() * EPOCH_SEC / 3600.0
        extra = {"ahi": round(n_resp / sleep_h, 1) if sleep_h > 0 else None}
        return dict(chin=chin, fs=fs, unit_known=unit_known,
                    chin_label="Chin1-Chin2", ecg=ecg, fs_ecg=fs, legs=[],
                    hypnogram=hyp, flags=flags, extra=extra,
                    eeg=eeg, fs_eeg=fs, eeg_label=eeg_name)


# ══════════════════════════════════════════════════════════════════════════════
#  HAAGLANDEN MEDISCH CENTRUM
# ══════════════════════════════════════════════════════════════════════════════

class HMC:
    name, db, mains = "hmc", "hmc-sleep-staging/1.1", 50.0
    STAGES = {"W": 0, "N1": 1, "N2": 2, "N3": 3, "R": 4}
    # v1.1 lists 151 recordings: SN001-SN154 minus SN014, SN064 and SN135,
    # which the release notes withdrew for erroneous, unfixable signal data.

    def records(self):
        return [Path(x).stem for x in _read_text(self.db, "RECORDS").split()]

    def demographics(self):
        return {}                       # only group-level age/sex are published

    def fetch(self, rec, scratch):
        return {"edf": download_any(f"recordings/{rec}.edf", Path(scratch) / f"{rec}.edf",
                                    _mirrors(self.db)),
                "txt": download_any(f"recordings/{rec}_sleepscoring.txt",
                                    Path(scratch) / f"{rec}_sleepscoring.txt",
                                    _mirrors(self.db))}

    def read(self, rec, files):
        from .edf import read_header, read_channel
        hdr = read_header(files["edf"])
        chin_ch = next(c for c in hdr.channels if "CHIN" in c.label.upper())
        ecg_ch = next((c for c in hdr.channels if "ECG" in c.label.upper()), None)
        eeg_ch = _central_eeg(hdr.channels)
        chin, fs, unit_known = read_channel(hdr, chin_ch)
        ecg, fs_e = (read_channel(hdr, ecg_ch)[:2]) if ecg_ch else (None, None)
        eeg, fs_g = (read_channel(hdr, eeg_ch)[:2]) if eeg_ch else (None, None)
        n_ep = int(len(chin) / fs // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        text = Path(files["txt"]).read_text(encoding="latin-1").splitlines()
        for line in text[1:]:
            parts = [p.strip() for p in line.split(",")]
            if len(parts) < 5 or not parts[4].startswith("Sleep stage"):
                continue
            code = self.STAGES.get(parts[4].split()[-1])
            e = int(round(float(parts[2]) / EPOCH_SEC))
            if code is not None and 0 <= e < n_ep:
                hyp[e] = code
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=chin_ch.label,
                    ecg=ecg, fs_ecg=fs_e, legs=[], hypnogram=hyp, flags={}, extra={},
                    eeg=eeg, fs_eeg=fs_g, eeg_label=eeg_ch.label if eeg_ch else None)


class HMCDegraded(HMC):
    """
    HMC recordings re-acquired, in software, the way the MGH chin EMG was:
    optionally scaled by `gain`, resampled to `fs_out` and quantised to
    `lsb_uv` microvolt steps. MGH stores chin EMG at 200 Hz with 1 uV
    resolution, and its quiet-REM level (~0.6 uV) is below that step; HMC's is
    ~1.5 uV. Scaling HMC down to the MGH level first puts the signal where the
    step size can matter. Comparing HMC with its degraded copies isolates how
    much of a site difference that hardware alone can create.
    """
    def __init__(self, name, fs_out=None, lsb_uv=None, gain=None):
        self.name, self.fs_out, self.lsb_uv, self.gain = name, fs_out, lsb_uv, gain

    def read(self, rec, files):
        d = super().read(rec, files)
        x, fs = d["chin"], d["fs"]
        if self.gain:
            x = x * self.gain
        if self.fs_out and self.fs_out != fs:
            from fractions import Fraction
            from scipy.signal import resample_poly
            f = Fraction(int(self.fs_out), int(fs)).limit_denominator(1000)
            x = resample_poly(x, f.numerator, f.denominator)
            fs = float(self.fs_out)
        if self.lsb_uv:
            x = np.round(x / self.lsb_uv) * self.lsb_uv
        d.update(chin=x, fs=fs)
        return d


class Chal18Noisy(Chal18):
    """
    MGH recordings with white noise added to the chin EMG, the reverse test:
    MGH's quiet-REM level (~0.6 uV) raised to HMC's (~1.5 uV), as a noisier
    amplifier or higher electrode impedance would. The noise is seeded by
    record name, so reruns are identical.
    """
    def __init__(self, name, noise_uv):
        self.name, self.noise_uv = name, noise_uv

    def read(self, rec, files):
        import zlib
        d = super().read(rec, files)
        rng = np.random.default_rng(zlib.crc32(rec.encode()))
        d["chin"] = d["chin"] + rng.normal(0.0, self.noise_uv, len(d["chin"]))
        return d


# ══════════════════════════════════════════════════════════════════════════════
#  EXTERNAL COHORTS
# ══════════════════════════════════════════════════════════════════════════════

class BOAS:
    """
    Bitbrain Open Access Sleep dataset (OpenNeuro ds005555, CC0): adults from
    the general population, one or more nights each (participants.tsv 'pid'),
    Micromed SD plus PSG, bipolar chin EMG at 256 Hz, consensus AASM stages of
    three scorers in the events file (0 W, 1 N1, 2 N2, 3 N3, 4 REM; 8 and -2 =
    disconnection / artefact). There is no ECG channel: heartbeats are found
    on the chin EMG itself (see `_beats`).
    """
    name, mains = "boas", 50.0
    BASE = "https://s3.amazonaws.com/openneuro.org/ds005555/"

    def _participants(self):
        txt = urllib.request.urlopen(self.BASE + "participants.tsv", timeout=60).read().decode()
        return list(csv.DictReader(io.StringIO(txt), delimiter="\t"))

    def records(self):
        return [r["participant_id"] for r in self._participants()]

    def demographics(self):
        num = lambda v: float(v) if v not in (None, "", "n/a") else None
        return {r["participant_id"]: {"age": num(r.get("age")), "sex": r.get("sex"),
                                      "pid": r.get("pid")} for r in self._participants()}

    def fetch(self, rec, scratch):
        from .fetch import download
        stem = f"{rec}/eeg/{rec}_task-Sleep_acq-psg"
        return {"edf": download(self.BASE + stem + "_eeg.edf", Path(scratch) / f"{rec}.edf"),
                "tsv": download(self.BASE + stem + "_events.tsv", Path(scratch) / f"{rec}.tsv")}

    def read(self, rec, files):
        from .edf import read_header, read_channel
        hdr = read_header(files["edf"])
        chin_ch = next(c for c in hdr.channels if c.label.upper().endswith("EMG"))
        chin, fs, unit_known = read_channel(hdr, chin_ch)
        eeg_ch = next((c for c in hdr.channels if c.label.upper().endswith(("C4", "C3"))), None)
        eeg, fs_g = read_channel(hdr, eeg_ch)[:2] if eeg_ch else (None, None)
        n_ep = int(len(chin) / fs // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        with open(files["tsv"], encoding="utf-8") as f:
            for r in csv.DictReader(f, delimiter="\t"):
                e, code = int(round(float(r["onset"]) / EPOCH_SEC)), int(float(r["stage_hum"]))
                if 0 <= e < n_ep and code in (0, 1, 2, 3, 4):
                    hyp[e] = code
        return dict(chin=chin, fs=fs, unit_known=unit_known, chin_label=chin_ch.label,
                    ecg=None, fs_ecg=None, beats_from_chin=True, legs=[], hypnogram=hyp,
                    flags={}, extra={}, eeg=eeg, fs_eeg=fs_g,
                    eeg_label=eeg_ch.label if eeg_ch else None)


class _RangeReader:
    """Byte ranges of one URL, with retries (Zenodo serves ~1.6 MB/s per connection,
    so large members are fetched as many ranges in parallel)."""
    def __init__(self, url):
        self.url = url

    def get(self, a, b, tries=8):
        import time
        for i in range(tries):
            try:
                req = urllib.request.Request(self.url, headers={"Range": f"bytes={a}-{b - 1}"})
                data = urllib.request.urlopen(req, timeout=300).read()
                if len(data) == b - a:
                    return data
            except Exception:
                pass
            time.sleep(min(60, 3 * 2 ** i))
        raise IOError(f"range {a}-{b} of {self.url} failed")


class _RangeFile(io.RawIOBase):
    """A seekable file over HTTP ranges - enough for zipfile to read a central directory."""
    def __init__(self, url, block=1 << 16):
        self.r, self.pos, self.block = _RangeReader(url), 0, block
        head = urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=60)
        self.size = int(head.headers["Content-Length"])
        self.c0, self.c = -1, b""

    def seekable(self):
        return True

    def readable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, off, whence=0):
        self.pos = off if whence == 0 else (self.pos + off if whence == 1 else self.size + off)
        return self.pos

    def read(self, n=-1):
        n = self.size - self.pos if n is None or n < 0 else min(n, self.size - self.pos)
        if n <= 0:
            return b""
        a, b = self.pos, self.pos + n
        if not (self.c0 <= a and b <= self.c0 + len(self.c)):
            self.c0 = a
            self.c = self.r.get(a, min(self.size, a + max(n, self.block)))
        self.pos = b
        return self.c[a - self.c0: b - self.c0]


class DOD:
    """
    Dreem Open Datasets (Zenodo 15900394, MIT licence): one h5 file per night
    inside a zip per dataset, with signals/eeg/*, signals/emg/* and ECG in uV at
    250 Hz and a consensus hypnogram of five scorers (-1 not scored, 0 W, 1 N1,
    2 N2, 3 N3, 4 REM). A record's member is fetched with parallel byte-range
    requests and inflated locally.
    """
    URL = "https://zenodo.org/records/15900394/files/{}.zip?download=1"
    UNSCORED = {"5d8e93a5-d26a-436d-bc2c-e9527dc92cbf"}   # no consensus (release notes)
    FS = 250.0

    def __init__(self, name, mains, threads=8, chunk=32 << 20):
        self.name, self.mains, self.threads, self.chunk = name, mains, threads, chunk
        self._members = None

    def members(self):
        if self._members is None:
            import zipfile
            z = zipfile.ZipFile(_RangeFile(self.URL.format(self.name)))
            self._members = {Path(i.filename).stem: i for i in z.infolist()
                             if i.filename.endswith(".h5") and "__MACOSX" not in i.filename
                             and i.file_size > 1e6}
        return self._members

    def records(self):
        return sorted(r for r in self.members() if r not in self.UNSCORED)

    def demographics(self):
        return {}

    def fetch(self, rec, scratch):
        import struct
        import zlib
        from concurrent.futures import ThreadPoolExecutor
        info = self.members()[rec]
        rr = _RangeReader(self.URL.format(self.name))
        head = rr.get(info.header_offset, info.header_offset + 30)
        n, m = struct.unpack("<HH", head[26:30])
        start = info.header_offset + 30 + n + m
        parts = [(a, min(a + self.chunk, start + info.compress_size))
                 for a in range(start, start + info.compress_size, self.chunk)]
        out = Path(scratch) / f"{rec}.h5"
        inflate = zlib.decompressobj(-15) if info.compress_type == 8 else None
        with ThreadPoolExecutor(self.threads) as ex, open(out, "wb") as f:
            for i in range(0, len(parts), self.threads):          # in order, bounded memory
                for data in ex.map(lambda p: rr.get(*p), parts[i: i + self.threads]):
                    f.write(inflate.decompress(data) if inflate else data)
            if inflate:
                f.write(inflate.flush())
        if out.stat().st_size != info.file_size:
            raise IOError(f"{rec}: inflated {out.stat().st_size} of {info.file_size} bytes")
        return {"h5": out}

    def read(self, rec, files):
        import h5py
        is_ecg = lambda k: "ECG" in k.upper() or "EKG" in k.upper()
        with h5py.File(files["h5"], "r") as h:
            sig = h["signals"]
            keys = [k for k in sig["emg"].keys() if not is_ecg(k)]
            emg_name = next((k for k in keys if any(t in k.upper() for t in
                                                    ("CHIN", "EMG", "MENT", "SUBM"))), keys[0])
            chin = np.asarray(sig["emg"][emg_name][:], dtype=np.float64)
            ecg = None
            for group in sig.values():
                if isinstance(group, h5py.Group):
                    for k in group.keys():
                        if is_ecg(k):
                            ecg = np.asarray(group[k][:], dtype=np.float64)
            eeg_name = next((k for k in ("C3_M2", "C4_M1") if k in sig["eeg"]), None)
            eeg = np.asarray(sig["eeg"][eeg_name][:], dtype=np.float64) if eeg_name else None
            stages = np.asarray(h["hypnogram"][:])
        n_ep = int(len(chin) / self.FS // EPOCH_SEC)
        hyp = np.full(n_ep, -1, np.int8)
        k = min(n_ep, len(stages))
        hyp[:k] = np.where(np.isin(stages[:k], (0, 1, 2, 3, 4)), stages[:k], -1)
        return dict(chin=chin, fs=self.FS, unit_known=True, chin_label=f"emg/{emg_name}",
                    ecg=ecg, fs_ecg=self.FS if ecg is not None else None, legs=[],
                    hypnogram=hyp, flags={}, extra={}, eeg=eeg, fs_eeg=self.FS,
                    eeg_label=eeg_name)


# 0.39 = median quiet-REM level MGH / HMC (0.59 / 1.52 uV). 1.5 uV of white noise
# at 200 Hz leaves ~1.4 uV in the 10-95 Hz band: sqrt(0.59^2 + 1.4^2) ~ 1.52 uV.
COHORTS = {"chal18": Chal18(), "hmc": HMC(),
           "hmc_q1uv": HMCDegraded("hmc_q1uv", lsb_uv=1.0),
           "hmc_mgh": HMCDegraded("hmc_mgh", fs_out=200, lsb_uv=1.0),
           "hmc_low": HMCDegraded("hmc_low", gain=0.39),
           "hmc_low_mgh": HMCDegraded("hmc_low_mgh", fs_out=200, lsb_uv=1.0, gain=0.39),
           "chal18_noise": Chal18Noisy("chal18_noise", noise_uv=1.5),
           "cap": CAP(),
           "boas": BOAS(), "dodh": DOD("dodh", mains=50.0), "dodo": DOD("dodo", mains=60.0)}


# ══════════════════════════════════════════════════════════════════════════════
#  COMMON: raw signals -> pack
# ══════════════════════════════════════════════════════════════════════════════

def _beats(d):
    """R-peak times (s) from the record's ECG, or none if they do not look real.
    Records without ECG may ask for the chin EMG itself to be searched for the
    heartbeat it carries (same detector, same plausibility test)."""
    if d.get("ecg") is not None:
        sig, fs = d["ecg"], d["fs_ecg"]
    elif d.get("beats_from_chin"):
        sig, fs = d["chin"], d["fs"]
    else:
        return np.empty(0)
    beats = r_peaks(sig, fs)
    return beats if plausible(beats, len(sig) / fs) else np.empty(0)


def extract_pack(cohort, rec, d, demo=None):
    """Envelopes, heartbeat removal and packing for one record's signals."""
    beats = _beats(d)
    x, fs, mains = d["chin"], d["fs"], cohort.mains
    rms, mav, band = envelope(x, fs, mains=mains, beats_s=beats)
    rms_h, mav_h, band_h = envelope(x, fs, band=HARMONIZED_BAND, mains=mains, beats_s=beats)
    rms_raw, _, _ = envelope(x, fs, mains=mains)
    legs = [envelope(l, fs_l, mains=mains)[0] for l, fs_l in d.get("legs", [])]
    meta = {
        "record": rec, "cohort": cohort.name, "chin_channel": d["chin_label"],
        "chin_fs": fs, "band": list(band), "band_harmonized": list(band_h),
        "unit_known": bool(d["unit_known"]), "mains_notch": mains,
        "ecg": {"beats": int(len(beats)),
                "artifact_ratio_before": round(artifact_ratio(rms_raw, ENV_FS, beats), 3),
                "artifact_ratio_after": round(artifact_ratio(rms, ENV_FS, beats), 3)},
        "duration_s": round(len(x) / fs, 1),
        **(demo or {}), **d.get("extra", {}),
    }
    chin = {"rms": rms, "mav": mav, "rms_h": rms_h, "mav_h": mav_h, "rms_with_ecg": rms_raw}
    return build_pack(chin, legs, d["hypnogram"], ENV_FS, EPOCH_SEC, meta, d.get("flags"))


# ══════════════════════════════════════════════════════════════════════════════
#  THE NOISE LADDER: the same night at several noise floors
# ══════════════════════════════════════════════════════════════════════════════

LADDER_UV = (0.5, 1.0, 1.5, 2.0, 3.0)   # in-band RMS of the added noise, uV
TARGETS_UV = (1.5, 2.0)                 # quiet-REM level after harmonisation, uV
EEG_BAND = (40.0, 90.0)                 # where EEG is mostly the system's own noise


def quiet_level(rms, hypnogram, env_fs=ENV_FS):
    """5th percentile of 1 s RMS over REM: the background the relative
    criteria are multiples of ('p5_rem' in rwa.py)."""
    from .rwa import mini_epoch_rms
    per = int(round(env_fs * EPOCH_SEC))
    hyp = np.asarray(hypnogram)[: len(rms) // per]
    rem = np.flatnonzero(hyp == 4)
    if len(rem) == 0:
        return float("nan")
    r, _ = mini_epoch_rms(rms[(rem[:, None] * per + np.arange(per)[None, :]).reshape(-1)],
                          env_fs, 1.0)
    return float(np.percentile(r, 5)) if len(r) >= 10 else float("nan")


def eeg_floor(eeg, fs, hypnogram, mains, band=EEG_BAND):
    """The 40-90 Hz floor (5th percentile of 1 s RMS) of an EEG channel in N2/N3
    and in REM: an estimate of the recording system's own noise, measured on a
    channel other than the chin."""
    from .extract import bandpass, bin_envelope
    if eeg is None or not fs or fs / 2.0 - 5.0 < band[0] + 10.0:
        return {}
    x, (lo, hi) = bandpass(eeg, fs, band, mains)
    sec, _ = bin_envelope(x, fs, env_fs=1)
    n = min(len(sec), len(hypnogram) * EPOCH_SEC)
    stage = np.repeat(np.asarray(hypnogram), EPOCH_SEC)[:n]
    out = {"band": [lo, hi]}
    for name, codes in (("nrem", (2, 3)), ("rem", (4,))):
        v = sec[:n][np.isin(stage, codes)]
        out[f"floor_{name}_uv"] = round(float(np.percentile(v, 5)), 4) if len(v) >= 60 else None
    return out


def extract_ladder_pack(cohort, rec, d, demo=None):
    """
    The night's chin EMG as recorded (native band, heartbeat removed) and with
    white noise added in band: at each LADDER_UV level, and at the level that
    raises its quiet-REM level to each TARGETS_UV. Filtering and heartbeat
    removal are linear, so the signal and one unit-RMS noise track (seeded by
    record name) are filtered once and every copy is their sum - the same noise
    draw scaled, i.e. common random numbers across levels.
    """
    import zlib
    from .ecg import subtract_heartbeat
    from .extract import bandpass, bin_envelope
    beats = _beats(d)
    x, fs, mains, hyp = d["chin"], d["fs"], cohort.mains, d["hypnogram"]
    xf, band = bandpass(x, fs, mains=mains)
    if len(beats):
        xf = subtract_heartbeat(xf, fs, beats)
    rms, mav = bin_envelope(xf, fs)
    chin = {"rms": rms, "mav": mav}
    level = quiet_level(rms, hyp)
    ladder = {}
    if d["unit_known"] and np.isfinite(level):
        rng = np.random.default_rng(zlib.crc32(rec.encode()))
        nf, _ = bandpass(rng.normal(0.0, 1.0, len(x)), fs, mains=mains)
        edge = int(5 * fs)
        nf /= np.sqrt(np.mean(nf[edge:-edge] ** 2))
        for u in LADDER_UV:
            r, m = bin_envelope(xf + u * nf, fs)
            chin[f"rms_n{round(u * 100):03d}"], chin[f"mav_n{round(u * 100):03d}"] = r, m
            ladder[f"n{round(u * 100):03d}"] = {"added_uv": u, "level_uv": round(quiet_level(r, hyp), 3)}
        for t in TARGETS_UV:
            # The 5th percentile catches seconds where the noise itself dips, so
            # sqrt(t^2 - level^2) undershoots; rescale until within 1% of t.
            s, r, m, got = float(np.sqrt(max(t * t - level * level, 0.0))), rms, mav, level
            for _ in range(4):
                if s <= 0 or abs(got - t) <= 0.01 * t:
                    break
                r, m = bin_envelope(xf + s * nf, fs)
                got = quiet_level(r, hyp)
                if abs(got - t) > 0.01 * t:
                    s *= np.sqrt((t * t - level * level) / max(got * got - level * level, 1e-9))
            chin[f"rms_h{round(t * 100):03d}"], chin[f"mav_h{round(t * 100):03d}"] = r, m
            ladder[f"h{round(t * 100):03d}"] = {"added_uv": round(s, 3), "level_uv": round(got, 3)}
        del nf
    meta = {
        "record": rec, "cohort": cohort.name, "chin_channel": d["chin_label"],
        "chin_fs": fs, "band": list(band), "unit_known": bool(d["unit_known"]),
        "mains_notch": mains, "duration_s": round(len(x) / fs, 1),
        "level_uv": round(level, 4) if np.isfinite(level) else None, "ladder": ladder,
        "ecg": {"beats": int(len(beats))},
        "eeg": dict(eeg_floor(d.get("eeg"), d.get("fs_eeg"), hyp, mains),
                    channel=d.get("eeg_label")),
        **(demo or {}), **d.get("extra", {}),
    }
    return build_pack(chin, [], hyp, ENV_FS, EPOCH_SEC, meta, d.get("flags"))

In [ ]:
%%writefile /content/rbd-rwa/src/rwa.py
"""The published methods for quantifying REM sleep without atonia (RWA).

Each method reduces one night to one or more percentages (or an index) that are
compared against a diagnostic cut-off. They differ in epoch length, amplitude
thresholds, what counts as "background", and which cut-off applies:

  Montreal  Montplaisir et al., Mov Disord 2010;25:2044 (after Lapierre &
            Montplaisir, Neurology 1992). Tonic: 20 s epochs, >50% of the
            epoch at >= 2x background or > 10 uV; positive at >= 30%.
            Phasic: 2 s mini-epochs containing bursts > 4x background;
            positive at >= 15%.
  SINBAR    Frauscher et al., Sleep 2012;35:835. Phasic bursts 0.1-5 s at
            > 2x background; tonic as Montreal but on 30 s epochs; "any"
            activity in 3 s mini-epochs. Mentalis-only "any" cut-off 18.2%
            (100% specificity). The better-known 32% needs bilateral flexor
            digitorum superficialis EMG, which most recordings lack.
  AASM      AASM scoring manual. Tonic epoch: >= 50% of a 30 s epoch above
            the minimum NREM amplitude. Phasic epoch: >= 5 of its ten 3 s
            mini-epochs contain 0.1-5 s bursts >= 4x background. The manual
            gives no numeric cut-off for "excessive".
  RAI       REM Atonia Index, Ferri et al., J Sleep Res 2008 and Sleep Med
            2010;11:947 ("improved" computation: subtract the minimum of the
            surrounding 60 one-second mini-epochs). RAI < 0.8 is the commonly
            used abnormal range.

Five COMPUTERISED indices (RAI among them) were compared on the same 146
recordings by Cesari et al., Sleep 2018;41(10):zsy133. This module puts the
visual-criteria methods and the RAI on one code path so they can be applied
unchanged to large clinical cohorts.

OPERATIONALISING "BACKGROUND". The visual methods judge background by eye.
Here it is an explicit, named choice (`baseline_amplitude`), because it
changes every downstream percentage. Values still listed in a method's
`verify` tuple are secondary-source readings not yet checked against the
original paper.
"""
from dataclasses import dataclass, field

import numpy as np

REM_CODE = 4
NREM_CODES = (2, 3)
# The 5th percentile of 1 s RMS values of band-limited white noise, relative to
# its RMS (0.910-0.913 for 10-95/100 Hz at 200, 256 and 512 Hz, simulated).
NOISE_P5_RATIO = 0.91


# ══════════════════════════════════════════════════════════════════════════════
#  SHARED MACHINERY
# ══════════════════════════════════════════════════════════════════════════════

def mini_epoch_rms(env, env_fs, seconds, how="rms"):
    """
    Split an envelope into consecutive mini-epochs and summarise each: "rms"
    for an RMS envelope (gives the RMS of the underlying signal), "mean" for a
    mean-rectified envelope (gives its average rectified amplitude).
    """
    w = int(round(env_fs * seconds))
    n = len(env) // w
    if n == 0:
        return np.empty(0), w
    blocks = env[: n * w].reshape(n, w)
    if how == "mean":
        return blocks.mean(axis=1), w
    return np.sqrt((blocks ** 2).mean(axis=1)), w


def stage_mask_at(env_fs, hypnogram, epoch_sec, codes, n_samples):
    """Boolean mask over envelope samples for the given sleep-stage codes."""
    per = int(round(env_fs * epoch_sec))
    mask = np.zeros(n_samples, dtype=bool)
    for i, code in enumerate(hypnogram):
        if code in codes:
            a = i * per
            if a >= n_samples:
                break
            mask[a: min(a + per, n_samples)] = True
    return mask


def baseline_amplitude(env, env_fs, hypnogram, epoch_sec, how, seconds=1.0):
    """
    Background EMG level against which activity is judged, from 1 s windows.

    how = "min_nrem" : lowest window in NREM (the AASM wording; a literal
                       minimum is set by one window, so unstable)
          "p5_nrem"  : 5th percentile in NREM (robust reading of the same)
          "min_rem" / "p5_rem" : the same within REM itself, i.e. the atonic
                       floor of REM - closer to what a visual scorer calls
                       "background" when judging a REM epoch
    """
    codes = NREM_CODES if how.endswith("nrem") else (REM_CODE,)
    m = stage_mask_at(env_fs, hypnogram, epoch_sec, codes, len(env))
    if m.sum() < env_fs * seconds * 10:
        return np.nan
    r, _ = mini_epoch_rms(env[m], env_fs, seconds)
    if len(r) == 0:
        return np.nan
    return float(r.min() if how.startswith("min") else np.percentile(r, 5))


def runs_above(env, env_fs, threshold, min_s=0.1, max_s=None):
    """
    Boolean array marking samples inside runs of consecutive supra-threshold
    samples lasting at least `min_s` (and at most `max_s`, if given). The upper
    bound separates phasic bursts from sustained tonic activity.
    """
    above = env > threshold
    if not above.any():
        return np.zeros(len(env), dtype=bool)
    edges = np.diff(np.r_[0, above.astype(np.int8), 0])
    starts = np.flatnonzero(edges == 1)
    stops = np.flatnonzero(edges == -1)
    dur = (stops - starts) / env_fs
    keep = dur >= min_s
    if max_s is not None:
        keep &= dur <= max_s
    # runs never touch (a stop is always followed by a sample below threshold),
    # so +1 at each kept start and -1 at its stop, summed, marks the runs
    step = np.zeros(len(env) + 1, dtype=np.int32)
    step[starts[keep]] += 1
    step[stops[keep]] -= 1
    return np.cumsum(step[:-1]) > 0


def _blocks(mask_idx, width):
    """Consecutive, non-overlapping blocks of `width` indices from an index list."""
    n = len(mask_idx) // width
    return mask_idx[: n * width].reshape(n, width)


# ══════════════════════════════════════════════════════════════════════════════
#  METHOD DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class Method:
    name: str
    tonic_epoch_s: float          # tonic scored on epochs of this length
    tonic_mult: float             # tonic threshold, multiples of background
    tonic_abs_uv: float = None    # "... or > X uV": an absolute alternative
    tonic_baseline: str = "p5_rem"
    phasic_mini_s: float = 3.0    # phasic scored on mini-epochs of this length
    phasic_mult: float = 4.0
    phasic_max_s: float = 5.0     # longest burst still counted as phasic
    phasic_baseline: str = "p5_rem"
    any_mini_s: float = None      # SINBAR "any" density on these mini-epochs
    any_mult: float = None
    phasic_epoch_rule: int = None # AASM: >= this many phasic mini-epochs / 30 s
    cutoffs: dict = field(default_factory=dict)   # metric -> % (positive if >=)
    verify: tuple = field(default_factory=tuple)
    # How "k x background" becomes an amplitude threshold (see `threshold`):
    # "relative" as published, "floor" corrected for the noise floor, "clamp"
    # with the background never taken below ref_floor_uv.
    mode: str = "relative"
    ref_floor_uv: float = None
    guard_mult: float = 1.5


def threshold(method, k, b):
    """
    The amplitude (uV RMS) that "activity above k x background" means for a
    night whose background is b.

    relative  k b, as published. Because b is mostly the recording's own noise
              floor, the muscle activity this requires, sqrt(k^2 - 1) b, rises
              and falls with the noise of the recording system.
    floor     max(sqrt(n^2 + (k^2 - 1) B^2), g b) with B = ref_floor_uv and
              n = b / NOISE_P5_RATIO the noise RMS that b implies: the muscle
              activity required is fixed at what the published rule requires
              on a system whose background is B, and the night's noise is
              subtracted (in power) instead of multiplied. The guard g b keeps
              noise alone from being counted: with g = 1.5, white noise puts a
              0.1 s run above threshold in < 1% of mini-epochs (0 for tonic).
    clamp     k max(b, B): the published rule with the background never taken
              to be lower than B (an absolute floor, no subtraction).
    """
    if method.mode == "floor":
        B, n = method.ref_floor_uv, b / NOISE_P5_RATIO
        return max(float(np.sqrt(n * n + (k * k - 1.0) * B * B)), method.guard_mult * b)
    if method.mode == "clamp":
        return k * max(b, method.ref_floor_uv)
    return k * b


def floor_corrected(method, ref_uv, mode="floor", guard=1.5):
    """A method's floor-corrected (or clamped) version, same name and cut-offs."""
    import dataclasses
    return dataclasses.replace(method, mode=mode, ref_floor_uv=float(ref_uv),
                               guard_mult=guard)


METHODS = {
    "montreal": Method(
        "Montreal", tonic_epoch_s=20.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=2.0, phasic_mult=4.0, phasic_max_s=15.0,
        cutoffs={"tonic_pct": 30.0, "phasic_pct": 15.0},
        verify=("phasic_max_s",),     # 10 s vs 15 s differs between sources
    ),
    "sinbar": Method(
        "SINBAR", tonic_epoch_s=30.0, tonic_mult=2.0, tonic_abs_uv=10.0,
        phasic_mini_s=3.0, phasic_mult=2.0, phasic_max_s=5.0,
        any_mini_s=3.0, any_mult=2.0,
        cutoffs={"any_pct": 18.2},    # mentalis only, 3 s mini-epochs
    ),
    "aasm": Method(
        "AASM", tonic_epoch_s=30.0, tonic_mult=1.0, tonic_baseline="p5_nrem",
        phasic_mini_s=3.0, phasic_mult=4.0, phasic_max_s=5.0,
        phasic_epoch_rule=5,
        cutoffs={},                   # the manual gives none
    ),
}


# ══════════════════════════════════════════════════════════════════════════════
#  SCORING A NIGHT (from a pack: REM epochs at full resolution, see pack.py)
# ══════════════════════════════════════════════════════════════════════════════

# band -> (RMS row, mean-rectified row) of the pack. Any other name X selects
# the rows "rms_X" and "mav_X" (the noise-ladder copies, see cohorts.py).
BANDS = {"native": ("rms", "mav"), "harmonized": ("rms_h", "mav_h"),
         "native_with_ecg": ("rms_with_ecg", None)}


def _select(pack, band, exclude=()):
    """REM epoch numbers and (n_epochs, bins) arrays for a band, minus any
    REM epoch carrying one of the `exclude` flags."""
    rows = pack["meta"]["rows"]
    rk, mk = BANDS.get(band, (f"rms_{band}", f"mav_{band}"))
    ep = pack["rem_epochs"]
    keep = np.ones(len(ep), bool)
    for name in exclude or ():
        flag = pack["epoch_flags"].get(name)
        if flag is not None and len(ep):
            keep &= flag[ep] == 0
    rms = pack["rem"][rows.index(rk)][keep] if rk in rows else None
    mav = pack["rem"][rows.index(mk)][keep] if mk in rows else None
    nrem = pack["nrem_1s"][rows.index(rk)] if (rk in rows and pack["nrem_1s"].size) \
        else np.empty(0)
    return ep[keep], rms, mav, nrem


def _segments(ep):
    """(start, stop) positions of runs of consecutive epoch numbers."""
    if len(ep) == 0:
        return []
    brk = np.flatnonzero(np.diff(ep) != 1) + 1
    return list(zip(np.r_[0, brk], np.r_[brk, len(ep)]))


def _runs(rem2d, ep, env_fs, thr, min_s, max_s):
    """runs_above within each stretch of consecutive REM epochs, so that a
    burst never bridges the gap between two separate REM periods."""
    out = np.zeros(rem2d.shape, bool)
    for a, b in _segments(ep):
        out[a:b] = runs_above(rem2d[a:b].reshape(-1), env_fs, thr, min_s,
                              max_s).reshape(b - a, -1)
    return out


def _background(rem2d, nrem_1s, env_fs, how):
    """
    Background from 1 s RMS values of REM or of N2/N3 (see baseline_amplitude).
    how = "<stat>_<stage>" with stat "min" or "pNN" (NN-th percentile) and stage
    "rem" or "nrem", e.g. "p5_rem" (the default), "p25_rem", "min_nrem".
    """
    stat, stage = how.split("_", 1)
    if stage == "nrem":
        vals = nrem_1s
    else:
        vals, _ = mini_epoch_rms(rem2d.reshape(-1), env_fs, 1.0)
    if len(vals) < 10:
        return np.nan
    if stat == "min":
        return float(vals.min())
    return float(np.percentile(vals, float(stat[1:])))


def _mini(mask2d, env_fs, seconds):
    """Split concatenated REM into mini-epochs; True where any sample is."""
    flat = mask2d.reshape(-1)
    w = int(round(env_fs * seconds))
    n = len(flat) // w
    return flat[: n * w].reshape(n, w)


def score_night(pack, method: Method, band="native", exclude=()):
    """Apply one method to one night. Returns a dict of its measures."""
    meta = pack["meta"]
    env_fs, epoch_sec = meta["env_fs"], meta["epoch_sec"]
    ep, env, _, nrem = _select(pack, band, exclude)
    out = {"method": method.name,
           "rem_minutes": round(len(ep) * epoch_sec / 60.0, 1)}
    if env is None:
        out["error"] = f"band {band} not stored"
        return out
    if len(ep) < 2:
        out["error"] = "less than one minute of REM"
        return out
    if method.mode != "relative" and not meta.get("unit_known", True):
        out["error"] = "floor correction needs the EMG in physical units"
        return out

    bt = _background(env, nrem, env_fs, method.tonic_baseline)
    bp = _background(env, nrem, env_fs, method.phasic_baseline)
    if not (np.isfinite(bt) and bt > 0 and np.isfinite(bp) and bp > 0):
        out["error"] = "background could not be estimated"
        return out

    # ── tonic ────────────────────────────────────────────────────────────────
    thr_t = threshold(method, method.tonic_mult, bt)
    if method.tonic_abs_uv is not None and meta.get("unit_known", True):
        thr_t = min(thr_t, method.tonic_abs_uv)      # "2x background OR > 10 uV"
    above = _mini(env > thr_t, env_fs, method.tonic_epoch_s)
    tonic = above.mean(axis=1) > 0.5

    # ── phasic ───────────────────────────────────────────────────────────────
    thr_p = threshold(method, method.phasic_mult, bp)
    burst = _runs(env, ep, env_fs, thr_p, 0.1, method.phasic_max_s)
    phasic = _mini(burst, env_fs, method.phasic_mini_s).any(axis=1)

    out.update(background_tonic_uV=round(bt, 3), background_phasic_uV=round(bp, 3),
               tonic_threshold_uV=round(thr_t, 3), phasic_threshold_uV=round(thr_p, 3),
               tonic_pct=round(100.0 * tonic.mean(), 2) if len(tonic) else np.nan,
               phasic_pct=round(100.0 * phasic.mean(), 2) if len(phasic) else np.nan)

    # ── AASM: an epoch is phasic when >= 5 of its ten 3 s mini-epochs are ──
    if method.phasic_epoch_rule:
        w = int(round(env_fs * method.phasic_mini_s))
        per = burst.shape[1]
        minis = burst[:, : (per // w) * w].reshape(len(burst), -1, w).any(axis=2)
        hits = minis.sum(axis=1) >= method.phasic_epoch_rule
        out["phasic_epoch_pct"] = round(100.0 * hits.mean(), 2)

    # ── SINBAR "any": tonic, phasic or both, above the same amplitude ──────
    if method.any_mini_s:
        active = _runs(env, ep, env_fs, threshold(method, method.any_mult, bp), 0.1, None)
        out["any_pct"] = round(100.0 * _mini(active, env_fs, method.any_mini_s)
                               .any(axis=1).mean(), 2)

    hits = [out.get(k, np.nan) >= v for k, v in method.cutoffs.items()
            if np.isfinite(out.get(k, np.nan))]
    out["positive"] = bool(any(hits)) if hits else None
    return out


def rem_atonia_index(pack, band="native", exclude=(), bin_width=1.0,
                     n_bins=20, noise="moving_min", window_s=60):
    """
    REM Atonia Index. Average rectified amplitude per 1 s mini-epoch of REM,
    noise-corrected, then binned by amplitude:

        RAI = A0 / (1 - A1)

    with A0 the fraction of mini-epochs <= 1 uV and A1 the fraction in
    (1, 2] uV. Runs from 0 (no atonia) to 1 (complete atonia). Uses the
    MEAN-RECTIFIED rows, which must be in microvolts: the bins are absolute.

    noise = "moving_min": subtract the minimum of the `window_s` mini-epochs
            surrounding each one (Ferri et al. 2010, "improved" RAI);
            "global_min": subtract the night's single minimum (2008 version).
    """
    meta = pack["meta"]
    ep, _, env_mav, _ = _select(pack, band, exclude)
    if env_mav is None:
        return {"method": "RAI", "error": f"no mean-rectified row for band {band}"}
    if not meta.get("unit_known", True):
        return {"method": "RAI", "error": "chin EMG has no physical units"}
    if len(ep) < 2:
        return {"method": "RAI", "error": "less than one minute of REM"}
    amp, _ = mini_epoch_rms(env_mav.reshape(-1), meta["env_fs"], 1.0, how="mean")
    if noise == "moving_min":
        from scipy.ndimage import minimum_filter1d
        amp = amp - minimum_filter1d(amp, size=int(window_s), mode="nearest")
    else:
        amp = amp - amp.min()
    edges = np.arange(0, (n_bins + 1) * bin_width, bin_width)
    hist, _ = np.histogram(amp, bins=np.r_[edges, np.inf])
    frac = hist / hist.sum()
    a0, a1 = float(frac[0]), float(frac[1])
    rai = a0 / (1.0 - a1) if (1.0 - a1) > 1e-9 else float("nan")
    return {
        "method": "RAI",
        "rai": round(rai, 4),
        "frac_silent": round(a0, 4),
        "rem_minutes": round(len(ep) * meta["epoch_sec"] / 60.0, 1),
        "positive": bool(rai < 0.8) if np.isfinite(rai) else None,
    }


def score_all(pack, band="native", exclude=()):
    """
    Run every method on one night (a pack; see pack.py).
    band = "native" uses each recording's own EMG band; "harmonized" uses the
    common 10-45 Hz band, the fair choice when comparing ACROSS recordings made
    on different equipment; "native_with_ecg" leaves the heartbeat in.
    exclude = epoch-flag names (e.g. "resp", "arousal") whose REM epochs are
    dropped before scoring.
    """
    rows = [score_night(pack, m, band, exclude) for m in METHODS.values()]
    rows.append(rem_atonia_index(pack, band, exclude))
    for r in rows:
        r["band"] = band
        r["excluded"] = "+".join(exclude) if exclude else ""
    return rows

In [ ]:
%%writefile /content/rbd-rwa/run_extract.py
"""Stream CAP recordings: download -> extract -> discard raw.

Designed for hosted notebooks, where the session can vanish at any moment.
Progress is written to a manifest after every recording, so re-running
continues from where it stopped and never re-downloads finished work. The next
few recordings download in the background while the current one is being
extracted, and raw EDFs are deleted as soon as their derived file exists, so
disk use stays flat no matter how many recordings are processed.

    python run_extract.py                  # every CAP record (default)
    python run_extract.py rbd1 rbd2 n1     # specific records
    python run_extract.py --cohorts rbd n  # whole cohorts by prefix
    python run_extract.py --keep-raw       # keep EDFs (local debugging only)
"""
import argparse
import json
import sys
import time
import traceback
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.fetch import fetch_cap, cap_records
from src.extract import extract_recording, summarise

MANIFEST = config.MANIFESTS / "cap_extract.json"
PREFETCH = 3          # recordings downloading ahead of the one being extracted


def load_manifest():
    if MANIFEST.exists():
        try:
            return json.loads(MANIFEST.read_text())
        except json.JSONDecodeError:
            pass
    return {"done": {}, "failed": {}}


def save_manifest(m):
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(json.dumps(m, indent=2))
    tmp.replace(MANIFEST)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("records", nargs="*")
    ap.add_argument("--cohorts", nargs="*", help="cohort prefixes, e.g. rbd n")
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    args = ap.parse_args()

    config.describe()
    groups = cap_records()
    if args.records:
        records = args.records
    elif args.cohorts:
        records = [r for c in args.cohorts for r in groups.get(c, [])]
    else:
        records = [r for g in groups.values() for r in g]
    print(f"\n{len(records)} records queued "
          f"(cohorts available: { {k: len(v) for k, v in groups.items()} })\n")

    m = load_manifest()
    todo = [r for r in records
            if r not in m["done"] or args.overwrite
            or not (config.DERIVED / f"{r}.npz").exists()]
    print(f"{len(m['done'])} already done, {len(todo)} to process\n")

    pool = ThreadPoolExecutor(max_workers=PREFETCH)
    pending = {}

    def prefetch(i):
        if i < len(todo) and i not in pending:
            pending[i] = pool.submit(fetch_cap, todo[i], config.SCRATCH)

    for i in range(PREFETCH):
        prefetch(i)

    t_start = time.time()
    for i, rec in enumerate(todo):
        prefetch(i + PREFETCH)
        t0 = time.time()
        print(f"[{i + 1}/{len(todo)}] {rec}", flush=True)
        edf = txt = None
        try:
            edf, txt = pending.pop(i).result()
            t_dl = time.time() - t0
            out = extract_recording(edf, txt, rec, overwrite=args.overwrite)
            print(f"    {summarise(out)}  (waited {t_dl:.0f}s for download, "
                  f"{time.time() - t0 - t_dl:.0f}s to extract)", flush=True)
            m["done"][rec] = {"file": out.name, "seconds": round(time.time() - t0, 1)}
            m["failed"].pop(rec, None)
        except Exception as e:
            print(f"    FAILED: {type(e).__name__}: {e}", flush=True)
            traceback.print_exc(limit=2)
            m["failed"][rec] = f"{type(e).__name__}: {e}"
        finally:
            if not args.keep_raw:
                # Raw is disposable once the derived file exists (or failed).
                for p in (edf, txt):
                    if p is not None:
                        Path(p).unlink(missing_ok=True)
        save_manifest(m)
    pool.shutdown(wait=True)

    derived = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    size = sum(p.stat().st_size for p in derived)
    print(f"\n{'=' * 70}")
    print(f"  {len(m['done'])} recordings extracted, {len(m['failed'])} failed "
          f"in {(time.time() - t_start) / 60:.1f} min")
    print(f"  derived data: {len(derived)} files, {size / 1e6:.0f} MB total")
    print(f"  location: {config.DERIVED}")
    if m["failed"]:
        print("\n  failures:")
        for k, v in m["failed"].items():
            print(f"    {k}: {v}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_cohort.py
"""Build packs for a whole open cohort: download -> extract -> discard raw.

    python run_cohort.py --cohort chal18             # 994 MGH PSGs
    python run_cohort.py --cohort hmc                # 148 Haaglanden PSGs
    python run_cohort.py --cohort chal18 --limit 5   # quick test
    python run_cohort.py --cohort cap                # repack CAP's derived files
    python run_cohort.py --cohort hmc --ladder       # noise ladder (see cohorts.py)

Several records are processed at once (--workers), each in its own process,
so downloads overlap with computation. Progress is written to a manifest
after every record, so re-running continues where it stopped. Raw files are
deleted as soon as their pack exists.
"""
import argparse
import json
import shutil
import sys
import time
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

from src import config
from src.pack import save_pack


def _work(name, rec, demo, scratch, out_path, keep_raw, ladder=False):
    """Runs in a worker process: one record from download to pack."""
    from src.cohorts import COHORTS, extract_pack, extract_ladder_pack
    cohort = COHORTS[name]
    extract = extract_ladder_pack if ladder else extract_pack
    t0 = time.time()
    folder = Path(scratch) / rec
    folder.mkdir(parents=True, exist_ok=True)
    try:
        files = cohort.fetch(rec, folder)
        t_dl = time.time() - t0
        d = cohort.read(rec, files)
        pack = extract(cohort, rec, d, demo)
        save_pack(Path(out_path), pack)
        m = pack["meta"]
        return rec, {"seconds": round(time.time() - t0, 1), "download_s": round(t_dl, 1),
                     "rem_min": round(m["n_rem_epochs"] * 0.5, 1),
                     "ecg": m["ecg"].get("artifact_ratio_before", m["ecg"]["beats"])}, None
    except Exception as e:
        return rec, None, f"{type(e).__name__}: {e} | {traceback.format_exc(limit=2)[-300:]}"
    finally:
        if not keep_raw:
            shutil.rmtree(folder, ignore_errors=True)


class Mirror:
    """
    Best-effort copy of finished packs to a second place (Google Drive). The
    Drive mount in Colab can drop out under load ("Transport endpoint is not
    connected"), so work happens on local disk and each small pack is copied
    over as it is finished; if the mirror fails, the run carries on.
    """
    def __init__(self, path):
        self.path = Path(path) if path else None
        self.ok = self.path is not None
        if self.ok:
            try:
                self.path.mkdir(parents=True, exist_ok=True)
            except OSError as e:
                self._fail(e)

    def _fail(self, e):
        if self.ok:
            print(f"  (mirror unavailable, continuing without it: {e})", flush=True)
        self.ok = False

    def restore(self, out_dir):
        """Bring back packs saved by an earlier session, so they are skipped."""
        if not self.ok:
            return 0
        n = 0
        try:
            for p in self.path.glob("*.npz"):
                if not (out_dir / p.name).exists():
                    shutil.copy2(p, out_dir / p.name)
                    n += 1
        except OSError as e:
            self._fail(e)
        return n

    def push(self, p):
        if self.ok:
            try:
                shutil.copy2(p, self.path / Path(p).name)
            except OSError as e:
                self._fail(e)


def repack_cap(out_dir, mirror):
    """CAP was extracted to full-night files; convert them to packs."""
    from src.extract import load_derived
    from src.pack import pack_from_full
    files = sorted(p for p in config.DERIVED.glob("*.npz") if ".tmp" not in p.name)
    for p in files:
        out = out_dir / p.name
        if out.exists():
            continue
        try:
            pack = pack_from_full(load_derived(p))
        except Exception as e:                       # unreadable file: report, go on
            print(f"  {p.name}: cannot read ({type(e).__name__}: {e})", flush=True)
            continue
        pack["meta"]["cohort"] = "cap"
        save_pack(out, pack)
        mirror.push(out)
    print(f"cap: {len(list(out_dir.glob('*.npz')))} packs in {out_dir}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--cohort", required=True)
    ap.add_argument("--limit", type=int, default=0,
                    help="stop once this many records are done (0 = all)")
    ap.add_argument("--workers", type=int, default=3)
    ap.add_argument("--keep-raw", action="store_true")
    ap.add_argument("--mirror", default=None,
                    help="also copy each finished pack here (e.g. a Drive folder)")
    ap.add_argument("--ladder", action="store_true",
                    help="build noise-ladder packs (DURABLE/ladder) instead of packs")
    args = ap.parse_args()

    kind = "ladder" if args.ladder else "packs"
    out_dir = config.DURABLE / kind / args.cohort
    out_dir.mkdir(parents=True, exist_ok=True)
    mirror = Mirror(Path(args.mirror) / args.cohort if args.mirror else None)
    restored = mirror.restore(out_dir)
    if restored:
        print(f"restored {restored} packs from {mirror.path}", flush=True)
    if args.cohort == "cap" and not args.ladder:
        repack_cap(out_dir, mirror)
        return

    from src.cohorts import COHORTS
    cohort = COHORTS[args.cohort]
    manifest = config.MANIFESTS / f"{args.cohort}_{kind}.json"
    m = json.loads(manifest.read_text()) if manifest.exists() else {"done": {}, "failed": {}}
    records = cohort.records()
    demo = cohort.demographics()
    todo = [r for r in records if not (out_dir / f"{r}.npz").exists()]
    done = len(records) - len(todo)
    if args.limit:
        todo = todo[: max(0, args.limit - done)]
    config.describe()
    print(f"\n{args.cohort}: {len(records)} records, {done} done, "
          f"{len(todo)} to process with {args.workers} workers\n", flush=True)

    t_start = time.time()
    with ProcessPoolExecutor(max_workers=args.workers) as ex:
        futs = [ex.submit(_work, args.cohort, r, demo.get(r), str(config.SCRATCH),
                          str(out_dir / f"{r}.npz"), args.keep_raw, args.ladder)
                for r in todo]
        for i, f in enumerate(as_completed(futs), 1):
            rec, info, err = f.result()
            if err:
                m["failed"][rec] = err
                print(f"[{i}/{len(todo)}] {rec} FAILED {err[:160]}", flush=True)
            else:
                m["done"][rec] = info
                m["failed"].pop(rec, None)
                mirror.push(out_dir / f"{rec}.npz")
                print(f"[{i}/{len(todo)}] {rec}  REM {info['rem_min']} min  "
                      f"ECG x{info['ecg']}  ({info['download_s']}s download, "
                      f"{info['seconds']}s total)", flush=True)
            tmp = manifest.with_suffix(".tmp")
            tmp.write_text(json.dumps(m, indent=1))
            tmp.replace(manifest)

    packs = list(out_dir.glob("*.npz"))
    size = sum(p.stat().st_size for p in packs) / 1e9
    print(f"\n{args.cohort}: {len(packs)} packs ({size:.2f} GB), {len(m['failed'])} failed, "
          f"{(time.time() - t_start) / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_score.py
"""Score every pack of every cohort with every RWA method.

Reads DURABLE/packs/<cohort>/*.npz (build them with run_cohort.py), writes one
tidy row per (recording, method, band, exclusion) to results/rwa_scores.csv,
and prints three summaries:

  1. RBD separation within CAP (AUC, 0.5 = chance), against CAP's healthy
     sleepers - most of whom were recorded on different equipment, so partly
     a hardware contrast - and against CAP patients recorded on the same
     system as the RBD group (512 Hz, microvolts), the fair comparison.
  2. How many people WITHOUT known RBD each method's published cut-off flags,
     per cohort: the false-positive burden a screening programme would face.
  3. The spread of each metric per cohort (median, 90th, 95th percentile).

    python run_score.py
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import score_all

CAP_GROUPS = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
              "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB", "brux": "bruxism"}
METRICS = ["tonic_pct", "phasic_pct", "phasic_epoch_pct", "any_pct", "atonia_loss"]
MIN_REM_MIN = 10.0     # nights with less REM than this are not scored in summaries


def group_of(cohort, record):
    if cohort == "cap":
        return CAP_GROUPS.get("".join(c for c in record if not c.isdigit()), "other")
    return "clinical"


def auc(pos, neg):
    """Probability a random positive outscores a random negative (ties = 1/2)."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan"), len(pos), len(neg)
    diff = pos[:, None] - neg[None, :]
    return float(((diff > 0) + 0.5 * (diff == 0)).mean()), len(pos), len(neg)


def rem_boundaries(pack):
    """Flag the first and last epoch of every REM period: transitional epochs,
    where scoring conventions differ most between laboratories."""
    flag = np.zeros(len(pack["hypnogram"]), np.int8)
    ep = pack["rem_epochs"]
    if len(ep):
        brk = np.flatnonzero(np.diff(ep) != 1)
        flag[ep[np.r_[0, brk + 1]]] = 1
        flag[ep[np.r_[brk, len(ep) - 1]]] = 1
    return flag


def score_everything():
    rows = []
    for cohort_dir in sorted((config.DURABLE / "packs").glob("*")):
        files = sorted(p for p in cohort_dir.glob("*.npz") if ".tmp" not in p.name)
        print(f"{cohort_dir.name}: {len(files)} packs", flush=True)
        for p in files:
            pack = load_pack(p)
            m = pack["meta"]
            cohort = m.get("cohort", cohort_dir.name)
            info = dict(cohort=cohort, record=m["record"], group=group_of(cohort, m["record"]),
                        age=m.get("age"), sex=m.get("sex"), ahi=m.get("ahi"),
                        chin_fs=m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                        unit_known=m.get("unit_known", True))
            exclusions = [()] + ([("resp", "arousal")] if pack["epoch_flags"] else [])
            pack["epoch_flags"] = dict(pack["epoch_flags"], boundary=rem_boundaries(pack))
            for band in ("native", "harmonized"):
                for ex in exclusions + ([("boundary",)] if band == "native" else []):
                    for r in score_all(pack, band, ex):
                        rows.append({**info, **r})
    df = pd.DataFrame(rows)
    if "rai" in df:
        df["atonia_loss"] = 1 - df["rai"]              # higher = more RWA, like the rest
    return df


def main():
    import argparse
    import shutil
    ap = argparse.ArgumentParser()
    ap.add_argument("--mirror", default=None, help="also copy the results table here")
    args = ap.parse_args()

    df = score_everything()
    for col in ["positive", "error", *METRICS]:          # absent if nothing produced them
        if col not in df:
            df[col] = np.nan
    out = config.RESULTS / "rwa_scores.csv"
    df.to_csv(out, index=False)
    print(f"\nsaved {len(df)} rows -> {out}")
    if args.mirror:
        try:
            Path(args.mirror).mkdir(parents=True, exist_ok=True)
            shutil.copy2(out, Path(args.mirror) / out.name)
            print(f"copied -> {args.mirror}")
        except OSError as e:
            print(f"(could not copy to {args.mirror}: {e})")
    print()
    base = df[(df.excluded == "") & (df.rem_minutes >= MIN_REM_MIN)]

    # 1. RBD separation within CAP ────────────────────────────────────────────
    cap = base[base.cohort == "cap"]
    if len(cap):
        same = (cap.chin_fs == 512) & cap.unit_known.astype(bool)
        print("1. RBD separation in CAP (AUC)           vs healthy      vs same-system patients")
        for (band, meth), g in cap.groupby(["band", "method"], sort=False):
            for metric in METRICS:
                if metric not in g or g[metric].notna().sum() == 0:
                    continue
                s = same.loc[g.index]
                a1, n1, h1 = auc(g.loc[g.group == "RBD", metric], g.loc[g.group == "healthy", metric])
                a2, n2, h2 = auc(g.loc[s & (g.group == "RBD"), metric],
                                 g.loc[s & ~g.group.isin(["RBD", "healthy"]), metric])
                print(f"   {band:<10} {meth:<9} {metric:<17} {a1:6.3f} ({n1}v{h1:<3})   "
                      f"{a2:6.3f} ({n2}v{h2})")
        print()

    # 2. Share flagged positive by published cut-offs ─────────────────────────
    lab = base.assign(who=np.where(base.cohort == "cap", "CAP " + base.group, base.cohort))
    pos = lab.dropna(subset=["positive"])
    pos = pos.assign(positive=pos.positive.astype(float))
    tab = pos.pivot_table(index=["band", "method"], columns="who", values="positive",
                          aggfunc="mean")
    n = pos.groupby("who").record.nunique()
    print("2. Share classified POSITIVE by each published cut-off (%), nights with >= "
          f"{MIN_REM_MIN:.0f} min REM")
    print("   n per group: " + ", ".join(f"{k} {v}" for k, v in n.items()))
    print((100 * tab).round(1).to_string())
    print()

    # 3. Distribution of each metric per cohort ────────────────────────────────
    print("3. Metric distribution per group (native band): median / p90 / p95")
    nat = lab[lab.band == "native"]
    for meth, g in nat.groupby("method", sort=False):
        for metric in METRICS:
            if metric not in g or g[metric].notna().sum() == 0:
                continue
            q = g.groupby("who")[metric].quantile([0.5, 0.9, 0.95]).unstack()
            cells = "  ".join(f"{w}: {r[0.5]:.1f}/{r[0.9]:.1f}/{r[0.95]:.1f}"
                              for w, r in q.iterrows())
            print(f"   {meth:<9} {metric:<17} {cells}")

    # 4. Respiratory events and arousals (cohorts that annotate them) ─────────
    ev = df[df.cohort == "chal18"]
    if len(ev) and (ev.excluded != "").any():
        print("\n4. Effect of removing REM epochs with respiratory events or arousals "
              "(chal18, native band, share positive %)")
        e = ev[(ev.band == "native") & (ev.rem_minutes >= MIN_REM_MIN)].dropna(subset=["positive"])
        print((100 * e.assign(positive=e.positive.astype(float))
               .pivot_table(index="method", columns="excluded", values="positive",
                            aggfunc="mean")).round(1).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_sensitivity.py
"""How much does the operational definition of "background" move the verdicts?

The visual methods judge activity against the background EMG a scorer sees by
eye. An automated version has to choose a number; this script re-scores every
night (native band, all REM) under several choices and reports, for Montreal,
SINBAR and AASM, the share of nights flagged positive per group and the RBD
separation (AUC vs same-system CAP patients). The RAI does not use a
background and is not affected.

    python run_sensitivity.py
"""
import dataclasses
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import METHODS, score_night

# name -> background used by the relative criteria. AASM's tonic criterion is
# defined against the minimum NREM amplitude and keeps that definition.
VARIANTS = ["p5_rem", "p10_rem", "p25_rem", "p50_rem", "min_rem", "p5_nrem"]
MIN_REM = 10.0
CAP_GROUPS = {"rbd": "RBD", "n": "healthy", "nfle": "NFLE", "plm": "PLM",
              "ins": "insomnia", "narco": "narcolepsy", "sdb": "SDB", "brux": "bruxism"}


def auc(pos, neg):
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    d = pos[:, None] - neg[None, :]
    return float(((d > 0) + 0.5 * (d == 0)).mean())


def main():
    rows = []
    for cdir in sorted((config.DURABLE / "packs").glob("*")):
        for p in sorted(cdir.glob("*.npz")):
            pack = load_pack(p)
            m = pack["meta"]
            cohort = m.get("cohort", cdir.name)
            group = CAP_GROUPS.get("".join(c for c in m["record"] if not c.isdigit()), "other") \
                if cohort == "cap" else "clinical"
            for v in VARIANTS:
                for key in ("montreal", "sinbar", "aasm"):
                    meth = METHODS[key]
                    meth = dataclasses.replace(meth, phasic_baseline=v, tonic_baseline=(
                        meth.tonic_baseline if key == "aasm" else v))
                    r = score_night(pack, meth)
                    rows.append({"cohort": cohort, "record": m["record"], "group": group,
                                 "chin_fs": m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                                 "unit_known": m.get("unit_known", True), "variant": v, **r})
        print(f"{cdir.name}: done", flush=True)
    df = pd.DataFrame(rows)
    df = df[df.rem_minutes >= MIN_REM]
    df.to_csv(config.RESULTS / "sensitivity_background.csv", index=False)
    who = np.where(df.cohort == "cap", "CAP " + df.group, df.cohort)
    df = df.assign(who=who)

    print("\nShare flagged POSITIVE (%) by background definition")
    pos = df.dropna(subset=["positive"]).assign(
        positive=lambda d: d.positive.map(lambda x: float(x in (True, "True", 1, 1.0))))
    t = pos.pivot_table(index=["method", "variant"], columns="who", values="positive",
                        aggfunc="mean") * 100
    cols = [c for c in ["CAP RBD", "CAP healthy", "CAP NFLE", "CAP PLM", "chal18", "hmc",
                        "hmc_q1uv", "hmc_mgh", "hmc_low", "hmc_low_mgh", "chal18_noise"]
            if c in t.columns]
    print(t[cols].round(0).to_string())

    print("\nRBD vs same-system CAP patients (AUC) by background definition")
    same = (df.cohort == "cap") & (df.chin_fs == 512) & df.unit_known.astype(bool)
    out = []
    for (meth, v), g in df[same].groupby(["method", "variant"]):
        for metric in ("tonic_pct", "phasic_pct", "any_pct", "phasic_epoch_pct"):
            if metric in g and g[metric].notna().any():
                out.append(dict(method=meth, variant=v, metric=metric,
                                auc=auc(g.loc[g.group == "RBD", metric],
                                        g.loc[~g.group.isin(["RBD", "healthy"]), metric])))
    print(pd.DataFrame(out).pivot_table(index=["method", "metric"], columns="variant",
                                        values="auc").round(3).to_string())


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_report.py
"""Turn results/rwa_scores.csv into the paper's tables.

    python run_report.py

Prints (and writes to results/report.md):

  T1  cohort description: nights scored, REM minutes, age and sex where known
  T2  share of nights each method's published cut-off flags, per group, with
      95% Wilson intervals - sensitivity in CAP's RBD patients, false-positive
      rate everywhere else
  T3  agreement between methods on the same nights (Cohen's kappa)
  T4  what predicts a false positive in the MGH cohort: age, sex, apnea-
      hypopnea index (logistic regression, one model per method)
  T5  removing REM epochs that contain respiratory events or arousals
  T6  positive predictive value of each method if used to screen a
      population in which a given share has RBD (specificity per hospital)
  T7  cut-offs re-derived for automated scoring (Youden, CAP RBD vs CAP
      patients recorded on the same system) and the external nights they flag
  T9  quiet-REM chin EMG level ("background") per group
  T10 T7 validated: leave-one-out cross-validation, and bootstrap intervals
      for the cut-offs and the external false-positive shares
  T11 positive calls by quiet-REM level within cohorts, and in MGH the odds
      ratio per halving of that level, adjusted for age, sex and AHI
  T12 simulated recordings (hardware, amplitude, noise floor) against the
      cohort they were made from, on the same nights
  T13 MGH positive calls by apnea-hypopnea index, all REM and event-free REM
  T15 the same without transitional epochs (first and last of each REM period)
  T16 the floor signature: quietest N2/N3 over quietest REM chin level
  T17 age in CAP: RBD against same-system patients under 50 and aged 50 or over
  T8  heartbeat artefact in the chin EMG before and after its removal
  T14 MGH positive calls by the residual heartbeat artefact

Nights with less than MIN_REM minutes of REM are left out, as clinical
scoring would.
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config

MIN_REM = 10.0
DECIDING = ["Montreal", "SINBAR", "RAI"]          # methods with a published cut-off
METRICS = ["tonic_pct", "phasic_pct", "phasic_epoch_pct", "any_pct", "atonia_loss"]
# Software copies of a cohort (src/cohorts.py) and the cohort each one copies.
SIMULATED = {"hmc_q1uv": "hmc", "hmc_mgh": "hmc", "hmc_low": "hmc", "hmc_low_mgh": "hmc_low",
             "chal18_noise": "chal18"}
# Recordings withdrawn by the data providers (HMC v1.1: erroneous signal data).
# RECORDS no longer lists them; this guards against older downloads.
WITHDRAWN = {("hmc", "SN014"), ("hmc", "SN064"), ("hmc", "SN135")}


def wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def paired_change(before, after):
    """Change in the share positive on the same nights: difference with a 95%
    Wald interval for paired proportions, and the exact McNemar test."""
    from scipy.stats import binomtest
    before, after = np.asarray(before, float), np.asarray(after, float)
    n = len(before)
    up = int(((before == 0) & (after == 1)).sum())
    down = int(((before == 1) & (after == 0)).sum())
    d = (up - down) / n
    se = np.sqrt(max(up + down - (up - down) ** 2 / n, 0.0)) / n
    p = binomtest(up, up + down).pvalue if up + down else 1.0
    return (f"diff {100 * d:+.1f} [{100 * (d - 1.96 * se):+.1f} to {100 * (d + 1.96 * se):+.1f}]  "
            f"{down} nights -, {up} +, McNemar p={p:.1g}")


def kappa(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    po = np.mean(a == b)
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


def groups(df):
    """Label each night: CAP subgroups, and the external clinical cohorts."""
    who = np.where(df.cohort == "cap", "CAP " + df.group.astype(str), df.cohort)
    return df.assign(who=who)


def main():
    df = pd.read_csv(config.RESULTS / "rwa_scores.csv")
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df = groups(df)
    # CAP publishes age and sex in a spreadsheet, not in the recordings
    from src.fetch import cap_demographics
    cap_demo = cap_demographics()
    if cap_demo:
        cap = df.cohort == "cap"
        df.loc[cap, "age"] = df.loc[cap, "record"].map(lambda r: cap_demo.get(r, {}).get("age"))
        df.loc[cap, "sex"] = df.loc[cap, "record"].map(lambda r: cap_demo.get(r, {}).get("sex"))
    base = df[(df.excluded.fillna("") == "") & (df.rem_minutes >= MIN_REM)]
    lines = []
    say = lambda s="": (print(s), lines.append(s))

    # T1 ─────────────────────────────────────────────────────────────────────
    nights = base.drop_duplicates(["cohort", "record"])
    say("T1  Nights scored (>= %.0f min REM)" % MIN_REM)
    t1 = nights.groupby("who").agg(nights=("record", "count"),
                                   rem_min_median=("rem_minutes", "median"),
                                   age_mean=("age", "mean"),
                                   male_pct=("sex", lambda s: 100 * (s == "M").mean()
                                             if s.notna().any() else np.nan),
                                   chin_fs=("chin_fs", "median"))
    say(t1.round(1).to_string())
    say()

    # T2 ─────────────────────────────────────────────────────────────────────
    say("T2  Share flagged POSITIVE by the published cut-off, % [95% CI]")
    rows = []
    for (band, meth, who), g in base[base.method.isin(DECIDING)].groupby(["band", "method", "who"]):
        v = g.positive.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        p, lo, hi = wilson(v.sum(), len(v))
        rows.append(dict(band=band, method=meth, who=who, n=len(v),
                         cell=f"{100 * p:.0f} [{100 * lo:.0f}-{100 * hi:.0f}]"))
    t2 = pd.DataFrame(rows).pivot_table(index=["band", "method"], columns="who",
                                        values="cell", aggfunc="first")
    say(t2.to_string())
    say()

    # T3 ─────────────────────────────────────────────────────────────────────
    say("T3  Agreement between methods on the same nights (Cohen's kappa, native band)")
    nat = base[(base.band == "native") & base.method.isin(DECIDING)]
    wide = nat.pivot_table(index=["who", "record"], columns="method", values="positive",
                           aggfunc="first").dropna()
    wide = wide.map(lambda x: float(x in (True, "True", 1, 1.0)))
    for who, g in wide.groupby(level="who"):
        pairs = [f"{a}-{b} {kappa(g[a], g[b]):.2f}" for i, a in enumerate(DECIDING)
                 for b in DECIDING[i + 1:] if a in g and b in g]
        disagree = (g.nunique(axis=1) > 1).mean() * 100
        say(f"  {who:<16} n={len(g):<5} {'  '.join(pairs)}   methods disagree on {disagree:.0f}% of nights")
    say()

    # T4 ─────────────────────────────────────────────────────────────────────
    mgh = base[(base.cohort == "chal18") & (base.band == "native") & base.method.isin(DECIDING)]
    try:
        import statsmodels.formula.api as smf
    except ImportError:
        smf = None
        say("T4  skipped: statsmodels is not installed\n")
    if smf is not None and len(mgh) and mgh.age.notna().any():
        say("T4  What predicts a positive call in the MGH cohort (odds ratios [95% CI])")
        for meth, g in mgh.groupby("method"):
            g = g.dropna(subset=["positive", "age", "sex", "ahi"]).copy()
            g["y"] = g.positive.map(lambda x: float(x in (True, "True", 1, 1.0)))
            g["age10"] = g.age / 10.0
            g["male"] = (g.sex == "M").astype(float)
            g["ahi10"] = g.ahi / 10.0
            if g.y.nunique() < 2:
                say(f"  {meth}: all nights {'positive' if g.y.iloc[0] else 'negative'}")
                continue
            m = smf.logit("y ~ age10 + male + ahi10", data=g).fit(disp=False)
            ci = np.exp(m.conf_int())
            cells = [f"{k} {np.exp(m.params[k]):.2f} [{ci.loc[k, 0]:.2f}-{ci.loc[k, 1]:.2f}]"
                     for k in ("age10", "male", "ahi10")]
            say(f"  {meth:<9} n={len(g):<5} per 10 y: {cells[0].split(' ', 1)[1]}   "
                f"male: {cells[1].split(' ', 1)[1]}   per 10 AHI: {cells[2].split(' ', 1)[1]}")
        say()

    # T5 ─────────────────────────────────────────────────────────────────────
    ev = df[(df.cohort == "chal18") & (df.band == "native") & df.method.isin(DECIDING)
            & (df.rem_minutes >= MIN_REM) & df.excluded.fillna("").isin(["", "resp+arousal"])]
    if len(ev) and (ev.excluded.fillna("") != "").any():
        say("T5  Excluding REM epochs with respiratory events or arousals (MGH), % positive")
        w = ev.pivot_table(index=["record", "method"], columns=ev.excluded.fillna(""),
                           values="positive", aggfunc="first").dropna()
        w = w.map(lambda x: float(x in (True, "True", 1, 1.0)))
        for meth, g in w.groupby(level="method"):
            a, b = g.iloc[:, 0], g.iloc[:, 1]
            say(f"  {meth:<9} all REM {100 * a.mean():.0f}%  ->  event-free REM {100 * b.mean():.0f}%  "
                f"(changed on {100 * (a != b).mean():.0f}% of nights)")
        say()

        # T13 ────────────────────────────────────────────────────────────────
        say("T13 MGH: % positive by apnea-hypopnea index, all REM and event-free REM")
        ev = ev.assign(y=ev.positive.map(lambda x: float(x in (True, "True", 1, 1.0))
                                         if pd.notna(x) else np.nan),
                       ahi_cat=pd.cut(ev.ahi, [-1, 5, 15, 30, np.inf],
                                      labels=["AHI<5", "5-15", "15-30", ">=30"]),
                       rem=np.where(ev.excluded.fillna("") == "", "all REM", "event-free"))
        t = ev.pivot_table(index=["method", "rem"], columns="ahi_cat", values="y",
                           aggfunc="mean", observed=False) * 100
        say(t.round(0).to_string())
        n = ev[(ev.method == "Montreal") & (ev.rem == "all REM")].ahi_cat.value_counts()
        say("  nights: " + "  ".join(f"{k} {n.get(k, 0)}" for k in t.columns))
        say()

    # T6 ─────────────────────────────────────────────────────────────────────
    say("T6  Screening: positive predictive value at a given RBD prevalence")
    say("    sensitivity from CAP RBD; specificity from each external cohort and pooled")
    for meth in DECIDING:
        g = base[(base.band == "native") & (base.method == meth)]
        y = lambda s: s.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        sens = y(g[g.who == "CAP RBD"].positive)
        for name, cohorts in (("pooled", ["chal18", "hmc"]), ("MGH", ["chal18"]), ("HMC", ["hmc"])):
            spec = 1 - y(g[g.cohort.isin(cohorts)].positive)
            if len(sens) == 0 or len(spec) == 0:
                continue
            se, sp = sens.mean(), spec.mean()
            ppv = [se * p / (se * p + (1 - sp) * (1 - p)) for p in (0.01, 0.02, 0.05, 0.10)]
            say(f"  {meth:<9} {name:<6} sens {100 * se:.0f}%  spec {100 * sp:.0f}%   PPV at 1/2/5/10% "
                "prevalence: " + " / ".join(f"{100 * v:.0f}%" for v in ppv))

    say()

    # T7 ─────────────────────────────────────────────────────────────────────
    say("T7  Cut-offs re-derived for automated scoring (Youden, CAP RBD vs same-system")
    say("    patients) and the share of external nights they flag, native band")
    for (meth, metric), g in base[base.band == "native"].melt(
            id_vars=["cohort", "record", "who", "method", "chin_fs", "unit_known"],
            value_vars=[m for m in METRICS if m in base], var_name="metric") \
            .dropna(subset=["value"]).groupby(["method", "metric"]):
        s = (g.cohort == "cap") & (g.chin_fs == 512) & g.unit_known.astype(bool)
        pos = g.loc[s & (g.who == "CAP RBD"), "value"].to_numpy()
        neg = g.loc[s & ~g.who.isin(["CAP RBD", "CAP healthy"]), "value"].to_numpy()
        if len(pos) < 5 or len(neg) < 5:
            continue
        cands = np.unique(np.r_[pos, neg])
        j = [(pos >= c).mean() - (neg >= c).mean() for c in cands]
        c = cands[int(np.argmax(j))]
        ext = g.loc[g.cohort.isin(["chal18", "hmc"])]
        fp = {k: 100 * (v.value >= c).mean() for k, v in ext.groupby("cohort")}
        say(f"  {meth:<9} {metric:<17} cut-off {c:7.2f}: sens {100 * (pos >= c).mean():3.0f}%  "
            f"same-system FP {100 * (neg >= c).mean():3.0f}%  "
            + "  ".join(f"{k} FP {v:3.0f}%" for k, v in sorted(fp.items())))
    say()

    # T9 ─────────────────────────────────────────────────────────────────────
    say("T9  Background chin EMG level (5th percentile of 1 s REM RMS, uV), median [IQR]")
    bg = base[(base.band == "native") & (base.method == "Montreal")]
    if "background_phasic_uV" in bg:
        for who, g in bg.groupby("who"):
            v = g.background_phasic_uV.dropna()
            if len(v):
                say(f"  {who:<16} n={len(v):<5} {v.median():6.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}]")
    say()

    # T10 ────────────────────────────────────────────────────────────────────
    say("T10 Re-derived cut-offs, validated: leave-one-out CV on CAP, and 2,000 bootstrap")
    say("    re-derivations for the uncertainty of the cut-off and of external false positives")
    rng = np.random.default_rng(0)

    def youden(p, n):
        c = np.unique(np.r_[p, n])
        j = (p[None, :] >= c[:, None]).mean(axis=1) - (n[None, :] >= c[:, None]).mean(axis=1)
        return c[int(np.argmax(j))]

    melted = base[base.band == "native"].melt(
        id_vars=["cohort", "record", "who", "method", "chin_fs", "unit_known"],
        value_vars=[m for m in METRICS if m in base], var_name="metric").dropna(subset=["value"])
    for (meth, metric), g in melted.groupby(["method", "metric"]):
        s = (g.cohort == "cap") & (g.chin_fs == 512) & g.unit_known.astype(bool)
        pos = g.loc[s & (g.who == "CAP RBD"), "value"].to_numpy(float)
        neg = g.loc[s & ~g.who.isin(["CAP RBD", "CAP healthy"]), "value"].to_numpy(float)
        if len(pos) < 5 or len(neg) < 5:
            continue
        cv_tp = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
        cv_fp = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
        ext = {k: v.value.to_numpy(float) for k, v in g[g.cohort.isin(["chal18", "hmc"])].groupby("cohort")}
        cuts, fps = [], {k: [] for k in ext}
        for _ in range(2000):
            c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
            cuts.append(c)
            for k, v in ext.items():
                fps[k].append((v >= c).mean())
        q = lambda a: f"{100 * np.median(a):.0f} [{100 * np.percentile(a, 2.5):.0f}-{100 * np.percentile(a, 97.5):.0f}]"
        say(f"  {meth:<9} {metric:<17} cut-off {np.median(cuts):6.2f} [{np.percentile(cuts, 2.5):.2f}-"
            f"{np.percentile(cuts, 97.5):.2f}]  CV sens {100 * cv_tp:3.0f}%  CV FP {100 * cv_fp:3.0f}%  "
            + "  ".join(f"{k} FP {q(v)}" for k, v in sorted(fps.items())))
    say()

    # T11 ────────────────────────────────────────────────────────────────────
    say("T11 Positive calls by quiet-REM level (tertiles of the background within each")
    say("    cohort, native band) and, in MGH, the odds ratio per halving of that level")
    quiet = base[(base.band == "native") & (base.method == "Montreal")][
        ["cohort", "record", "background_phasic_uV"]].rename(columns={"background_phasic_uV": "quiet"})
    nat = base[(base.band == "native") & base.method.isin(DECIDING)].merge(
        quiet, on=["cohort", "record"], how="left").dropna(subset=["quiet", "positive"])
    nat["y"] = nat.positive.map(lambda x: float(x in (True, "True", 1, 1.0)))
    for (who, meth), g in nat[nat.cohort.isin(["chal18", "hmc"])].groupby(["who", "method"]):
        t = pd.qcut(g["quiet"], 3, labels=False)
        say(f"  {who:<8} {meth:<9} " + "   ".join(
            f"{g['quiet'][t == k].median():.2f} uV: {100 * g.y[t == k].mean():3.0f}%" for k in range(3)))
    if smf is not None:
        for meth, g in nat[nat.cohort == "chal18"].groupby("method"):
            g = g.dropna(subset=["age", "sex", "ahi"]).copy()
            if g.y.nunique() < 2:
                continue
            g["halvings"] = -np.log2(g["quiet"])
            g["age10"], g["male"], g["ahi10"] = g.age / 10.0, (g.sex == "M").astype(float), g.ahi / 10.0
            m = smf.logit("y ~ halvings + age10 + male + ahi10", data=g).fit(disp=False)
            lo, hi = np.exp(m.conf_int().loc["halvings"])
            say(f"  MGH {meth:<9} OR per halving {np.exp(m.params['halvings']):.2f} [{lo:.2f}-{hi:.2f}]"
                f" adjusted for age, sex and AHI (n={len(g)})")
    say()

    # T12 ────────────────────────────────────────────────────────────────────
    say("T12 Simulated recordings against their source on the same nights (native band)")
    for sim, src in SIMULATED.items():
        pair = nat[nat.cohort == sim].merge(nat[nat.cohort == src], on=["record", "method"],
                                            suffixes=("", "_src"))
        if pair.empty:
            continue
        lvl = pair[pair.method == "Montreal"]
        say(f"  {sim:<12} vs {src:<8} n={len(lvl):<4} quiet-REM level {lvl.quiet_src.median():.2f}"
            f" -> {lvl.quiet.median():.2f} uV")
        for meth, g in pair.groupby("method"):
            say(f"      {meth:<9} {100 * g.y_src.mean():3.0f}% -> {100 * g.y.mean():3.0f}%   "
                + paired_change(g.y_src.to_numpy(), g.y.to_numpy()))
    say("    (paired difference with 95% CI; exact McNemar test on the nights that changed)")
    say()

    # T15 ────────────────────────────────────────────────────────────────────
    bd = df[(df.band == "native") & df.method.isin(DECIDING) & (df.rem_minutes >= MIN_REM)
            & df.excluded.fillna("").isin(["", "boundary"])]
    if (bd.excluded.fillna("") == "boundary").any():
        say("T15 Without the first and last epoch of every REM period (transitional epochs),")
        say("    % positive: all REM -> without them")
        bd = bd.assign(y=bd.positive.map(lambda x: float(x in (True, "True", 1, 1.0))
                                         if pd.notna(x) else np.nan))
        w = bd.pivot_table(index=["who", "record", "method"], columns=bd.excluded.fillna(""),
                           values="y", aggfunc="first").dropna()
        gap = {}
        for (who, meth), g in w.groupby(level=["who", "method"]):
            if who in SIMULATED or len(g) < 10:
                continue
            a, b = g[""].mean(), g["boundary"].mean()
            gap[who, meth] = (a, b)
            say(f"  {who:<16} {meth:<9} {100 * a:3.0f}% -> {100 * b:3.0f}%   (n={len(g)})")
        for meth in DECIDING:
            if ("chal18", meth) in gap and ("hmc", meth) in gap:
                (a1, b1), (a2, b2) = gap["chal18", meth], gap["hmc", meth]
                say(f"  MGH-HMC gap, {meth}: {100 * (a1 - a2):+.0f} -> {100 * (b1 - b2):+.0f} points")
        say()

    # T16 ────────────────────────────────────────────────────────────────────
    say("T16 Floor signature: quietest N2/N3 chin level over quietest REM level (1 = the two")
    say("    are indistinguishable, i.e. the floor is the recording's own noise), median [IQR]")
    aasm = base[(base.band == "native") & (base.method == "AASM")]
    for who, g in aasm.groupby("who"):
        r = (g.background_tonic_uV / g.background_phasic_uV).dropna()
        if len(r) >= 5:
            say(f"  {who:<16} n={len(r):<5} {r.median():.2f} [{r.quantile(.25):.2f}-{r.quantile(.75):.2f}]")
    say()

    # T17 ────────────────────────────────────────────────────────────────────
    if cap_demo:
        from run_sensitivity import auc
        yes = lambda s: s.dropna().map(lambda x: float(x in (True, "True", 1, 1.0)))
        say("T17 Age in CAP: RBD patients against same-system patients under 50 and aged 50 or")
        say("    over (native band, published cut-offs; RBD and patients on the main system)")
        cap = base[(base.cohort == "cap") & (base.band == "native")]
        same = (cap.chin_fs == 512) & cap.unit_known.astype(bool)
        rbd = cap[same & (cap.who == "CAP RBD")]
        pat = cap[same & ~cap.who.isin(["CAP RBD", "CAP healthy"])]
        parts = {"patients < 50": pat[pat.age < 50], "patients >= 50": pat[pat.age >= 50], "RBD": rbd}
        for name, g in parts.items():
            a = g.drop_duplicates("record").age
            say(f"  {name:<15} n={len(a):<3} age {a.mean():.0f} +/- {a.std():.0f} ({a.min():.0f}-{a.max():.0f}), "
                f"diagnoses: " + ", ".join(f"{k.replace('CAP ', '')} {v}" for k, v in
                                          g.drop_duplicates('record').who.value_counts().items()))
        for meth in DECIDING:
            say(f"  {meth:<9} flagged: " + "   ".join(
                f"{name} {100 * yes(g[g.method == meth].positive).mean():3.0f}%" for name, g in parts.items()))
        old = parts["patients >= 50"]
        for meth, metric in [("Montreal", "tonic_pct"), ("Montreal", "phasic_pct"),
                             ("SINBAR", "any_pct"), ("RAI", "atonia_loss")]:
            r_ = rbd[rbd.method == meth][metric]
            say(f"  AUC {meth:<9} {metric:<12} vs all patients {auc(r_, pat[pat.method == meth][metric]):.3f}"
                f"   vs patients >= 50 {auc(r_, old[old.method == meth][metric]):.3f}")
        say()

    # T8 ─────────────────────────────────────────────────────────────────────
    say("T8  Heartbeat artefact in chin EMG (heartbeat-locked ratio; 1 = none), median [p90]")
    import json
    residual = {}                                    # (cohort, record) -> ratio after removal
    for cdir in sorted((config.DURABLE / "packs").glob("*")):
        before, after = [], []
        for p in cdir.glob("*.npz"):
            try:
                meta = json.loads(str(np.load(p)["meta"]))
            except Exception:
                continue
            e = meta.get("ecg") or meta.get("chin_info", {}).get("ecg") or {}
            if e.get("beats"):
                before.append(e.get("artifact_ratio_before", np.nan))
                after.append(e.get("artifact_ratio_after", np.nan))
                residual[(cdir.name, meta.get("record", p.stem))] = after[-1]
        if before:
            b, a = np.array(before, float), np.array(after, float)
            say(f"  {cdir.name:<7} n={len(b):<5} before {np.nanmedian(b):.2f} [{np.nanpercentile(b, 90):.2f}]"
                f"   after {np.nanmedian(a):.2f} [{np.nanpercentile(a, 90):.2f}]")
    say()

    # T14 ────────────────────────────────────────────────────────────────────
    art = nat[nat.cohort == "chal18"].copy()
    art["art"] = [residual.get(("chal18", r), np.nan) for r in art.record]
    art = art.dropna(subset=["art"])
    if len(art):
        say("T14 MGH: % positive by residual heartbeat artefact (tertiles of the ratio after removal)")
        art["t"] = pd.qcut(art.art, 3, labels=False)
        for meth, g in art.groupby("method"):
            say(f"  {meth:<9} " + "   ".join(f"ratio {g.art[g.t == k].median():.2f}: "
                                            f"{100 * g.y[g.t == k].mean():3.0f}%" for k in range(3)))

    out = config.RESULTS / "report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_figures.py
"""The paper's figures, drawn from the tables that run_score.py and
run_sensitivity.py write.

    python run_figures.py        # -> figures/fig1..fig5 (.png and .pdf)

  fig1  share of nights each method flags, per group, with 95% intervals
  fig2  quiet-REM chin EMG level per cohort (A) and the hardware simulation (B)
  fig3  how the definition of "background" moves the verdicts and the separation
  fig4  where the published and the re-derived cut-offs fall on each group
  fig5  positive calls by quiet-REM level within the MGH and HMC cohorts
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from run_report import MIN_REM, WITHDRAWN, wilson
from run_sensitivity import auc

YES = (True, "True", 1, 1.0)
GROUPS = ["CAP RBD", "CAP patients", "CAP healthy", "chal18", "hmc"]
LABEL = {"CAP RBD": "CAP\nRBD", "CAP patients": "CAP\npatients", "CAP healthy": "CAP\nhealthy",
         "chal18": "MGH", "hmc": "HMC", "hmc_q1uv": "HMC\n1 µV", "hmc_mgh": "HMC\n1 µV\n200 Hz",
         "hmc_low": "HMC\n×0.39", "hmc_low_mgh": "HMC\n×0.39\n1 µV\n200 Hz",
         "chal18_noise": "MGH\n+1.5 µV\nnoise"}
COLOR = {"Montreal": "#1b9e77", "SINBAR": "#d95f02", "RAI": "#7570b3", "AASM": "#e7298a"}
GROUP_COLOR = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "CAP healthy": "#2ca02c",
               "chal18": "#1f77b4", "hmc": "#9467bd"}
VARIANTS = ["min_rem", "p5_rem", "p10_rem", "p25_rem", "p50_rem", "p5_nrem"]
VARIANT_LABEL = ["min", "p5\n(default)", "p10", "p25", "p50", "p5\nNREM"]

plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 9, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "legend.fontsize": 7, "savefig.dpi": 300})


def label_groups(df):
    """CAP RBD / CAP healthy / CAP patients on CAP's main system (512 Hz), and cohorts."""
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    who = np.select([cap & (df.group == "RBD"), cap & (df.group == "healthy"),
                     same & ~df.group.isin(["RBD", "healthy"])],
                    ["CAP RBD", "CAP healthy", "CAP patients"], default="")
    return df.assign(who=np.where(cap, who, df.cohort), same=same)


def load_scores():
    df = pd.read_csv(config.RESULTS / "rwa_scores.csv")
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df = df[(df.excluded.fillna("") == "") & (df.rem_minutes >= MIN_REM) & (df.band == "native")]
    df = label_groups(df)
    return df.assign(y=df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan))


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(config.FIGURES / f"{name}.{ext}")
    plt.close(fig)
    print(f"  {name}")


def dots_with_ci(ax, df, groups, methods=("Montreal", "SINBAR", "RAI"), show_n=True):
    width = 0.8 / len(methods)
    for j, meth in enumerate(methods):
        xs, ps, err = [], [], [[], []]
        for i, g in enumerate(groups):
            v = df[(df.who == g) & (df.method == meth)].y.dropna()
            p, lo, hi = wilson(v.sum(), len(v))
            xs.append(i + (j - (len(methods) - 1) / 2) * width)
            ps.append(100 * p)
            err[0].append(100 * (p - lo))
            err[1].append(100 * (hi - p))
        ax.errorbar(xs, ps, yerr=err, fmt="o", ms=4, capsize=2, lw=1, color=COLOR[meth], label=meth)
    n = [df[(df.who == g) & (df.method == methods[0])].y.notna().sum() for g in groups]
    ax.set_xticks(range(len(groups)),
                  [f"{LABEL[g]}\nn={k}" if show_n else LABEL[g] for g, k in zip(groups, n)])
    ax.set_ylim(0, 100)
    ax.set_ylabel("nights flagged positive (%)")


def fig1(df):
    fig, ax = plt.subplots(figsize=(6.5, 3.0), layout="constrained")
    dots_with_ci(ax, df, GROUPS)
    ax.axvline(0.5, color="0.85", lw=0.8, zorder=0)
    ax.legend(frameon=False, ncol=3, loc="upper right")
    ax.set_title("Published cut-offs: sensitivity in RBD, positives everywhere else")
    save(fig, "fig1_positivity")


def fig2(df):
    fig, (a, b) = plt.subplots(1, 2, figsize=(7.2, 3.2), layout="constrained",
                               gridspec_kw={"width_ratios": [1, 1.3]})
    lvl = df[df.method == "Montreal"]
    data = [lvl.loc[lvl.who == g, "background_phasic_uV"].dropna() for g in GROUPS]
    rng = np.random.default_rng(0)
    for i, (g, v) in enumerate(zip(GROUPS, data)):
        a.scatter(i + rng.uniform(-0.25, 0.25, len(v)), v, s=3, alpha=0.35 if len(v) > 200 else 0.7,
                  color=GROUP_COLOR[g], lw=0)
    a.boxplot(data, positions=range(len(GROUPS)), widths=0.55, showfliers=False,
              medianprops={"color": "k"}, boxprops={"lw": 0.8}, whiskerprops={"lw": 0.8})
    a.axhline(1.0, ls="--", color="k", lw=0.8, label="MGH storage step (1 µV)")
    a.legend(frameon=False, loc="upper left")
    a.set_yscale("log")
    ticks = [0.2, 0.5, 1, 2, 5, 10, 20, 50]
    a.set_yticks(ticks, [f"{t:g}" for t in ticks])
    a.set_ylim(0.15, 1.3 * max(v.max() for v in data if len(v)))
    a.set_xticks(range(len(GROUPS)), [LABEL[g] for g in GROUPS])
    a.set_ylabel("quiet-REM chin EMG level (µV)")
    a.set_title("A  Quiet-REM level")
    sims = [c for c in ["hmc", "hmc_q1uv", "hmc_mgh", "hmc_low", "hmc_low_mgh", "chal18",
                        "chal18_noise"] if (df.who == c).any()]
    dots_with_ci(b, df, sims, show_n=False)
    b.tick_params(axis="x", labelsize=7)
    b.legend(frameon=False, ncol=3, loc="upper left")
    b.set_title("B  Recording simulations")
    save(fig, "fig2_level_and_hardware")


def fig3(df):
    sens = pd.read_csv(config.RESULTS / "sensitivity_background.csv")
    sens = label_groups(sens)
    sens = sens.assign(y=sens.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan))
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.1), layout="constrained")
    x = np.arange(len(VARIANTS))
    for k, (ax, meth) in enumerate(zip(axes[:2], ["Montreal", "SINBAR"])):
        for g in GROUPS:
            s = sens[(sens.method == meth) & (sens.who == g)].groupby("variant").y.mean() * 100
            ax.plot(x, [s.get(v, np.nan) for v in VARIANTS], "o-", ms=3, lw=1,
                    color=GROUP_COLOR[g], label=LABEL[g].replace("\n", " "))
        ax.set_ylim(0, 100)
        ax.set_ylabel("nights flagged positive (%)")
        ax.set_title(f"{'AB'[k]}  {meth}")
    same = sens[sens.who.isin(["CAP RBD", "CAP patients"])]
    for meth, metric, ls in [("Montreal", "phasic_pct", "-"), ("Montreal", "tonic_pct", "--"),
                             ("SINBAR", "phasic_pct", "-"), ("SINBAR", "tonic_pct", "--"),
                             ("AASM", "phasic_pct", "-")]:
        g = same[same.method == meth]
        vals = [auc(g.loc[(g.variant == v) & (g.who == "CAP RBD"), metric],
                    g.loc[(g.variant == v) & (g.who == "CAP patients"), metric]) for v in VARIANTS]
        axes[2].plot(x, vals, "o" + ls, ms=3, lw=1, color=COLOR[meth],
                     label=f"{meth} {metric.split('_')[0]}")
    axes[2].axhline(0.5, color="0.7", lw=0.8)
    axes[2].set_ylim(0.4, 1.0)
    axes[2].set_ylabel("AUC, RBD vs other CAP patients")
    axes[2].legend(frameon=False, loc="lower left")
    axes[2].set_title("C  Separation")
    for ax in axes:
        ax.set_xticks(x, VARIANT_LABEL, fontsize=7)
        ax.set_xlabel("background definition")
    fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=5,
               frameon=False)
    save(fig, "fig3_background_definition")


def youden(pos, neg):
    c = np.unique(np.r_[pos, neg])
    j = (pos[None, :] >= c[:, None]).mean(axis=1) - (neg[None, :] >= c[:, None]).mean(axis=1)
    return c[int(np.argmax(j))]


def fig4(df):
    panels = [("Montreal", "phasic_pct", 15.0, "Montreal phasic (% of 2 s mini-epochs)"),
              ("SINBAR", "any_pct", 18.2, "SINBAR any (% of 3 s mini-epochs)")]
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.2), layout="constrained")
    rng = np.random.default_rng(0)
    for k, (ax, (meth, metric, published, title)) in enumerate(zip(axes, panels)):
        d = df[df.method == meth]
        # derived as in run_report.py T7: RBD and other patients on the same system
        pos = d.loc[(d.who == "CAP RBD") & d.same, metric].dropna().to_numpy(float)
        neg = d.loc[d.who == "CAP patients", metric].dropna().to_numpy(float)
        cut = youden(pos, neg)
        ticks = []
        for i, g in enumerate(GROUPS):
            v = d.loc[d.who == g, metric].dropna().to_numpy(float)
            ax.scatter(i + rng.uniform(-0.25, 0.25, len(v)), v, s=3, lw=0, color=GROUP_COLOR[g],
                       alpha=0.35 if len(v) > 200 else 0.8)
            ticks.append(f"{LABEL[g]}\n{100 * (v >= published).mean():.0f}→"
                         f"{100 * (v >= cut).mean():.0f}%")
        ax.axhline(published, color="k", ls="--", lw=0.8, label=f"published cut-off ({published:g}%)")
        ax.axhline(cut, color="k", lw=0.8, label=f"re-derived cut-off ({cut:.1f}%)")
        ax.set_xticks(range(len(GROUPS)), ticks, fontsize=7)
        ax.set_xlabel("flagged positive: published → re-derived cut-off")
        ax.set_ylim(0, 100)
        ax.set_ylabel(title)
        ax.legend(frameon=False, loc="upper right", fontsize=6.5)
        ax.set_title(f"{'AB'[k]}  {meth}")
    save(fig, "fig4_cutoffs")


def fig5(df):
    quiet = df[df.method == "Montreal"][["cohort", "record", "background_phasic_uV"]].rename(
        columns={"background_phasic_uV": "quiet"})
    d = df[df.method.isin(["Montreal", "SINBAR", "RAI"])].merge(quiet, on=["cohort", "record"])
    d = d.dropna(subset=["quiet", "y"])
    fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.8), sharey=True, layout="constrained")
    for ax, cohort, name in zip(axes, ["chal18", "hmc"], ["A  MGH", "B  HMC"]):
        g = d[d.cohort == cohort]
        edges = g[g.method == "Montreal"].quiet.quantile([0, 1 / 3, 2 / 3, 1]).to_numpy()
        t = np.clip(np.searchsorted(edges, g.quiet, side="right") - 1, 0, 2)
        for j, meth in enumerate(["Montreal", "SINBAR", "RAI"]):
            m = g.method.to_numpy() == meth
            ys = [100 * g.y.to_numpy()[m & (t == k)].mean() for k in range(3)]
            ax.bar(np.arange(3) + (j - 1) * 0.27, ys, width=0.26, color=COLOR[meth], label=meth)
        ax.set_xticks(range(3), [f"lowest third\n< {edges[1]:.2f} µV", "middle third",
                                 f"highest third\n> {edges[2]:.2f} µV"])
        ax.set_xlabel("night's quiet-REM chin EMG level")
        ax.set_title(name)
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[1].legend(frameon=False, loc="upper left")
    save(fig, "fig5_level_vs_verdict")


def main():
    config.FIGURES.mkdir(parents=True, exist_ok=True)
    df = load_scores()
    print(f"figures -> {config.FIGURES}")
    fig1(df)
    fig2(df)
    if (config.RESULTS / "sensitivity_background.csv").exists():
        fig3(df)
    fig4(df)
    fig5(df)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_ladder.py
"""The noise ladder: how each method's verdict moves with the noise floor.

Every ladder pack (run_cohort.py --ladder) holds one night's chin EMG as
recorded and with white noise added in band: at fixed levels (n050 ... n300 =
0.5 ... 3 uV RMS) and at the level that brings its quiet-REM level to a common
target (h150, h200 = 1.5 and 2.0 uV). This scores every copy with every method
and reports, against the criteria fixed in paper/preregistration_noise_ladder.md:

  L1  dose-response: quiet-REM level and share flagged, per group and step
  L2  matched floor: MGH against HMC after harmonising both (criteria 1, 2)
  L3  RBD sensitivity as the floor rises (criterion 3)
  L4  the remedy: cut-offs re-derived on harmonised CAP, leave-one-out and
      bootstrap, applied to harmonised MGH and HMC (criterion 4)
  L5  the recording systems' own noise, measured on EEG (criterion 5)

    python run_ladder.py   -> results/ladder_scores.csv, results/ladder_report.md,
                              figures/fig6_noise_ladder, figures/fig7_harmonisation
"""
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import score_all
from run_report import MIN_REM, WITHDRAWN, wilson
from run_score import group_of

LADDER = ["native", "n050", "n100", "n150", "n200", "n300"]
STEPS = LADDER + ["h150", "h200"]
TARGET = "h200"
DECIDING = ["Montreal", "SINBAR", "RAI"]
WHO = ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"]
REMEDY = [("Montreal", "phasic_pct"), ("Montreal", "tonic_pct"), ("SINBAR", "any_pct"),
          ("SINBAR", "phasic_pct"), ("SINBAR", "tonic_pct"), ("AASM", "phasic_epoch_pct"),
          ("RAI", "atonia_loss")]
KEY = [("Montreal", "phasic_pct"), ("Montreal", "tonic_pct"), ("SINBAR", "any_pct")]
YES = (True, "True", 1, 1.0)


def score_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    eeg = m.get("eeg") or {}
    info = dict(cohort=m["cohort"], record=m["record"], group=group_of(m["cohort"], m["record"]),
                chin_fs=m.get("chin_fs"), unit_known=m.get("unit_known", True),
                eeg_nrem=eeg.get("floor_nrem_uv"), eeg_rem=eeg.get("floor_rem_uv"),
                eeg_channel=eeg.get("channel"))
    rows = []
    for step in STEPS:
        if step != "native" and f"rms_{step}" not in m["rows"]:
            continue
        for r in score_all(pack, step):
            r.pop("band", None)
            rows.append({**info, "step": step, **r})
    return rows


def load_scores():
    files = sorted(p for p in (config.DURABLE / "ladder").glob("*/*.npz") if ".tmp" not in p.name)
    print(f"scoring {len(files)} ladder packs", flush=True)
    with ProcessPoolExecutor(max_workers=2) as ex:
        rows = [r for part in ex.map(score_file, files, chunksize=4) for r in part]
    df = pd.DataFrame(rows)
    df = df[~df.apply(lambda r: (r.cohort, r.record) in WITHDRAWN, axis=1)]
    df["atonia_loss"] = 1 - df["rai"]
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    # a night's quiet-REM level belongs to the night and step, not to the method:
    # take it from the Montreal row and attach it to every method's row
    level = df[df.method == "Montreal"].set_index(["cohort", "record", "step"]).background_phasic_uV
    df["level"] = level.reindex(pd.MultiIndex.from_frame(df[["cohort", "record", "step"]])).to_numpy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select(
        [cap & same & (df.group == "RBD"), cap & same & ~df.group.isin(["RBD", "healthy"]),
         cap & (df.group == "healthy"), df.cohort == "chal18", df.cohort == "hmc"],
        ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"], default="")
    return df


def youden(p, n):
    c = np.unique(np.r_[p, n])
    j = (p[None, :] >= c[:, None]).mean(axis=1) - (n[None, :] >= c[:, None]).mean(axis=1)
    return c[int(np.argmax(j))]


def pct(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    p, lo, hi = wilson(v.sum(), len(v))
    return p, lo, hi, len(v)


def main():
    df = load_scores()
    df.to_csv(config.RESULTS / "ladder_scores.csv", index=False)
    base = df[df.rem_minutes >= MIN_REM]
    lines = []
    say = lambda s="": (print(s), lines.append(s))

    # L1 ─────────────────────────────────────────────────────────────────────
    say("L1  Dose-response: median quiet-REM level (uV) and % flagged by the published cut-offs")
    for who in WHO:
        g = base[base.who == who]
        if g.empty:
            continue
        say(f"  {who} (n={g[g.step == 'native'].record.nunique()})")
        for step in STEPS:
            s = g[g.step == step]
            if s.empty:
                continue
            cells = "  ".join(f"{m} {100 * s[s.method == m].y.mean():5.1f}%" for m in DECIDING)
            say(f"    {step:<7} level {s[s.method == 'Montreal'].level.median():5.2f}   {cells}")
    say()

    # L2 ─────────────────────────────────────────────────────────────────────
    say("L2  Matched floor: MGH against HMC (criterion 1: <= 3 points Montreal, <= 5 SINBAR)")
    for step in ["native", "h150", "h200"]:
        cells = []
        for m in DECIDING:
            a = pct(base[(base.who == "MGH") & (base.step == step) & (base.method == m)].y)
            b = pct(base[(base.who == "HMC") & (base.step == step) & (base.method == m)].y)
            cells.append(f"{m} MGH {100 * a[0]:.0f} [{100 * a[1]:.0f}-{100 * a[2]:.0f}] vs HMC "
                         f"{100 * b[0]:.0f} [{100 * b[1]:.0f}-{100 * b[2]:.0f}] (diff {100 * (a[0] - b[0]):+.1f})")
        say(f"  {step:<7} " + ";  ".join(cells))
    say("  criterion 2: MGH's ladder interpolated at HMC's native median level")
    hmc_level = base[(base.who == "HMC") & (base.step == "native")].level.median()
    for m in DECIDING:
        g = base[(base.who == "MGH") & (base.method == m) & base.step.isin(LADDER)]
        x = np.log([g[g.step == s].level.median() for s in LADDER])
        y = [100 * g[g.step == s].y.mean() for s in LADDER]
        at = np.interp(np.log(hmc_level), x, y)
        h = 100 * base[(base.who == "HMC") & (base.step == "native") & (base.method == m)].y.mean()
        say(f"    {m:<9} MGH at {hmc_level:.2f} uV: {at:.1f}%   HMC native: {h:.1f}%   (diff {at - h:+.1f})")
    say()

    # L3 ─────────────────────────────────────────────────────────────────────
    say("L3  Sensitivity for RBD (CAP, main system) as the floor rises (criterion 3: < 70% at h200)")
    for m in DECIDING:
        cells = []
        for step in STEPS:
            p, lo, hi, n = pct(base[(base.who == "CAP RBD") & (base.step == step) & (base.method == m)].y)
            if n:
                cells.append(f"{step} {100 * p:.0f}")
        say(f"  {m:<9} " + "  ".join(cells) + f"   (n={n})")
    say()

    # L4 ─────────────────────────────────────────────────────────────────────
    say("L4  Remedy: cut-offs re-derived on CAP, applied to MGH and HMC, with and without")
    say("    harmonising every night to 2.0 uV (criterion 4 for Montreal phasic/tonic and")
    say("    SINBAR any: |FP MGH - FP HMC| <= 5 points and leave-one-out sensitivity >= 80%)")
    rng = np.random.default_rng(0)
    remedy = []
    for step in ["native", TARGET]:
        for meth, metric in REMEDY:
            g = base[(base.step == step) & (base.method == meth)]
            pos = g.loc[g.who == "CAP RBD", metric].dropna().to_numpy(float)
            neg = g.loc[g.who == "CAP patients", metric].dropna().to_numpy(float)
            if len(pos) < 5 or len(neg) < 5:
                continue
            cut = youden(pos, neg)
            cv_s = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
            cv_f = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
            ext = {k: g.loc[g.who == k, metric].dropna().to_numpy(float) for k in ("MGH", "HMC")}
            boot = {k: [] for k in ext}
            for _ in range(2000):
                c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
                for k, v in ext.items():
                    boot[k].append((v >= c).mean())
            fp = {k: (v >= cut).mean() for k, v in ext.items()}
            ci = {k: np.percentile(v, [2.5, 97.5]) for k, v in boot.items()}
            remedy.append(dict(step=step, method=meth, metric=metric, cut=cut, cv_sens=cv_s, cv_fp=cv_f,
                               fp_mgh=fp["MGH"], fp_hmc=fp["HMC"], ci_mgh=ci["MGH"], ci_hmc=ci["HMC"]))
            r = remedy[-1]
            ok = ""
            if (meth, metric) in KEY:
                ok = "  PASS" if abs(r["fp_mgh"] - r["fp_hmc"]) <= 0.05 and cv_s >= 0.80 else "  fail"
            say(f"  {step:<7} {meth:<9} {metric:<17} cut {cut:7.2f}  CV sens {100 * cv_s:3.0f}%  "
                f"CV FP {100 * cv_f:3.0f}%  MGH FP {100 * fp['MGH']:3.0f} [{100 * ci['MGH'][0]:.0f}-"
                f"{100 * ci['MGH'][1]:.0f}]  HMC FP {100 * fp['HMC']:3.0f} [{100 * ci['HMC'][0]:.0f}-"
                f"{100 * ci['HMC'][1]:.0f}]  gap {100 * (fp['MGH'] - fp['HMC']):+.0f}{ok}")
    say()

    # L5 ─────────────────────────────────────────────────────────────────────
    say("L5  The systems' own noise: EEG 40-90 Hz floor (uV), median [IQR], and the chin's")
    say("    quiet-REM level (criterion 5: HMC/MGH EEG ratio >= 1.5 = system-wide)")
    nat = base[(base.step == "native") & (base.method == "Montreal")]
    by = {"CAP (main system)": nat[(nat.cohort == "cap") & (nat.chin_fs == 512)],
          "MGH": nat[nat.cohort == "chal18"], "HMC": nat[nat.cohort == "hmc"]}
    med = {}
    for name, g in by.items():
        e = g.eeg_nrem.dropna().astype(float)
        r = g.eeg_rem.dropna().astype(float)
        if len(e):
            med[name] = e.median()
            say(f"  {name:<18} n={len(e):<4} EEG N2/N3 {e.median():.2f} [{e.quantile(.25):.2f}-{e.quantile(.75):.2f}]"
                f"   EEG REM {r.median():.2f}   chin quiet-REM {g.level.median():.2f}")
    if "MGH" in med and "HMC" in med:
        say(f"  HMC/MGH EEG floor ratio {med['HMC'] / med['MGH']:.2f}")

    out = config.RESULTS / "ladder_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figures(base, remedy)


def figures(base, remedy):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import COLOR, save

    colors = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "MGH": "#1f77b4", "HMC": "#9467bd"}
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.9), sharey=True, layout="constrained")
    for ax, m in zip(axes, DECIDING):
        for who, col in colors.items():
            g = base[(base.who == who) & (base.method == m)]
            if g.empty:
                continue
            x = [g[g.step == s].level.median() for s in LADDER]
            y = [100 * g[g.step == s].y.mean() for s in LADDER]
            ax.plot(x, y, "o-", ms=3, lw=1, color=col, mfc="white", label=who)
            ax.plot(x[0], y[0], "o", ms=5, color=col)                  # as recorded
            h = g[g.step == TARGET]
            ax.plot(h.level.median(), 100 * h.y.mean(), "*", ms=8, color=col)
        ax.set_xscale("log")
        ax.set_xticks([0.5, 1, 2, 4], ["0.5", "1", "2", "4"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("quiet-REM level (µV)")
        ax.set_title(m)
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].set_ylim(0, 100)
    axes[0].legend(frameon=False, loc="upper right", fontsize=6.5)
    save(fig, "fig6_noise_ladder")

    rem = pd.DataFrame(remedy)
    if rem.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.9), layout="constrained")
    # A: published cut-offs, as recorded vs harmonised
    ax = axes[0]
    groups = ["CAP RBD", "MGH", "HMC"]
    for j, (m, step, hatch) in enumerate([("Montreal", "native", ""), ("Montreal", TARGET, "//"),
                                          ("SINBAR", "native", ""), ("SINBAR", TARGET, "//")]):
        vals = [100 * base[(base.who == w) & (base.step == step) & (base.method == m)].y.mean()
                for w in groups]
        ax.bar(np.arange(3) + (j - 1.5) * 0.2, vals, width=0.19, color=COLOR[m], hatch=hatch,
               edgecolor="white", label=f"{m}, {'as recorded' if step == 'native' else 'harmonised 2 µV'}")
    ax.set_xticks(range(3), groups)
    ax.set_ylabel("nights flagged positive (%)")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False, fontsize=6, loc="upper right")
    ax.set_title("A  Published cut-offs")
    # B: re-derived cut-offs on harmonised data: external false positives
    ax = axes[1]
    k = rem[(rem.step == TARGET) & rem.apply(lambda r: (r.method, r.metric) in KEY, axis=1)]
    for i, r in enumerate(k.itertuples()):
        for dx, fp, ci, col in ((-0.12, r.fp_mgh, r.ci_mgh, "#1f77b4"), (0.12, r.fp_hmc, r.ci_hmc, "#9467bd")):
            ax.errorbar(i + dx, 100 * fp, yerr=[[100 * (fp - ci[0])], [100 * (ci[1] - fp)]],
                        fmt="o", ms=4, capsize=2, color=col)
    ax.plot([], [], "o", color="#1f77b4", label="MGH")
    ax.plot([], [], "o", color="#9467bd", label="HMC")
    ax.set_xticks(range(len(k)), [f"{r.method} {r.metric.split('_')[0]}\nLOO sens {100 * r.cv_sens:.0f}%"
                                  for r in k.itertuples()], fontsize=7)
    ax.set_ylabel("external nights flagged (%)")
    top = max(max(r.ci_mgh[1], r.ci_hmc[1]) for r in k.itertuples())
    ax.set_ylim(0, max(30, 100 * top + 5))
    ax.legend(frameon=False, fontsize=6.5)
    ax.set_title("B  Re-derived on harmonised CAP")
    save(fig, "fig7_harmonisation")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_floor.py
"""Floor-corrected RWA criteria: the published rules with the noise floor subtracted.

The relative criteria count activity above k x background, and background is
mostly the recording's own noise floor (run_ladder.py), so the muscle activity
a criterion requires scales with the noise of the recording system. The
floor-corrected (FC) rules fix that requirement in microvolts (src/rwa.py,
`threshold`): at a reference background B they are the published rules, below
it they become absolute, above it the noise is subtracted instead of
multiplied. A "clamp" variant (background never below B, no subtraction)
separates the two ingredients.

Scored on the ordinary packs (every night as recorded, and HMC scaled by 0.39
for gain) and on the noise-ladder packs (the same nights with noise added),
against the criteria fixed in paper/preregistration_floor_corrected.md:

  FC1  site comparison as recorded (criteria 3, 4, 5)
  FC2  noise invariance on the ladder (criteria 1, 2)
  FC3  the reference background B from 0.5 to 2.0 uV
  FC4  discrimination (AUC, RBD vs same-system CAP patients)
  FC5  cut-offs re-derived under FC (leave-one-out, bootstrap)
  FC6  gain: HMC scaled by 0.39, and the size of muscle activity per cohort
  FC7  Montreal phasic bursts up to 10 s instead of 15 s
  FC8  MGH: the floor by record batch, and the ladder's 300 against the rest
  FC9  age: RBD against same-system CAP patients under 50 and aged 50 or over
  FC10 CAP's healthy volunteers night by night (mixed recording systems)

CAP RBD and CAP patients are the nights recorded on CAP's main system (512 Hz,
microvolts): 21 RBD and 51 other patients.

    python run_floor.py -> results/floor_scores.csv, floor_ladder_scores.csv,
                           floor_report.md, figures/fig8_floor_corrected
"""
import dataclasses
import sys
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from src.pack import load_pack
from src.rwa import (METHODS, NOISE_P5_RATIO, _select, floor_corrected, mini_epoch_rms,
                     rem_atonia_index, score_night)
from run_ladder import STEPS, YES, pct, youden
from run_report import MIN_REM, WITHDRAWN, paired_change
from run_score import auc, group_of

REFS = (0.5, 0.75, 1.0, 1.25, 1.5, 2.0)     # reference background B, uV
PRIMARY = 1.0
COHORTS = ("cap", "chal18", "hmc", "hmc_low")
PAIRED = "h150"                              # the ladder step the criteria use
WHO = ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC"]
KEY = {"Montreal": ["tonic_pct", "phasic_pct"], "SINBAR": ["any_pct"]}


def variants():
    """(variant, reference uV, Method) for every scoring of Montreal and SINBAR."""
    out = []
    for key in ("montreal", "sinbar"):
        m = METHODS[key]
        out.append(("published", np.nan, m))
        out += [("FC", ref, floor_corrected(m, ref)) for ref in REFS]
        out.append(("clamp", PRIMARY, floor_corrected(m, PRIMARY, mode="clamp")))
    out.append(("max10", np.nan, dataclasses.replace(METHODS["montreal"], phasic_max_s=10.0)))
    return out


def scale(pack):
    """The size of the night's chin EMG in 1 s RMS (uV): quiet-REM level b, REM
    median and 95th percentile, N2/N3 median, and each with the noise implied by
    b (b / 0.91) removed in power ("_net")."""
    ep, env, _, nrem = _select(pack, "native")
    if env is None or len(ep) < 2:
        return {}
    r, _ = mini_epoch_rms(env.reshape(-1), pack["meta"]["env_fs"], 1.0)
    if len(r) < 10:
        return {}
    b = float(np.percentile(r, 5))
    out = {"rem_p50": float(np.median(r)), "rem_p95": float(np.percentile(r, 95))}
    if len(nrem) >= 60:
        out["nrem_p50"] = float(np.median(nrem))
        out["nrem_p5"] = float(np.percentile(nrem, 5))
    n2 = (b / NOISE_P5_RATIO) ** 2
    out.update({f"{k}_net": float(np.sqrt(max(v * v - n2, 0.0))) for k, v in list(out.items())})
    return dict(out, level_1s=b)


def score_rows(pack, band, info):
    rows = []
    for variant, ref, m in variants():
        rows.append({**info, **score_night(pack, m, band), "variant": variant, "ref_uv": ref})
    rows.append({**info, **rem_atonia_index(pack, band), "variant": "published", "ref_uv": np.nan})
    return rows


def _info(m, cohort):
    ecg = m.get("ecg") or m.get("chin_info", {}).get("ecg") or {}
    return dict(cohort=cohort, record=m["record"], group=group_of(cohort, m["record"]),
                chin_fs=m.get("chin_fs", m.get("chin_info", {}).get("fs")),
                unit_known=m.get("unit_known", True), age=m.get("age"), pid=m.get("pid"),
                beats=int(ecg.get("beats") or 0))


def score_pack_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    info = _info(m, m.get("cohort", path.parent.name))
    rows = score_rows(pack, "native", info)
    rows[0].update(scale(pack))                  # one row per night carries the scale
    return rows


def score_ladder_file(path):
    pack = load_pack(path)
    m = pack["meta"]
    info = _info(m, m["cohort"])
    rows = []
    for step in STEPS:
        if step == "native" or f"rms_{step}" in m["rows"]:
            rows += score_rows(pack, step, {**info, "step": step})
    return rows


def label(df):
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record).isin(gone)].copy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select(
        [same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"]),
         cap & (df.group == "healthy"), df.cohort == "chal18", df.cohort == "hmc",
         df.cohort == "hmc_low"],
        ["CAP RBD", "CAP patients", "CAP healthy", "MGH", "HMC", "HMC x0.39"], default="")
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    df["ref_key"] = df.ref_uv.fillna(0.0)
    return df


def score(files, fn):
    with ProcessPoolExecutor(max_workers=2) as ex:
        return pd.DataFrame([r for part in ex.map(fn, files, chunksize=4) for r in part])


def sel(df, method, variant="published", ref=PRIMARY):
    s = (df.method == method) & (df.variant == variant)
    if variant in ("FC", "clamp"):
        s &= np.isclose(df.ref_key, ref)
    return df[s]


def share(g):
    p, lo, hi, n = pct(g.y)
    return f"{100 * p:3.0f} [{100 * lo:.0f}-{100 * hi:.0f}]" if n else "   -"


def main():
    packs = [p for c in COHORTS for p in sorted((config.DURABLE / "packs" / c).glob("*.npz"))
             if ".tmp" not in p.name]
    ladder = sorted(p for p in (config.DURABLE / "ladder").glob("*/*.npz") if ".tmp" not in p.name)
    print(f"scoring {len(packs)} packs and {len(ladder)} ladder packs", flush=True)
    nat = label(score(packs, score_pack_file))
    nat.to_csv(config.RESULTS / "floor_scores.csv", index=False)
    lad = label(score(ladder, score_ladder_file))
    lad.to_csv(config.RESULTS / "floor_ladder_scores.csv", index=False)
    nat = nat[nat.rem_minutes >= MIN_REM]
    lad = lad[lad.rem_minutes >= MIN_REM]
    level = lad[(lad.method == "Montreal") & (lad.variant == "published")] \
        .set_index(["cohort", "record", "step"]).background_phasic_uV
    lad["level"] = level.reindex(pd.MultiIndex.from_frame(lad[["cohort", "record", "step"]])).to_numpy()

    lines = []
    say = lambda s="": (print(s), lines.append(s))
    verdict = {}

    # FC1 ────────────────────────────────────────────────────────────────────
    say(f"FC1 Share flagged positive as recorded (all nights, published % cut-offs), % [95% CI];")
    say(f"    FC and clamp with B = {PRIMARY} uV")
    say(f"    {'':<22}" + "".join(f"{w:<16}" for w in WHO) + "MGH-HMC")
    gaps = {}
    for meth in ("Montreal", "SINBAR", "RAI"):
        for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
            g = sel(nat, meth, variant)
            cells = [share(g[g.who == w]) for w in WHO]
            gap = 100 * (g[g.who == "MGH"].y.mean() - g[g.who == "HMC"].y.mean())
            gaps[meth, variant] = gap
            say(f"    {meth + ' ' + variant:<22}" + "".join(f"{c:<16}" for c in cells) + f"{gap:+.1f}")
    for meth, limit in (("Montreal", 5), ("SINBAR", 10)):
        sens = sel(nat, meth, "FC")
        sens = 100 * sens[sens.who == "CAP RBD"].y.mean()
        pub_fp = sel(nat, meth)
        fc_fp = sel(nat, meth, "FC")
        verdict[f"3 {meth}"] = abs(gaps[meth, "FC"]) <= limit
        verdict[f"4 {meth}"] = sens >= 85
        verdict[f"5 {meth}"] = fc_fp[fc_fp.who == "CAP patients"].y.mean() <= \
            pub_fp[pub_fp.who == "CAP patients"].y.mean()
        say(f"  criterion 3 ({meth}-FC gap <= {limit}): {gaps[meth, 'FC']:+.1f} -> "
            f"{'met' if verdict[f'3 {meth}'] else 'not met'};  criterion 4 (RBD sensitivity >= 85%): "
            f"{sens:.0f}% -> {'met' if verdict[f'4 {meth}'] else 'not met'};  criterion 5 (CAP patients "
            f"not above published): {'met' if verdict[f'5 {meth}'] else 'not met'}")
    say()

    # FC2 ────────────────────────────────────────────────────────────────────
    say("FC2 The same nights as the noise floor rises: % positive per ladder step, published")
    say(f"    cut-offs; FC and clamp with B = {PRIMARY} uV; median quiet-REM level in the first line")
    steps = [s for s in STEPS if s in set(lad.step)]
    say(f"    {'':<30}" + "".join(f"{s:>7}" for s in steps))
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC"):
        g = lad[lad.who == who]
        if g.empty:
            continue
        lv = [sel(g, "Montreal")[lambda d: d.step == s].level.median() for s in steps]
        say(f"  {who} (n={g[g.step == 'native'].record.nunique()})")
        say(f"    {'quiet-REM level, uV':<28}" + "".join(f"{v:7.2f}" for v in lv))
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
                h = sel(g, meth, variant)
                say(f"    {meth + ' ' + variant:<28}" + "".join(
                    f"{100 * h[h.step == s].y.mean():7.1f}" for s in steps))
    say(f"  Paired change, as recorded -> {PAIRED} (same nights):")
    for who in ("MGH", "CAP RBD", "CAP patients", "HMC"):
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC", "clamp")):
                h = sel(lad[lad.who == who], meth, variant)
                w = h.pivot_table(index="record", columns="step", values="y", aggfunc="first")
                if "native" not in w or PAIRED not in w:
                    continue
                w = w[["native", PAIRED]].dropna()
                d = 100 * (w[PAIRED].mean() - w["native"].mean())
                say(f"    {who:<13} {meth + ' ' + variant:<20} {100 * w['native'].mean():5.1f} -> "
                    f"{100 * w[PAIRED].mean():5.1f}   " + paired_change(w["native"], w[PAIRED]))
                if variant == "FC" and who == "MGH":
                    verdict[f"1 {meth}"] = abs(d) <= 5
                if variant == "FC" and who == "CAP RBD":
                    verdict[f"2 {meth}"] = abs(d) <= 10
    for meth in ("Montreal", "SINBAR"):
        say(f"  criterion 1 ({meth}-FC, MGH within 5 points): "
            f"{'met' if verdict.get(f'1 {meth}') else 'not met'};  criterion 2 ({meth}-FC, "
            f"CAP RBD within 10 points): {'met' if verdict.get(f'2 {meth}') else 'not met'}")
    say()

    # FC3 ────────────────────────────────────────────────────────────────────
    say("FC3 The reference background B: as recorded (sensitivity, CAP patients, MGH, HMC, gap)")
    say(f"    and on the ladder (change as recorded -> {PAIRED}, MGH and CAP RBD), points")
    for meth in ("Montreal", "SINBAR"):
        for variant, ref in [("published", np.nan)] + [("FC", r) for r in REFS] + [("clamp", PRIMARY)]:
            g = sel(nat, meth, variant, ref)
            v = {w: 100 * g[g.who == w].y.mean() for w in ("CAP RBD", "CAP patients", "MGH", "HMC")}
            h = sel(lad, meth, variant, ref)
            ch = {}
            for w in ("MGH", "CAP RBD"):
                p = h[h.who == w].pivot_table(index="record", columns="step", values="y", aggfunc="first")
                p = p[["native", PAIRED]].dropna() if {"native", PAIRED} <= set(p) else p.iloc[:0]
                ch[w] = 100 * (p[PAIRED].mean() - p["native"].mean()) if len(p) else np.nan
            name = meth + " " + variant + ("" if variant == "published" else f" {ref:g}")
            say(f"  {name:<20} RBD {v['CAP RBD']:3.0f}  CAP patients {v['CAP patients']:3.0f}  "
                f"MGH {v['MGH']:3.0f}  HMC {v['HMC']:3.0f}  gap {v['MGH'] - v['HMC']:+5.1f}   "
                f"ladder MGH {ch['MGH']:+5.1f}  RBD {ch['CAP RBD']:+5.1f}")
    say()

    # FC4 ────────────────────────────────────────────────────────────────────
    say("FC4 RBD vs same-system CAP patients (AUC), as recorded")
    for meth, metrics in KEY.items():
        for metric in metrics:
            cells = []
            for variant, ref in [("published", np.nan)] + [("FC", r) for r in REFS]:
                g = sel(nat, meth, variant, ref)
                cells.append(f"{'pub' if variant == 'published' else f'FC {ref:g}'} "
                             f"{auc(g[g.who == 'CAP RBD'][metric], g[g.who == 'CAP patients'][metric])[0]:.3f}")
            say(f"  {meth:<9} {metric:<11} " + "  ".join(cells))
    say()

    # FC5 ────────────────────────────────────────────────────────────────────
    say("FC5 Cut-offs re-derived on CAP (Youden, RBD vs same-system patients), leave-one-out")
    say("    and 2,000 bootstrap re-derivations; external shares flagged [95% interval]")
    rng = np.random.default_rng(0)
    for variant in ("published", "FC"):
        for meth, metrics in KEY.items():
            g = sel(nat, meth, variant)
            for metric in metrics:
                pos = g.loc[g.who == "CAP RBD", metric].dropna().to_numpy(float)
                neg = g.loc[g.who == "CAP patients", metric].dropna().to_numpy(float)
                ext = {k: g.loc[g.who == k, metric].dropna().to_numpy(float) for k in ("MGH", "HMC")}
                cut = youden(pos, neg)
                cv_s = np.mean([pos[i] >= youden(np.delete(pos, i), neg) for i in range(len(pos))])
                cv_f = np.mean([neg[i] >= youden(pos, np.delete(neg, i)) for i in range(len(neg))])
                boot = {k: [] for k in ext}
                for _ in range(2000):
                    c = youden(rng.choice(pos, len(pos)), rng.choice(neg, len(neg)))
                    for k, v in ext.items():
                        boot[k].append((v >= c).mean())
                fp = {k: 100 * (v >= cut).mean() for k, v in ext.items()}
                ci = {k: np.percentile(v, [2.5, 97.5]) * 100 for k, v in boot.items()}
                say(f"  {variant:<9} {meth:<9} {metric:<11} cut {cut:6.2f}  LOO sens {100 * cv_s:3.0f}%  "
                    f"LOO FP {100 * cv_f:3.0f}%  MGH {fp['MGH']:3.0f} [{ci['MGH'][0]:.0f}-{ci['MGH'][1]:.0f}]  "
                    f"HMC {fp['HMC']:3.0f} [{ci['HMC'][0]:.0f}-{ci['HMC'][1]:.0f}]  gap {fp['MGH'] - fp['HMC']:+.0f}")
    say()

    # FC6 ────────────────────────────────────────────────────────────────────
    say("FC6 Gain: HMC scaled by 0.39 against HMC, same nights")
    for meth in ("Montreal", "SINBAR", "RAI"):
        for variant in (("published",) if meth == "RAI" else ("published", "FC")):
            g = sel(nat, meth, variant)
            w = g[g.cohort.isin(["hmc", "hmc_low"])].pivot_table(index="record", columns="cohort",
                                                                  values="y", aggfunc="first").dropna()
            if {"hmc", "hmc_low"} <= set(w):
                say(f"  {meth + ' ' + variant:<20} {100 * w.hmc.mean():5.1f} -> {100 * w.hmc_low.mean():5.1f}   "
                    + paired_change(w.hmc, w.hmc_low))
    say("  Size of the chin EMG (1 s RMS, uV, median [IQR]); _net = noise (b / 0.91) removed")
    sc = nat[(nat.method == "Montreal") & (nat.variant == "published")]
    cols = [c for c in ("level_1s", "rem_p50_net", "rem_p95_net", "nrem_p50_net") if c in sc]
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC", "HMC x0.39"):
        g = sc[sc.who == who]
        say(f"    {who:<13} n={len(g):<4} " + "  ".join(
            f"{c} {g[c].median():.2f} [{g[c].quantile(.25):.2f}-{g[c].quantile(.75):.2f}]" for c in cols))
    m, h = sc[sc.who == "MGH"], sc[sc.who == "HMC"]
    say("    MGH/HMC ratio of medians: " + "  ".join(f"{c} {m[c].median() / h[c].median():.2f}" for c in cols))
    say()

    # FC7 ────────────────────────────────────────────────────────────────────
    say("FC7 Montreal, phasic bursts up to 10 s instead of 15 s (published cut-offs), % positive")
    a, b = sel(nat, "Montreal"), sel(nat, "Montreal", "max10")
    for who in ("CAP RBD", "CAP patients", "MGH", "HMC"):
        w = pd.merge(a[a.who == who][["record", "y", "phasic_pct"]], b[b.who == who][["record", "y", "phasic_pct"]],
                     on="record", suffixes=("", "_10"))
        say(f"  {who:<13} {100 * w.y.mean():5.1f} -> {100 * w.y_10.mean():5.1f}   nights changed "
            f"{int((w.y != w.y_10).sum())}   phasic % median {w.phasic_pct.median():.1f} -> {w.phasic_pct_10.median():.1f}")
    say()

    # FC8 ────────────────────────────────────────────────────────────────────
    say("FC8 MGH: quiet-REM level and share positive (published) by record batch, and the")
    say("    noise-ladder subset against the rest")
    mg = sel(nat, "Montreal")
    sb = sel(nat, "SINBAR")
    mg = mg[mg.who == "MGH"].merge(sb[sb.who == "MGH"][["record", "y"]], on="record", suffixes=("", "_s"))
    mg["batch"] = mg.record.str[:4]
    for b_, g in mg.groupby("batch"):
        say(f"  {b_:<6} n={len(g):<4} level {g.background_phasic_uV.median():.2f}  Montreal "
            f"{100 * g.y.mean():3.0f}%  SINBAR {100 * g.y_s.mean():3.0f}%")
    sub = set(lad[lad.cohort == "chal18"].record)
    for name, g in (("ladder 300", mg[mg.record.isin(sub)]), ("the rest", mg[~mg.record.isin(sub)])):
        say(f"  {name:<10} n={len(g):<4} level {g.background_phasic_uV.median():.2f}  Montreal "
            f"{100 * g.y.mean():3.0f}%  SINBAR {100 * g.y_s.mean():3.0f}%  age {pd.to_numeric(g.age).mean():.0f}")
    from scipy.stats import spearmanr
    per = mg.groupby("batch").agg(level=("background_phasic_uV", "median"), m=("y", "mean"), s=("y_s", "mean"))
    if len(per) >= 4:
        say(f"  across batches, Spearman rho of median level with share positive: Montreal "
            f"{spearmanr(per.level, per.m)[0]:+.2f}, SINBAR {spearmanr(per.level, per.s)[0]:+.2f} "
            f"({len(per)} batches)")
    say()

    # FC9 ────────────────────────────────────────────────────────────────────
    from src.fetch import cap_demographics
    demo = cap_demographics()
    if demo:
        say("FC9 Age in CAP: RBD against same-system patients under 50 and aged 50 or over")
        cap = nat[nat.who.isin(["CAP RBD", "CAP patients"])].copy()
        cap["age"] = cap.record.map(lambda r: demo.get(r, {}).get("age"))
        parts = {"patients < 50": lambda d: d[(d.who == "CAP patients") & (d.age < 50)],
                 "patients >= 50": lambda d: d[(d.who == "CAP patients") & (d.age >= 50)],
                 "RBD": lambda d: d[d.who == "CAP RBD"]}
        one = cap[(cap.method == "Montreal") & (cap.variant == "published")]
        say("  " + "   ".join(f"{k}: n={len(f(one))}, age {f(one).age.mean():.0f}" for k, f in parts.items()))
        for meth in ("Montreal", "SINBAR"):
            for variant in ("published", "FC"):
                g = sel(cap, meth, variant)
                say(f"  {meth + ' ' + variant:<20} flagged: " + "   ".join(
                    f"{k} {100 * f(g).y.mean():3.0f}%" for k, f in parts.items()))
        for meth, metrics in KEY.items():
            for metric in metrics:
                for variant in ("published", "FC"):
                    g = sel(cap, meth, variant)
                    r_ = parts["RBD"](g)[metric]
                    say(f"  AUC {meth + ' ' + variant:<20} {metric:<11} vs all patients "
                        f"{auc(r_, g[g.who == 'CAP patients'][metric])[0]:.3f}   vs patients >= 50 "
                        f"{auc(r_, parts['patients >= 50'](g)[metric])[0]:.3f}")
        say()

    # FC10 ───────────────────────────────────────────────────────────────────
    say("FC10 CAP healthy volunteers, night by night (mixed recording systems): positive")
    say("     published / floor-corrected (- = not scored: no physical units)")
    h = nat[nat.who == "CAP healthy"]
    flag = lambda g: "-" if g.empty or pd.isna(g.y.iloc[0]) else str(int(g.y.iloc[0]))
    for rec, g in h.groupby("record"):
        mp = sel(g, "Montreal")
        say(f"  {rec:<4} fs {g.chin_fs.iloc[0]:>5g}  units {'yes' if bool(g.unit_known.iloc[0]) else 'no ':<3}  "
            f"level {mp.background_phasic_uV.iloc[0]:6.2f}   Montreal {flag(mp)}/{flag(sel(g, 'Montreal', 'FC'))}"
            f"   SINBAR {flag(sel(g, 'SINBAR'))}/{flag(sel(g, 'SINBAR', 'FC'))}")
    say()

    say("Pre-specified criteria: " + "  ".join(f"{k}: {'met' if v else 'not met'}" for k, v in sorted(verdict.items())))
    out = config.RESULTS / "floor_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figure(nat, lad)


def figure(nat, lad):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import COLOR, save

    colors = {"CAP RBD": "#d62728", "CAP patients": "#7f7f7f", "MGH": "#1f77b4", "HMC": "#9467bd"}
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8), layout="constrained",
                             gridspec_kw={"width_ratios": [1, 1, 1.25]})
    ladder = [s for s in STEPS if s.startswith("n") or s == "native"]
    for ax, meth in zip(axes[:2], ("Montreal", "SINBAR")):
        for who, col in colors.items():
            for variant, ls in (("published", ":"), ("FC", "-")):
                g = sel(lad[lad.who == who], meth, variant)
                if g.empty:
                    continue
                lv = sel(lad[lad.who == who], "Montreal")
                x = [lv[lv.step == s].level.median() for s in ladder]
                y = [100 * g[g.step == s].y.mean() for s in ladder]
                ax.plot(x, y, ls, marker="o", ms=2.5, lw=1, color=col,
                        label=who if variant == "FC" else None)
        ax.set_xscale("log")
        ax.set_xticks([0.5, 1, 2, 4], ["0.5", "1", "2", "4"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("quiet-REM level (µV)")
        ax.set_ylim(0, 100)
        ax.set_title(f"{'AB'[meth == 'SINBAR']}  {meth}")
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].plot([], [], ":", color="k", lw=1, label="published")
    axes[0].plot([], [], "-", color="k", lw=1, label="floor-corrected")
    axes[0].legend(frameon=False, fontsize=6, loc="lower left", bbox_to_anchor=(0.0, 0.38))
    ax = axes[2]
    groups = ["CAP RBD", "CAP patients", "MGH", "HMC"]
    for j, (meth, variant, hatch) in enumerate([("Montreal", "published", ""), ("Montreal", "FC", "//"),
                                               ("SINBAR", "published", ""), ("SINBAR", "FC", "//")]):
        g = sel(nat, meth, variant)
        vals = [100 * g[g.who == w].y.mean() for w in groups]
        ax.bar(np.arange(4) + (j - 1.5) * 0.2, vals, width=0.19, color=COLOR[meth], hatch=hatch,
               edgecolor="white", label=f"{meth}, {'published' if variant == 'published' else 'floor-corrected'}")
    ax.set_xticks(range(4), ["CAP\nRBD", "CAP\npatients", "MGH", "HMC"])
    ax.set_ylim(0, 130)
    ax.set_yticks(range(0, 101, 20))
    ax.set_ylabel("nights flagged positive (%)")
    ax.legend(frameon=False, fontsize=5.5, loc="upper center", ncol=2)
    ax.set_title("C  As recorded")
    save(fig, "fig8_floor_corrected")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/rbd-rwa/run_external.py
"""External cohorts: the noise floor and the floor-corrected rule on new systems.

Scores the packs of the development cohorts (CAP, MGH, HMC) and of the external
ones (BOAS, DOD-H, DOD-O) with the published and floor-corrected (B = 1.0 uV)
Montreal and SINBAR rules and the RAI, and reports against the predictions
fixed in paper/preregistration_external.md:

  X1  cohorts: nights, REM, age, floor, floor signature, heartbeat removal
  X2  scale: net N2/N3 and REM levels (E1)
  X3  share flagged: published, floor-corrected, SINBAR-FC at 49.1% (E2, E3, E5)
  X4  floor against share flagged across all groups without RBD (E4)
  X5  DOD-O against MGH's noise ladder (E6)
  X6  BOAS: every night, and by age (descriptive)
  X7  heartbeat check (exploratory, not pre-specified)

BOAS has several nights for some participants; its primary rows are each
participant's first recording. CAP RBD and CAP patients are the nights on
CAP's main system (512 Hz, microvolts).

    python run_external.py -> results/external_scores.csv, external_report.md,
                              figures/fig9_external
"""
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import numpy as np
import pandas as pd

from src import config
from run_floor import YES, pct, score, score_pack_file, sel
from run_report import MIN_REM, WITHDRAWN

COHORTS = ("cap", "chal18", "hmc", "boas", "dodh", "dodo")
NAME = {"chal18": "MGH", "hmc": "HMC", "boas": "BOAS", "dodh": "DOD-H", "dodo": "DOD-O"}
HEALTHY = ["BOAS", "DOD-H"]
WITHOUT_RBD = ["CAP patients", "MGH", "HMC", "BOAS", "DOD-H", "DOD-O"]
CUT_FC_SINBAR = 49.1          # SINBAR-FC "any" cut-off re-derived on CAP (results_floor, FC5)
MGH_NREM_NET = 1.60           # MGH's median net N2/N3 level, uV (results_floor, FC6)


def label(df):
    gone = {f"{c}/{r}" for c, r in WITHDRAWN}
    df = df[~(df.cohort + "/" + df.record).isin(gone)].copy()
    cap = df.cohort == "cap"
    same = cap & (df.chin_fs == 512) & df.unit_known.astype(bool)
    df["who"] = np.select([same & (df.group == "RBD"), same & ~df.group.isin(["RBD", "healthy"])],
                          ["CAP RBD", "CAP patients"],
                          default=df.cohort.map(NAME).fillna(""))
    df["y"] = df.positive.map(lambda x: float(x in YES) if pd.notna(x) else np.nan)
    df["ref_key"] = df.ref_uv.fillna(0.0)
    # BOAS: a participant's first recording is primary
    df["primary"] = True
    b = df.cohort == "boas"
    if b.any():
        num = df.loc[b, "record"].str.extract(r"(\d+)")[0].astype(int)
        pid = df.loc[b, "pid"].fillna(df.loc[b, "record"]).astype(str)
        first = num.groupby(pid).transform("min")
        df.loc[b, "primary"] = (num == first).to_numpy()
    return df


def periodicity(pack, band):
    """How strongly the REM envelope repeats at a heart rate: the largest spectral peak of
    the envelope between 0.7 and 2 Hz over the median in 0.5-3 Hz (Welch, 20-s windows,
    REM stretches of at least a minute); higher = more heartbeat in the EMG."""
    from scipy.signal import welch
    from src.rwa import _segments, _select
    ep, env, _, _ = _select(pack, band)
    if env is None or len(ep) < 2:
        return np.nan
    fs = pack["meta"]["env_fs"]
    acc, n = None, 0
    for a, b in _segments(ep):
        x = env[a:b].reshape(-1).astype(float)
        if len(x) >= fs * 60:
            f, p = welch(x - x.mean(), fs=fs, nperseg=int(fs * 20))
            acc, n = (p if acc is None else acc + p), n + 1
    if not n:
        return np.nan
    band_, hr = (f >= 0.5) & (f <= 3.0), (f >= 0.7) & (f <= 2.0)
    return float(acc[hr].max() / np.median(acc[band_]))


def heartbeat_check(path):
    """Exploratory (not pre-specified): the heartbeat periodicity of the REM envelope with
    and without heartbeat removal, and the calls with the heartbeat left in."""
    from src.pack import load_pack
    from src.rwa import METHODS, floor_corrected, score_night
    pack = load_pack(path)
    m = pack["meta"]
    out = {"cohort": m.get("cohort", path.parent.name), "record": m["record"],
           "period_native": periodicity(pack, "native"),
           "period_with_ecg": periodicity(pack, "native_with_ecg")}
    for key in ("montreal", "sinbar"):
        for tag, meth in (("pub", METHODS[key]), ("fc", floor_corrected(METHODS[key], 1.0))):
            r = score_night(pack, meth, "native_with_ecg")
            out[f"{key}_{tag}_with_ecg"] = float(bool(r.get("positive"))) if r.get("positive") is not None else np.nan
    return [out]


def main():
    packs = [p for c in COHORTS for p in sorted((config.DURABLE / "packs" / c).glob("*.npz"))
             if ".tmp" not in p.name]
    print(f"scoring {len(packs)} packs", flush=True)
    df = label(score(packs, score_pack_file))
    from src.fetch import cap_demographics
    demo = cap_demographics()
    if demo:
        c = df.cohort == "cap"
        df.loc[c, "age"] = df.loc[c, "record"].map(lambda r: demo.get(r, {}).get("age"))
    df.to_csv(config.RESULTS / "external_scores.csv", index=False)
    hb = score([p for p in packs if p.parent.name in ("chal18", "hmc", "boas", "dodo", "dodh")],
               heartbeat_check)
    all_nights = df[(df.rem_minutes >= MIN_REM) & (df.who != "")]
    d = all_nights[all_nights.primary]

    lines = []
    say = lambda s="": (print(s), lines.append(s))
    verdict = {}
    groups = ["CAP RBD"] + WITHOUT_RBD
    one = d[(d.method == "Montreal") & (d.variant == "published")]
    one = one.assign(signature=one.nrem_p5 / one.level_1s)

    # X1 ─────────────────────────────────────────────────────────────────────
    say("X1  Cohorts (nights with >= 10 min REM; BOAS: first night per participant)")
    for w in groups:
        g = one[one.who == w]
        if g.empty:
            continue
        age = pd.to_numeric(g.age, errors="coerce")
        say(f"  {w:<13} n={len(g):<4} REM {g.rem_minutes.median():5.1f} min  age "
            f"{age.mean():4.0f} ({age.min():.0f}-{age.max():.0f})  fs {g.chin_fs.median():g}  "
            f"floor {g.level_1s.median():.2f} [{g.level_1s.quantile(.25):.2f}-{g.level_1s.quantile(.75):.2f}] uV  "
            f"signature {g.signature.median():.2f}  heartbeat removed {int((g.beats > 0).sum())}/{len(g)}")
    say()

    # X2 ─────────────────────────────────────────────────────────────────────
    say("X2  Scale: 1 s RMS with the noise removed (uV, median [IQR]); E1: net N2/N3 within")
    say(f"    0.67-1.5 x MGH's {MGH_NREM_NET} uV")
    for w in groups:
        g = one[one.who == w]
        if g.empty:
            continue
        v = g.nrem_p50_net
        ok = 0.67 * MGH_NREM_NET <= v.median() <= 1.5 * MGH_NREM_NET
        if w in ("BOAS", "DOD-H", "DOD-O"):
            verdict[f"E1 {w}"] = ok
        say(f"  {w:<13} N2/N3 {v.median():.2f} [{v.quantile(.25):.2f}-{v.quantile(.75):.2f}]  "
            f"REM {g.rem_p50_net.median():.2f}  REM p95 {g.rem_p95_net.median():.2f}"
            + (f"   E1 {'met' if ok else 'not met'}" if w in ("BOAS", "DOD-H", "DOD-O") else ""))
    say()

    # X3 ─────────────────────────────────────────────────────────────────────
    say("X3  Share flagged, % [95% CI]: published cut-offs; floor-corrected (B = 1.0 uV);")
    say(f"    SINBAR-FC at the cut-off re-derived on CAP ({CUT_FC_SINBAR}%)")
    rows = {}
    for w in groups:
        g = d[d.who == w]
        if g.empty:
            continue
        cells = {}
        for meth in ("Montreal", "SINBAR", "RAI"):
            for variant in (("published",) if meth == "RAI" else ("published", "FC")):
                cells[f"{meth} {variant}"] = sel(g, meth, variant).y
        fc_any = sel(g, "SINBAR", "FC").any_pct
        cells["SINBAR-FC >= 49.1"] = (fc_any >= CUT_FC_SINBAR).astype(float).where(fc_any.notna())
        rows[w] = {k: pct(v) for k, v in cells.items()}
        say(f"  {w}")
        for k, (p, lo, hi, n) in rows[w].items():
            say(f"    {k:<22} {100 * p:5.1f} [{100 * lo:.0f}-{100 * hi:.0f}]  (n={n})")
    floor = one.groupby("who").level_1s.median()
    # E2: quiet healthy systems flag healthy people
    for w in HEALTHY:
        if w in rows and floor.get(w, np.inf) < 1.0:
            verdict[f"E2 {w}"] = rows[w]["SINBAR published"][0] >= 0.30
    # E3: two healthy cohorts, floors differing >= 1.5x
    if all(w in rows for w in HEALTHY):
        a, b = sorted(HEALTHY, key=lambda w: floor[w])          # a = quieter
        if floor[b] / floor[a] >= 1.5:
            for meth, need in (("SINBAR", 0.0), ("Montreal", 0.05)):
                pub = rows[a][f"{meth} published"][0] - rows[b][f"{meth} published"][0]
                fc = rows[a][f"{meth} FC"][0] - rows[b][f"{meth} FC"][0]
                if meth == "SINBAR" or pub >= need:
                    verdict[f"E3 {meth}"] = pub > 0 and abs(fc) <= abs(pub) / 2
                    say(f"  E3 {meth}: {a} - {b} published {100 * pub:+.1f}, floor-corrected {100 * fc:+.1f} points")
        else:
            say(f"  E3 not applicable: healthy floors {floor[a]:.2f} and {floor[b]:.2f} uV differ < 1.5x")
    # E5: the portable cut-off stays specific
    for w, limit in (("BOAS", 0.10), ("DOD-H", 0.10), ("DOD-O", 0.20)):
        if w in rows:
            verdict[f"E5 {w}"] = rows[w]["SINBAR-FC >= 49.1"][0] <= limit
    say()

    # X4 ─────────────────────────────────────────────────────────────────────
    from scipy.stats import spearmanr
    say("X4  Across groups without RBD: median floor against share flagged (Spearman rho)")
    have = [w for w in WITHOUT_RBD if w in rows]
    x = [floor[w] for w in have]
    for meth in ("SINBAR", "Montreal"):
        pub = [rows[w][f"{meth} published"][0] for w in have]
        fc = [rows[w][f"{meth} FC"][0] for w in have]
        rp, rf = spearmanr(x, pub)[0], spearmanr(x, fc)[0]
        say(f"  {meth:<9} published rho {rp:+.2f}   floor-corrected rho {rf:+.2f}   ({len(have)} groups: "
            + ", ".join(f"{w} {floor[w]:.2f} uV" for w in have) + ")")
        if meth == "SINBAR":
            verdict["E4"] = rp < 0 and abs(rf) < abs(rp)
    say()

    # X5 ─────────────────────────────────────────────────────────────────────
    say("X5  DOD-O against MGH's noise ladder (published cut-offs; E6: within 10 points)")
    lad_path = config.RESULTS / "ladder_scores.csv"
    if "DOD-O" in rows and lad_path.exists():
        lad = pd.read_csv(lad_path)
        lad = lad[(lad.who == "MGH") & (lad.rem_minutes >= MIN_REM)]
        steps = ["native", "n050", "n100", "n150", "n200", "n300"]
        f0 = floor["DOD-O"]
        ok_all = True
        for meth in ("Montreal", "SINBAR"):
            g = lad[lad.method == meth]
            lv = [g[g.step == s].level.median() for s in steps]
            yy = [100 * g[g.step == s].y.mean() for s in steps]
            if not (min(lv) <= f0 <= max(lv)):
                say(f"  {meth}: DOD-O floor {f0:.2f} uV outside the ladder's {min(lv):.2f}-{max(lv):.2f} uV: not applicable")
                ok_all = None
                continue
            pred = float(np.interp(np.log(f0), np.log(lv), yy))
            obs = 100 * rows["DOD-O"][f"{meth} published"][0]
            say(f"  {meth:<9} predicted at {f0:.2f} uV: {pred:5.1f}%   observed {obs:5.1f}%   (diff {obs - pred:+.1f})")
            if ok_all is not None:
                ok_all = ok_all and abs(obs - pred) <= 10
        if ok_all is not None:
            verdict["E6"] = ok_all
    say()

    # X6 ─────────────────────────────────────────────────────────────────────
    b = all_nights[all_nights.cohort == "boas"]
    if len(b):
        say("X6  BOAS: every night (n=%d) and first nights by age, %% flagged" % b.record.nunique())
        for name, g in (("all nights", b), ("first nights < 50 y", d[(d.cohort == "boas") & (pd.to_numeric(d.age, errors='coerce') < 50)]),
                        ("first nights >= 50 y", d[(d.cohort == "boas") & (pd.to_numeric(d.age, errors='coerce') >= 50)])):
            cells = []
            for meth in ("Montreal", "SINBAR"):
                for variant in ("published", "FC"):
                    cells.append(f"{meth} {variant} {100 * sel(g, meth, variant).y.mean():4.0f}")
            n = g.record.nunique()
            say(f"  {name:<22} n={n:<4} " + "  ".join(cells))
        say()

    # X7 ─────────────────────────────────────────────────────────────────────
    say("X7  Heartbeat check (exploratory, not pre-specified): periodicity of the REM envelope")
    say("    at a heart rate (higher = more heartbeat), and the share flagged with it left in")
    h = hb.merge(d[(d.method == "Montreal") & (d.variant == "published")][["cohort", "record", "who"]],
                 on=["cohort", "record"])
    for w in ("MGH", "HMC", "DOD-O", "DOD-H", "BOAS"):
        g = h[h.who == w]
        if g.empty:
            continue
        cells = []
        for meth in ("Montreal", "SINBAR"):
            for variant, tag in (("published", "pub"), ("FC", "fc")):
                removed = sel(d[d.who == w], meth, variant).set_index("record").y
                left = g.set_index("record")[f"{meth.lower()}_{tag}_with_ecg"].reindex(removed.index)
                cells.append(f"{meth} {variant} {100 * removed.mean():.0f} -> {100 * left.mean():.0f}")
        say(f"  {w:<6} n={len(g):<4} periodicity after removal {g.period_native.median():.2f} "
            f"[p90 {g.period_native.quantile(.9):.2f}], left in {g.period_with_ecg.median():.2f}")
        say("         % flagged, removed -> left in: " + "   ".join(cells))
    ref = h[h.who == "MGH"].period_native.quantile(0.9)
    b = h[h.who == "BOAS"]
    if len(b):
        clean = set(b[b.period_native <= ref].record)
        say(f"  BOAS nights no more periodic than MGH's after removal (<= {ref:.2f}, MGH p90): "
            f"{len(clean)}/{len(b)}; % flagged among them:")
        say("    " + "   ".join(
            f"{meth} {variant} {100 * sel(d[(d.who == 'BOAS') & d.record.isin(clean)], meth, variant).y.mean():.0f}"
            for meth in ("Montreal", "SINBAR") for variant in ("published", "FC")))
    say()

    say("Pre-specified predictions: " + "  ".join(
        f"{k}: {'met' if v else 'not met'}" for k, v in sorted(verdict.items())))
    out = config.RESULTS / "external_report.md"
    out.write_text("```\n" + "\n".join(lines) + "\n```\n", encoding="utf-8")
    print(f"\nwritten -> {out}")
    figure(one, rows, floor)


def figure(one, rows, floor):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from run_figures import save

    style = {"CAP RBD": ("#d62728", "*"), "CAP patients": ("#7f7f7f", "o"), "MGH": ("#1f77b4", "o"),
             "HMC": ("#9467bd", "o"), "DOD-O": ("#17becf", "o"), "BOAS": ("#2ca02c", "s"),
             "DOD-H": ("#bcbd22", "s")}
    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.0), sharey=True, layout="constrained")
    for ax, meth in zip(axes, ("SINBAR", "Montreal")):
        for w, (col, mk) in style.items():
            if w not in rows:
                continue
            x = floor[w]
            p, f = 100 * rows[w][f"{meth} published"][0], 100 * rows[w][f"{meth} FC"][0]
            ax.annotate("", xy=(x, f), xytext=(x, p),
                        arrowprops=dict(arrowstyle="->", color=col, lw=0.8, alpha=0.7))
            ax.plot(x, p, mk, color=col, mfc="white", ms=6)
            ax.plot(x, f, mk, color=col, ms=6, label=w)
        ax.set_xscale("log")
        ax.set_xticks([0.3, 0.5, 1, 2, 3], ["0.3", "0.5", "1", "2", "3"])
        ax.xaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
        ax.set_xlabel("median quiet-REM level of the cohort (µV)")
        ax.set_ylim(0, 100)
        ax.set_title(f"{'AB'[meth == 'Montreal']}  {meth}")
    axes[0].set_ylabel("nights flagged positive (%)")
    axes[0].plot([], [], "o", color="k", mfc="white", label="published")
    axes[0].plot([], [], "o", color="k", label="floor-corrected")
    axes[0].legend(frameon=False, fontsize=6, loc="upper right")
    axes[1].legend(frameon=False, fontsize=6, loc="upper right", ncol=2)
    save(fig, "fig9_external")


if __name__ == "__main__":
    main()

## 1. CAP: extract every night (~12 minutes), then pack

In [ ]:
!cd /content/rbd-rwa && python run_extract.py && python run_cohort.py --cohort cap {PACK_MIRROR}

## 2. HMC: 151 nights (about 15 GB streamed, ~10 minutes)

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort hmc --workers 3 {PACK_MIRROR}

## 3. Challenge 2018: 994 nights (about 130 GB streamed, roughly an hour)

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort chal18 --workers 3 {PACK_MIRROR}

## 4. Hardware simulation: HMC recorded the way MGH records chin EMG

The same 151 HMC nights again, quantised to 1 uV steps (`hmc_q1uv`), and also
resampled to 200 Hz (`hmc_mgh`), as the MGH chin EMG is stored. MGH's quiet
REM level (~0.6 uV) is below that step while HMC's (~1.5 uV) is above it, so
both are repeated after scaling HMC down to the MGH level (`hmc_low`,
`hmc_low_mgh`). Any change against the matching undegraded copy is caused by
the recording hardware alone (~25 minutes).

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort hmc_q1uv --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_mgh --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_low --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort hmc_low_mgh --workers 3 {PACK_MIRROR}

The reverse test: 300 MGH nights with white noise added to the chin EMG, which raises
their quiet-REM level from ~0.6 uV to HMC's ~1.5 uV (`chal18_noise`, ~15 minutes).

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort chal18_noise --limit 300 --workers 3 {PACK_MIRROR}

## 5. The noise ladder

CAP, HMC and 300 MGH nights streamed once more; each night's chin EMG is kept as
recorded and with white noise added at 0.5-3 uV, and at the level that brings its
quiet-REM level to 1.5 and 2.0 uV. The analysis and its decision criteria were fixed
in advance (paper/preregistration_noise_ladder.md). About 40 minutes; the ladder
files stay on this machine (they are large) and only the results go to Drive.

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort cap --ladder --workers 3 && python run_cohort.py --cohort hmc --ladder --workers 3 && python run_cohort.py --cohort chal18 --ladder --limit 300 --workers 3 && python run_ladder.py

## 5b. Floor-corrected criteria

The published rules with the noise floor subtracted instead of multiplied, scored on
every night as recorded and on the noise ladder, against the criteria fixed in advance
(paper/preregistration_floor_corrected.md). About 15 minutes.

In [ ]:
!cd /content/rbd-rwa && python run_floor.py

## 5c. External cohorts

Three recording systems not used to find the effect: BOAS (OpenNeuro, 128 nights of adults
from the general population), DOD-O (55 patients, Stanford) and DOD-H (25 healthy
volunteers, France), scored against the predictions fixed in advance
(paper/preregistration_external.md). BOAS takes ~15 minutes; the Dreem sets come from
Zenodo at a few MB/s and take about an hour each.

In [ ]:
!cd /content/rbd-rwa && python run_cohort.py --cohort boas --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort dodo --workers 3 {PACK_MIRROR} && python run_cohort.py --cohort dodh --workers 3 {PACK_MIRROR} && python run_external.py

## 6. Score every night with every method

In [ ]:
!cd /content/rbd-rwa && python run_score.py {RESULT_MIRROR}

## 7. Background definition sensitivity, then the report tables

In [ ]:
!cd /content/rbd-rwa && python run_sensitivity.py && python run_report.py
if MIRROR:
    !mkdir -p {MIRROR}/results && cp /content/durable/results/* {MIRROR}/results/

## 8. Figures

In [ ]:
!cd /content/rbd-rwa && python run_figures.py
if MIRROR:
    !mkdir -p {MIRROR}/figures && cp /content/durable/figures/* {MIRROR}/figures/
from IPython.display import Image, display
for name in sorted(os.listdir('/content/durable/figures')):
    if name.endswith('.png'):
        display(Image(f'/content/durable/figures/{name}', width=900))